# ДЗ 2, опыт 22: cosine LR на последних 25k шагах

Восстанавливаем **№21/75000** и повторяем **75001…100000** с cosine **0.0001→0.00001**. Меняется только LR: прежние 4+4 слоя, d256, SwiGLU, Pre-LN, Qwen-style BPE8k, 500k пар, batch8, FP32 и кеш. Веса, AdamW, RNG и покрытие данных восстанавливаются; нового warmup нет.

Контроль уже получен: исходный №21 прошёл те же последние 25k при постоянном LR 0.0001. Сравнение при одинаковых итоговых 100k отделит влияние LR от добавочного обучения. По прежнему темпу это около **17 минут чистого обучения**, плюс подготовка, проверки и инференс; фактическое время измеряется заново.

Открыть в новой Colab-сессии с T4 и выполнить ячейки по порядку. В окне загрузки выбрать **`hw02-encoder4-decoder4-qwen-bpe8k-swiglu-500k-100000steps-results.zip`** или только **`step-75000.pt` из него**. `latest.pt` и `step-100000.pt` для этого опыта не подходят. Другие старые ноутбуки не нужны.

Полный heldout1024/greedy/manual74 на 75k и 100k; loss на тех же 1024 каждые 5k между ними. Лог loss16 каждые 50 шагов сохранён. Исходный ручной счёт и дополнительный счёт с `+=/-=` выводятся отдельно; эталоны обучения не меняются. Fresh1024 остаётся неоценённым для финального сравнения.

При прерывании выполнить последнюю ячейку и сохранить частичный ZIP. Train-ячейку повторно не запускать. Вернуть итоговый ZIP и этот notebook с outputs.


In [ ]:
%pip install -q tokenizers==0.23.2 pyarrow


## 1. Настройки и сохранённые контроли


In [ ]:
import ast
import base64
import io
import contextlib
import zlib
import hashlib
import json
import math
import os
import random
import time
import zipfile
from collections import Counter
from pathlib import Path
from urllib.request import urlretrieve

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import tokenizers
from tokenizers import Tokenizer, Regex, models, trainers, pre_tokenizers, decoders
import pyarrow.parquet as pq
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence

SEED, TRAIN_SIZE, BATCH_SIZE = 42, 500000, 8
INITIAL_STEP, TOTAL_STEPS, SCHEDULE_TOTAL_STEPS = 75000, 100000, 100000
CONTINUATION_START_LR, CONTINUATION_MIN_LR = 0.0001, 0.00001
EVALUATION_STEPS = (75000, 80000, 85000, 90000, 95000, 100000)
FULL_EVALUATION_STEPS = (75000, 100000)
HELDOUT_SIZE, FRESH_HELDOUT_SIZE = 1024, 1024
LOG_EVERY, CHECKPOINT_EVERY = 50, 1000
D_MODEL, ENCODER_LAYERS, DECODER_LAYERS, MAX_POSITIONS = 256, 4, 4, 2048
SWIGLU_DIM = 680  # 2/3 * 1024 rounded to a nearby multiple of8; almost equal parameter budget
LEARNING_RATE, MIN_LEARNING_RATE = 0.001, 0.0001
WARMUP_STEPS = 0  # no new warmup; original optimizer base LR remains0.001
PAD, BOS, EOS, TOKEN_OFFSET, TARGET_VOCAB = 0, 1, 2, 3, 8192
TOKENIZER_TRAIN_SIZE = TRAIN_SIZE
GENERATION_CAP, INFERENCE_BATCH, BYTE_BUDGET = 512, 32, 512
LOGIT_ATOL, LOGIT_RTOL = 1e-4, 1e-4
RUN_DIR = Path(f'hw02-encoder4-decoder4-qwen-bpe8k-swiglu-500k-cosine-tail-{INITIAL_STEP}-to-{TOTAL_STEPS}')
RESULTS_FILE = RUN_DIR / 'results.json'
TOKENIZER_FILE = RUN_DIR / 'tokenizer.json'
LATEST_CHECKPOINT = RUN_DIR / 'latest.pt'
ARCHIVE_FILE = Path(f'hw02-encoder4-decoder4-qwen-bpe8k-swiglu-500k-cosine-tail-{INITIAL_STEP}-to-{TOTAL_STEPS}-results.zip')
assert tokenizers.__version__ == '0.23.2', 'Run the installation cell in a fresh session'
assert torch.cuda.is_available(), 'Включи GPU в настройках Colab'
assert not RESULTS_FILE.exists() and not LATEST_CHECKPOINT.exists(), 'Сначала сохрани прежний запуск; используй новую сессию'
RUN_DIR.mkdir(exist_ok=True)
device = torch.device('cuda')
random.seed(SEED)
torch.manual_seed(SEED)

def sync():
    if device.type == 'cuda': torch.cuda.synchronize()
def sha(raw): return hashlib.sha256(raw).hexdigest()
def data_hash(rows):
    return sha(json.dumps(rows, ensure_ascii=False, sort_keys=True).encode())
def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(path)
def save_results(): write_json(RESULTS_FILE, result)

result = {'experiment': '22-cosine-tail-75000-to100000-encoder4-decoder4-qwen-bpe8k-swiglu-500k', 'phase': 'setup', 'actual_steps': 0,
    'runtime': {'device': str(device), 'gpu': torch.cuda.get_device_name(0),
                'torch': torch.__version__, 'cuda': torch.version.cuda, 'tokenizers': tokenizers.__version__, 'cpu_threads': torch.get_num_threads()},
    'training_log': [], 'diagnostics': {}, 'checkpoint_files': {},
    'training_seconds_excluding_evaluation_logging_and_checkpoints': 0.0,
    'run_finished': False, 'test_evaluated': False, 'functional_quality_evaluated': False,
    'historical_bpe4k18': {}}
result['historical_bpe4k18'] = json.loads(zlib.decompress(base64.b64decode('eNrtvWtvXEl2pvtXEvo0c1BFxf0yNQMc2+MZDNzHp+Fuz3wwDCKuEqcokmaSdXGj/vs8K3aSTKoktdrObhAaCahSKnNn7IgVa73rfVdE7PzDq5vb6x/GVblq49V/2r36Xflh9N39ftzufu9246ebcXvxblzd6bRr11d3F1f35e7i+krH13e35eLKK/X967/+7d+673flzvM6+u9fa978bnd1vatlPy4vrsbudtzeX53t/vrnu7H7+9/8hn/T2P7u9r7dcbd5e/1ut1833t+/ezf6t3fX34+r3d/87a5c9d2b68u+W++8rtJAu76/utufvfpm92p/fX/bxvnt2N9f3u3P92+L8UGG4abTLehQYm19RtvDDMGoZrO1zaegShvJ6j5tKLpbO1Iu0SblptY5qiSN35Tb8m7cjds9LQargtaGt3u5K7zxh1e344eLPbaQ+1UTjQspz6liHrkZM2b3Ls2hrauzmqxs6LVKu8twF1dvzsdP5d3N5ZDmsSN/ZERvy23fnzMBnbf/6Q/bG7xUq0d3b+Vu0oXN/q/p5N2368tnvPyX+3EntxA7SbMGE+SYw9bwwTZZy8h9MrMUbXq0VZdQbYppuBycC637zjiwRRzDzTlKas2kFBuWGe3Y8Nc/rts89F7+fb5v5epqdf/xffq1F7c5x9VuvTZi0Oz4oF33cc4b59e35++ub8UFvdOJT3CcNxf18vEeOjsVpC1Gdnferm8uxh4Ltsv7Ph7s87bs356/HZf9+v5ufe/4Ap1pmYtKa+MGtztfBjy/KRdrgnV0Xudfvtk9mVz/EZPrj5k86OifmdwbY3rwseiCYYu3eupqjFd2Wh8d9vcuTsW8eJVTjl0VE3NtaQZeh3QikyfzEZPbkD9scm3+fSY3yxAfN3ly/pnJzR8xufmYyY1x7pnJW1DJOKVHySGUUUqy0SudysD2PbfYvG/NquGMbo6XggyjtI71m+3j802uo4kpfcTk6iNe7pIKHzJ5iN7Yf4/JA3H6CZN7FWz45Z9lcONytLsDfk0+vhMw3z30CPy++lZu9e3hVt+sd6RfO7nprrTb6/1essJ+tPu7ix/G7uZCTLLbMEydnX23u76ltatyqVfTP5V2t7u5HfPip+92D9b6z/8Fy7wWE/1nXuz+8ff/Le3WzH636/c3lxetgPlrFI9f2ZNA7niDe5X9ro4pNn1A1qMsELRLdrreAEE1p58GqJtzFue7jzbr3mZTutQwGmjnVAsj1NBiNi01m57avLxuK+8d55hmrDXFu5KNpc3ke8mlZT2aKjYXM9woPc7YzLAmlTramFGZQb5RoZa4XJgMadT3R61qqwBAmwN4YKbSCn+MoWlDEnHkK5XJao7+ukF+wWe7BTNMcNZnq9R4aFUsftQs0OP0yLqNDPJz/xGHTp1kqCrOP6MpWklguJRV81NSpYmluByHizUfd3abwfP7q/a2XL1ZbkcqH8c3/vgly57BHY/YeK1CdKYqVwhYukZPeMeqpmrPxarpJb6Z6BQq8cpERs/sEZFGWena/dUFznFw9kc3WXHmbZSAevSlx48fY84FIyHzQ7m86GuWjzNzcM8+etZzNTou5cj4tTetLDPvxyRLuhZrhWMkldwA3lOh9z6VOSYDwQk8fMIr6fn4iU4/hvPRjQHPX338dOticNeRpx3KzzSYvOECE2WLwSl8dy0P4K+RU5p2I1VjG85TICH8Y64kfjV+/NCNY0jvffh025FMtNN7+E4ouXnYA240Z5Jk1SqfqF7naIYvQKcEbKyfhEO3s9vaV0Rd35XLD45YGTH24RP51/GddYLLmRwN0RH81Di0x2mZ3jAIbjNB75mKL97riIt7l5N3NSuPXdzykkPLtHgMEtF6YmeMWAiJqqCBM+mketIKc4IXyXur4mh8GEgbyg2Vs8l+ThvnAolnyH9Zbg6OeLjfAbc/cpEg6uGKCZE9gvYBC/9hxc4fXv3aTAvSjw1UALjYlPI+91D18MGronpP+KevTjwmWz1GgnWkpmeAomYfGoaKsBEZx7y4XCpgdeQB+L996MjZ/96TLR4x8f3xHAYBW38cwkcu+XXqkTHt+gUXX5EiHnPQcf45Qv9NMEiW0bvrqx/5e8kN7vxw8euVp7b8uTI5L9fIHsayI2fspqQmrnoHqbhgbLvruavXd2+1F+Gho7RKb4h+VA/f4StCzBEju7t7ZM3d2wtS0f3Vq18e3VpmRdBvj2Y6ZIybywemkB4YxLtRrj59qT4TetCvf7y6vC79nO6cP1qNV2TdvjiJPiP7xCCiJfMnyrcuru5kjFfbtQ8jR8vRCr1v17c39/vd/uJfx9JXTSanvBkPGX1L3+i53fiXe+xzfwP0jdcHD9zV+/5m3G2m2dq4vL4aywT0al68Wf66H2sYzjylZa49Vjq13LW3690tEMQ+j2bc342bzdcPVx8mYY3o8Nk/+e2z6Ndf26XCbOb95eX5n/KFDivr41Ls6RcruxI6cnt+WX7e1N/SfeMDb74lxa80I7edV+f94t0i2osmjst7gniOW9ob508fHwK4317fHNDhbBMmkKjyEBa/+/Hiv//mH3f/4XcX/P+NcKD/ZydautzuZGTjP7463LJelP1Ter26vn13vkLq6b3L23UPBIuE583dxTuMLu+9+qte3v0vaejHcfHm7d05Yyw/bxfLte3y4uZcWlwOuSK3vR39/nJ9+w+vrtDH0sxvyrvay2/+4dXRFfL+1t9zAvTdPXD3dlxBafe8+epXE32YmleHa4/nfzmWfOvxXZe3t29G+f782eighefvli/JO+utZYzzzYm3i/X4dpFvQu8OYSEtznK5F1PJHWAtF2K9J1MdQuDViq0+fkK8y+f7/6LPzn7/3eHT/yXYsN9Jnx7e+v321ry8vr79bvdou91q5FDHWPBzOQReti8tvLq8PS+TKD5/INDnb3n1r9dPI5N463c/b7HNHcqdNZtZvx9XD/MLfCwu/e3l+GFc7v76t38rl/wAma1iRrUqA1LXASzOL/r59Zz7IWMXxrS/Ge2CO18sD/8DCedBgdTr/UEej/XK/HJ842PWaaHgLRtdHZmUhBO1r1CRWotPtkJyky/JQHmbzsHN6ErVuqc+uBSyJ719V346v2H+HxidUU5w4s24ArVWfLeVWPyqzhyhysXVIfRkPGbN3dMbr357O779zd8zFe12SIELoPu7/7lrhUl69Xw0mwh8KoUt5peeVXIkIYG0t3LzI29an6PBjj8+hxEvBz0EJw3cXRwC4SgOSu9rxA8hcpQaHi6R4D98++JfH3Hj/z84zO4AIM+vQC9p/1oy28pdYPio19ff7yQUEFH744pf3C1UXDliAcUq8fHnoLj44Ofdgwke0OixaHb+lFYP1TLQ8OYBP2UYxvvFNPc/Xrz51Wc2+fCAoQspVg1Om7xqQIevPH3wUJzj+iJSdPvofFMfXPHtAgjYOPIaRnr4+5dFsB8GfD5B4OsVN4dS58AGh6DcBn53vSUNgplR7p5KoMtSUgh9/d9+a83r5YXp9bLa63/4+/++bPgQyrvf/MNChC2OXx1KisIExHgP87g8sUvI7v7Hf93LvIXXf/3dTprfbWi939pf1Oev/r/fPnPIh5wuqbV9f8ArBvotmmf3N7/9x4UKAFN5s199A6fImjvgkNy+FVs3fPpGImQVF3hxkOGgIrN0FPMPnf7DK27VPw48QmUewuGD8fQYdI8D4SJ87d1y/MewWr3ccvb5ArP7y3L78xOiMY8Ql09ccCLIk9QoobV89grWsQXug/rd35SFNQ/9xsjwgTfjp6foJ1dhDngWsPTzgVMcTaIQtWXOq8ufH/3HbK74XkVkY3Jdyib4wzENfvXrNk9cb/hzQf9GHG/xQ6ljXdysOXkY8vrH/d1M5w91wD+8Wi57TDKFaK8SnT0L0S0+sNEDvYq7SHCpiUnx8yZ7cYL1PilnSygyyfVG0IQBfvoWzp3ZiPh6vIN7vIFTj+2np+aNtU+yYQ1hQeDj7aQ3Z1k7o6O23qios//lUEz8U8ZujDsz0ZmYHrvmn/pmVo+3zrmt94fB6z9h8MGdJW3CorUH+z7dwqfHO2hzNP7s/vj4XVDWpZDQ+s4p+8sv7xdj/g3OsSo6D46RQOFI35/mLeUnzwjhseuki8eu4/qfMM5x+86fee3DUfPmyfGcfWw9msfG8+e4hQ9O4UDOkhPz5/nFcbcMupEvx6NRmyOXeJowa9Njx5yLnzlq3OHYouap7aNQ0+rJoPjOZ/iC19E7iAFwkbUxv/zyUB/T2fxbHGGrsD1CBJI7GQPcHf489j+YI6zQR1jxNGkufcojnt0IoDDPHE7Hx+btk+XTkXWs/+PWSTEq/nOySsb0fo5HPOuW0emMb8cPjD8+dVDnI99I4QkulPrc8Ytz5KPbxA/6STjCjCcz66g+w09UJEVZTYg4AY/lJ1fjxxjSv8VLtnLoUSKxEFdixK4/T4jnj70kHHnJE6i64D5hpWc3wku08i6koxg9wlT7FEfJHYVR+gzsMLIiTPKml0xD+hxPedY1YuRMBxMDxOH9SPlIYjmawk8nlmc3CgEYgds/GvuD2SV/2FPSZ2UX78n4QcXoE065POXm9rqOI0c5f+hbOjBYoVFSjyv3SLSjWpwO8SynQI5TNMkf6e8mKDcuuS3BPebOjWR677O2VqKg3l9c9ocGD4t6crOjvqx7LkiwRGu0kbvph3s9lgN++/PdW7Se1AHfjFvRD98s0g/n3V/fipjYj3f1Ek6PTXaiLw/64lmpdHV9Bzm+v1nsct1bFOJBD6wq7JEi+G7VTjfZsMmF7bs/lsvLdcWSDFBZKeVs8uZ8k+uiE+9+fjS0jodi4t3t9eXmH8to69VTDf/9uvhWu7u6pysX7dFSj9/Y1kAfNaM56MuH8tzD9x6myaxl2KfP76/29zc317eypvpwzfLIRwq+nOLxC8ccO/eOm6XoGkQY9ptHT40cnrtxaNE5JCBH035AfptJvXCJTjMa44ufiwsf6h3cfX9X7qSudH6Ndrj4aRzd9Hx/Xzc184cHQ6xQjGuJrKNKpBxxKNtptd59HNaBs+5/vrorP52P29ulhTehczDO043Km9ux6ibLVhIaP95e46sPVz7NZV610yl1iIHEvri8vx1PH5vVh9uxv778ga49jfLY3g9Xu2P3XAr8YW1gzcjux7fXe9lsdOjk7mK/e2xkd+jZTnzt7btxd9G+2x26w2V35Wfx2D6uroGYcicFu6Mu7J76+N3TDfa7d3ytjt0a+1YSR09eiIakS/v9/6uXz+PfMl8FDFqR+Tgg4+2mKM/Hu4u746Ealx6EldS3zn+k02u5jGuPAen5rD6361GJ7AFU9vdSy01nNttMEgsp55xW5pJSJ/P3ZN77d7Icst+KEgP9KSHcHy3+BlS62X+zxlx2pNfdnu9fYvSHKtuywHe7y+v9/vWiaUemf0SRtQ55eSkdO98qDWtn1Ka2vckK/gzlczkptzapHEXkca7yQQf/7ONDlWBLMr9uO4RFX+TekhgeG1sFVkPkoZjI1wE9rFdN4cHhnxa1ZHVq3fyfXpmYWyl1dJNKLFkj/CuiWTTvdLLclqOJJTUVlSxLds/di9K+N5+jV8sMZhrbY0d7p2q869YWItfJok6q3sgipoEPxJjz8CXaMpPzoQ+cCAjJS4lbW3ucnU91tmb4GnywVc+QXXG6kkdTSXZ4r1y1U6HYZqnBtTFQ/nWrwrjQi+rFD6tHtKOoXLxTA/LNP0W6RzVapnNF08HA0ILNqXve9WZua7wuzVlzlhVTrfJUloQYXJwz+NjbSNWG6VqyPvhSuymzuCgr1I1BJd0X4rmRlevThJmK9lzhuh6tDhqk460NaMG0DcuHFkxFIhavzLB95thaW20Exqxm15rBuuE88zpiNIOmph1SnTDQeNcwqpVNWrPkVrum6xO5sfUjGlWUAcOTSTX1is41OQR0XcsmeQZgbYamDN3KUKbk6vpoPShfRuzbKn8cJamm4xw9xp47Q6qFJMA8hR6Kj7XmklLk3rwczlY1i8GgNsZAjpA2Um4qV8MFXk9sOOht03TdzRhbmKbY6udghhxt9wgvwMQlT9/JNGHNSxoN/aTmnKQf71NhGggvP1Sv3uFj0LqYQijTYIGoDF5Sp2Yyp2mwmtXGrKpnWrd6Vu3TmNVbVfHn6GeP+FQDVFSjdzN3G1v0tvjqe24O911bSTJTrmstGMMOOxyWSjis0VnRHq5SspJtirZow81wnmCN7moUAqTWudqQbZt91MT8MtDeSKd6CGEkPFyOcSZuEx1TEL2vsahJ3wxZFX+3fe3FzKXkHHLrOKrtynum1xjxptlzxcG5b2txELSRWOupulgIzZJBh9pWP4oKDJVgt8ZpFNQY2thY8OVC8ERMAYEtqjWtmAVd6EIQIjCa6U63VQEu2oXYmXxhg028w5agXeHiDhil6arDoFPDHwj4UBsaRLay2pTAiDWW4k11uROZa+uJp4EZtZQHGq+KbAjJuahRa9UFp/a6EivTGPoWcth2DlkZONFcVfZaJa9qjcEDvs4BVroXIsDV2PTgf8VOFwbB27NTNY+st21CNtcWa8h2GucKmJaxOK5qMe9QGMNLjBEOygRlW7WuAUJ0xCmcdbXRO58H2ZrEjNAF4rVG+Q6xQVDbWQgpPzAEs+xN81Vp7XObM7oR1m6PFoDQmdPEIsFGlZtl3EqhYRcYEaXVeswDVirTBr20sRPahdkzfqvMI6Z7K6aBjRMQq7kTUUQsIx8IYKntpNRsJ1aKk5pmZTpwTEWk6776IcGeZTuMM9bVapU2JblqmEc3iFljlHYEM5ZoroFPOo0cFEjJrNe8fGyYOPOY3A/AK7BFWxS5W/lprZJQUYQRUaYAwVYbfmj9kLsWPXrftvtMi0NH6zwQbmcDEs1IsM+sceiBpzlcL0w8UCXXnC1JdvYQQZgAjF/9mD6PWAeTZZnHnnIEF7xvzLFlivrIqRKfYWpiUdE8U87EIFVSC9GU1QbtA2VzMh+y9Qx0drEnDMTfBmRmcptTScvOG1XXnpYcUw90R4dZX/3zL893CX1kg8znKoHD5R+XAmnt7vy0FNA2fKYWCLJVPNtUSyC7hhoYb8DrfAyBadZAFb6M51fTo6k51EiIA+Qy/A09/x1aYBPYv9ICq9r0XAss/fOeGBAC9kTiHt/2n6ER9JIbHxYJYZWIPyoScvxTRMLW7y9JJZBq80dkQl676D8tE6z6uE5YX/+oTjDxDGyV7ZE4HlkhvUydEAALd5ZccmR9GLb9uEwwsIhg/gSdEJJZtbaP6QTyF6AbnIWtmIUjnxYKTCUcCFlAFoQZwfAcMT5BvgYNhkePZOAGwvJ6zBAeN8i3wzSTfbZtETDlEAEDmgdV9DBREpNUCmEPjZeQWcinIUvqkLMJGfQVbukSIJManH1rA/HgXU1Vlvka7KZ06EMNsI4s9CBAxg1JhDxMxiDPltIsjF0b0qSfhzYS9AQqlZ038E6vSUQQHFKrIk3EqUnu/JNUsnqXu5tKzpm0Ul1ayUClNLQfWtLhyj+ptE4qxUSyp5OkRmrUlswZYfXkZtgY6d6ZlpjJTTipAiXmLUd2UlkzDFry0Fh4nZICC5QeZSacr4v4wUBwpwGamhGQCquNAcSa2eZA8o1i4BHQoqmHNb2QjCdiyHstNFTqNk1BVQz3q9jMebWEAgqKpgsUBl0CSZbyKAoD0lBs6aq6QnZHENXAWCDHA1morGzdrdOnvEg+DFBMJlqjexIfAqgb1Bd5XxmcA9rDTFYNQRqwIWebzfAkvBDKARtfbRQ5V1JLKriRqR5+pWFGMUODQkXeTa2hqCWP4LGVRp9B1zOjUBlepA5tcKvMZKcwLDy9SK2AXhcl247Fryq8P82k4J102uWhC2ykQZGgJ6uN4fusoVcomxIF6SCyCd9AMqkAOwketgZbgglIKiyIMQst7cLdVVgb5I2is9CN0Br8G3FYaoR6ZlFBU8lucQIFn8hxTCxlGAHshKSZk8Vjl6+fRJwnWX4iftDUE0MMUS6J6VQKP9Si6vyEz0N7XFJyZielyQQ6PgFqwskEPubO0AdYgdVODmhhR/hnJxDpL9QVLkdHw5RBWDnDhRsHVzVBOzZ7WM2UMDHNw8sQ8MNDztA11iLQAZUgm6nhjlU1GmdCCAjViGwLIKH6T1VokJ3hKJSJdmNI0ETYOPo8VURvxkWhzTKNwkF9kcUtTWikXEXk4fsLP9A/mvnoBS8GEQj+sWQ0EIAWMoKhURN0JiJbYZTIJsIzylRN5mDFC46C3hJw0fgodkH6Ist9G5BnpL0JU3UtOkERS3mM6Y1abgj9hlCvNlrpwEyb3nKdEzNUi0SywoeL7gNh5RAzRs4dFii7yihzwBMtVrJbvm4nagLFgU7MjvjIMZMfkmB2lPSicI8mE68a4ZdIPhhWJLKz+GkcL6p4Q7AFi8AiIMEbVVDOTYQDOI2zBK4bYEKzNjeDlEUJMI9Eu0cIoIq2NiwubJoiOrDcGAjwgZwggvECPAKVy1t2GEyhqxz1CAE3IJlp5sGeqoiE68rBThkQWhnBO4AKhCOSDxE/OmnYIYLoCGiGh80QWkX30H+QLS3KTqPcsYRqiBiptgD6CdujOlHfiJtR0M2hKSYcZEPkk/I7A2V+K7lgtYFroTEz4WlAsMLHcnwHpQ52O4tUInJqltITYh9DTWSvhiBoExwxttoo0ScBWyOFAR8Br4T9ixQDGrKro7cieMUYZw4zAuHD5I5gVxZ6sewBMOSGnCdYumOoXBsBV0MugCK4gl+IqK9yjAJ08xnUFbsU0iE6z2xtDB1QQICQX4BAZo3iGdPhdMAKWaYgGSt4Z2MCUuGvQNTICFYceBX3cFqDGdIkKIQwDMCDoPBjILLFm3zuvfS6EEKiE2D0YKUWajLrqQqEgcFpqbwQJY3oVZFhVqR7igh0AmDCoukKaC2beEgxIG6Fm0B8gMk1t0HczxopnMFfpPhSEi7fOzNQNYKeVudSw0Y3BH1CXJP9yB+9dXcodi6hSC6JHf5EzqMN0juauUCR+eNgM2EAwzVqB2tlDmslbsmc+Gc/WbFT8loB2EaQdEp+gtG4AUfJCQoIpYCRFPIOEcObQMVclcgOxCaQ/VQF0ziJfe+lWpFiZibIxBKP0CsHY4FMCclCMTg82ncG0CSLN60rGJNWnoPemCRBqVUiddHphv9APyCgWHFWkpeRQ7qwREv0EHgQh5o8mFzsxh3AOrKzN5IO6CqWVyABxCyA2lJW9a1JSJkJTOqS5XyTM3Q0ZUsWqycr/gqfJCyJAggx5BTEJj4bMQl+zt6dxUyyvSbGiqs3ErLtUPcqEKm3fjSpTPl14qQNgS+uBBjGtNiyVh0FhwjkorEDFjDigXaCZqC/ellFaFVX5+CXzHYl6g3QI6Gn5Dw2QkCXCpDSPSloMjt+oDDRw7SE472oQnYYE3dCWmhcrC4VGUBd0qJGuEOJe0QA2kwQWAznHEHhGTbRiX5bXApKAZakSQebmvBAtKNEGEkHS7oJc8epoY5WaqG9e52ByUg0NgO2nKygnhE3zKWEq9BI0jSgjCYcwokh8KR3wsDQCKGqSZxMNzxrwi0JYL34WCZZkAuqqCuYBiyKyZxhACRkuoCynFgXPg92O91J9wipPIgxYp2bvKjCPimI2YPejq4hAcyIJRsK8wH+aZmcKQVj6BRRRnaQasQYnTwfRHlviwOaq/FoKxtsmQaFLE0a8E0jQNSJnyaqkpmX7bkebdqkLg4GkL66P9UCAzkdktx7JYQgB/TQBzkV2zALmGKQpXhdIKalspKAcJ1HRorFMkTorjbQQqUJq2Z28B44vtBe1IyB4YHO4voIySw1e6gaECTrEDAdt06JrgUGLAEQZHFODTdAeDrALKObcfxECMGQbLbkRzgZM5OYeGga8GFpbdmjwnhkB7Mn+SWp9neRo6b7FWazJtCLmYzCBoFIwhjyl8KiLLUfFktOseAitDdX+hK9IefKFkDBpQg6ggkZkEMQMjB6qKTLPdhJZsCyaQLDq40AchBEzkjoD+9Qb7DhgLadQSN0eyFJa+ZmMkvETBUGhwJWlSzjV8xVIEoWe6xGkNZUEXyQzSbqD4Y2kDcaCo+CAh1IPnRRk7OaZfqjOMxqI0XZjh6gsAOF4kwmKRApMA9XZNUnytaYKnvXFHA9pwbUAvJjRPL8YW4hX7Pg5VhwaWvoL0yKeTRAq01CW+A+3KGTizPgC2G0cuy9SZrxJ1vEggZElCNczxfSDNzQEVjETpBqB+ySOPXwBe+C6AESLhygg4wONDLzVAthdXJvLz6oMAp/YaAGgAlfdi6JjcmAKDUg1UwcKQPRXp4WAEXHQ9ZimhePk7sUUlkzscuOdCQNX+TOxuEGExGO4IxSSSBNkuBSmtDXiOeeakGuFQIWSKGHRo7w9AmvS01ynuApeZn8D0+AycOTbKGpAuvS5IOKrNz6QS4FCzCAU3bdUZbf0LUFpYr8Z2qHHPUOslCn58gMrBKTUluAYy4Mkvmmj1hcC2KMCZYQ5zrAZ1MTaR4M5FdqSFGKBHIYHkoSyEX44cZhcKcsyRM0pgWwi7hDz8sGTTyki65EdTrQEOW/tmzSONmCr2ccfPmHyOLZDP8xlZiOf9ItTQ7Br/EEEouc4HAd1QhbJJpJXRDtKtjtDoucJEDYJQGVQa0iosmaPkFIogxxNU0SNgRpIR9r/JG38FdiUlpQbWsDvC9SVpE9yFLbk+1sifBfhSyvQLIBqyKOcC/4HGEv2grJhSMNf7LF1gBBIZ/DwHCjBBiCDMUXNCjgE4uUXlwXcgBKChHQiGl0koffRbc9TKKD1HDiQqAi8HXsQrLW4x1gZVkK1sqbbiHRdgKKrjcsbwgXa9BHZRtLsYSE4HxE6QjSgdsE0AyYHr8h2yT0PlSYnguww3NoiYjyuO6Wbzv8CaHGRAxJ6MZwdYN36wKPqoZcLSVYkiCkh7YjHCKqaWXEhLeOWxupQTc8TABMj0YezgDlCFJ7bshV6RMK2Uq9EbiyRkkMlsEwpJ5utgVsL+Wv4ZEF0RFNUyonZHTXA/gNP8yQyoR+B2CgHVke/mBkaXOgDo072SK4okHY3sxwLynhk6RB28G3uTlKiOALSTC9SBkWJWwR2QQu/AFsqKdaSB8MGIKGZmRyBjiMqu1QAM3UWiWFRhJwcsJjKumgkEocYtgKXunSVm4gETKVxcCjyLlT42aVBEdYE0cAKhkS9yDmIH1BluEx0iDv069Kg2teRnJ4YjEikC1R5/E5LxXSHnDVMgyhgN8nKdUOWUAXcqIdrxBFUOetDTxfNxgLshOvzsIkgE+SOzwOmo2tpjwZJk7JNsBlrcIiE12c07mtDawgBoDtaDmqE0sECTWhXPiETFuQnbAFeQxTrBnxX5heeYYIuBW2NrLtSGrQLYJCDsaDiXGNQDqqUcjkQAfNUbs8JSbK41VsEDjG1eHmC09F4DfyCQFrpvMNdqm1ktIeSQr8Qd8AP3S6wbullt8bc04SGFIl3vrxYjZJwG9IbGRaeZwcNFeWGABnKRrLlCvFX4V5FFqRsQttFlJQR6KmAwZNESdCFxopP6NVhDHBnsAv4rbAr+WBWFLPgadokHytIfANNwDlDZP/3Zs1VhvMEuzByzJWnsKXcVCSqq+irLz4LjGKo1cgFHrpFfBJJkvia0m/+ue1MX89tuHrHvA/Zd/Hr3d3H+8Ktx/cFW4/vMfDfsYej7VS/pF94OtoyMf3gYf/y/eBh4/tA982Z39yg4f7xD7wT+7v8GfkOwl9VJDs78gvdB+4c1GfET1RCqaHs3N/mX3giHfsAyohSaEun7O/I8H4ERSRTN1krwQJSlYTRDNBW8aEMpJBspJCRW/QSSSp8vKQOgU1W+vmWtigI2caFBgEGtzUTZZzyVuwBAQ6+kDWB2UFqCNdMlzcWahjQJ74k615l/UsU9jvgBPIsxKz7AEgBzMKDX4PWfiIpdO3iK7tswRkFm9WqIFRp1rzlm0gUlHOkETyJGTNlx5gSjXLOmnropyynHyUnLiWvWHJkskRLSFta94oJ5dCzUEoaJWKv6wkQctpsGMaLasUWvapOKmg0iHG1QP9kqp7/9I2xntEAHSwwajxMancNFoKJohCDKKVoLWpoiZl2Ql+6giUZqqiH2izcbq1Iqnxh2CcPIwQk/URoBqytUOZmeBNOC4UGWtuMjtgsNFVS0ME+IoX6C+tKxiklCRENvM/SK32SGk0pKcd2YCQMCMcHsJVpJbn4WnC3uzJNuhbHJj58ynBhRa3IuK5unqp/hs0sdG4RKdBqc75nhzEOQQndeihT7fJXygaMSRVBdk2ZEdtQRunsnxjIIWtSJsmD1NEkWSM0yvuJ1T/sP729aDAl3tQ4CQ1Z9fhsgrfJiZ9HRG0IloMYJy7JAYN2AZVYPz0oqET0KWI0aAtdy8nq7GeZJN/zILpSEPS6mxZC1eProDasYTsvbayDCIL8sR9ha8ztzOkLI/FdpuOPk19w3J5Fs2Nm+Ne3JN5bZgG7DO5KHxDtnQSf+CHPLO2Ntp3IJyUiFYbej3xsss2A8sgBrf3Xmo1UjAnMSApIwm9jLYKn7IKksTmRABBEU9WI0GBk75tjbKQZLR49porNDqZz0qhtEjGEBXea4iSy4bN4F60OONWE2jig0F1jWzGK7SW5XrhqnyDIeo8jZmwJUukVS1bUXMDQJsI3e0Rzl8PPfxfe+jhveMM8Zn6jcl8SP2u0xEfUL/h12chPuvQw7rrxw49+E8p4nVu+vMPPcQv79DDei7KhzSx3h538GlRvJ6z8xFVvBTfx089+DM0X0QZIqn4Y/0LPfYAW0lnIJjxsFDIv/vLnXsIWg5awPMM2XI9E+bPL4u/Hnt479iDSM0qG+pq0vKQ69rRM1qetiZLHnAZDOPJylAn9CTaIcgAMe+cMW3L/rRBFxiIrHYOl2BnAcoEs2wJqkdKVFOOGyjZZwNljb6ScGV/KFmubtsxFOpINj6WVGEeBf1isoWdyPqLmr3ZFJZ8ku2YbsoBAaPJb6WjKmyx/iUd4VAzw+WLsF7hV1Az4fmlSyUByTA0SXzIkjAUAI7UoYbC8Ivzeuhc8smOgZygBPT1KMl7R0mg1MipDARZg3LhWw7SIVufa8AREMuoorVajnBBMgTVZAetLPnirfVrWe0DZbWXchzlBOW9F3MMBEiI68qWwHw9UVyzGnJiLGXIrPgkcVB8GkU26KPG3FDydM6uDrnhJEdJULvgBOkEHRdAWy9lAWBQtnlZYHRIXbJxK/kdhIqGBoSsiBx6je+drmQ6vEoYwoQoC2XyKysaRSbPL8RZJt5vZJfalO0jqG61dmIHNzu534x8srJrw6FFs6JCK4nUyEGDhJHI7d6IIgTDUxnMC00C4LJh1SRXJ/AKX/l6LOYDx2KQ1MQ2qbDLpngsALIosAJxzXeHbJ+0DapohstNCniG+Wfw3hQuHSc7FuMIUxdIPeRDVLjR8KaMtleEpYUKJshqkx3DyBKlO+wHhklgAgMhbxh0ipI6TKc4nBI6ilHWjq5U5GTzICUlkgBBvtbSR/QhSgHYJlAbc0AHQvSnO+LzQo7nnOJoDckUhk5/h/gPFHRIXpKCHSKHi4aXLcZIkbkyuzw+UjZaFWYat9raKIlolR0tzAGklJQNUciiXnwFhEDz2RE4sqm2y9xAbEmhsl9bFi3Vi1ouOcXxHG5C38FI2xMgB3fFqSMMQwp0Wuo4NYKVsq0HxAYJsAlSDj/mX9txg5eybEPHJ3hDumyhTOHWDf8j0vEghY81Z0fDjjCBLD+W02TrDDzIaTlEk7blEpIKdC5HwF4TIF028BLDXhA+Fq1B4ULmUIAqsrXJ+TlvCJoiZHU7snCSY0IvZQnqFEeNvqRlrK9HjZ4fNTrJMSGL4pF9gAhkUadIzxosE1MbbgCpBV6N7GinW7J6IAcmZbnNiRh35mRHjYaXp+OH9aB22XCn5ChMMjQpLsY3Ecxy5BJyDVHDNUOV8pZi+kih8etxpQ8cV7KmI8r7FEyIa4syDbaKFEDdCpXFu6C2KagkC2Cdblsnzo95kairjSQn/6WOCeiMQqAADGgUWCDZBDKCYUmgqF8uYYzZMiyILgAeyV/2JR1XIvENNWKAqMkSoSJxZPk0JiEI4gZG8oZkWPwyyhnzosao8luK3WpzsiNPWguoy+/kDZQd2tACIgm9UCexTc4uIpcxWZYwiOAzjpgtcAJlauOLOzZ1imVwhwQG11HaMIghG+e5j8Hv5UnSSF+YcVGVBpuI/ZiYSKKW4cqOaltOd3zLyvKtnPIhe8mPfshKOKkbHgNAwCuyPAUF5kTflctFSogk3wqdgqDFr0fA/jxHwF7MsakTHHnqRkA0Cx91pNnlZ1M2fhd4vV6eBxsE5wBRbAMrt7LDnaic8Pgt77+c40qnOGp0gq0nL+W40os5JvR1G82zbTQnOa6Eti86S2WfXEZKI6WYWr2SZ9b0pDU0LhhZTuT7EIsqB8pJNtyGmR5f3JGnExxXOsX2JhRjJ3UqASsoh/GtdClAyE9jx1xlSayZjE6zbT09i2kn7ruCkTKizaZfj029f2xKAMikJMtv3DqQ0uTQbIxS7cE7rOzWkhOW8L4hv1s2ZT0C+Qtuzq1G/4Udvdp+9fDr2at/+/Yz+97Zq/TRX+R4b5uZ/vCGNP05x7HcJ36Ww53uZznsl3ccy330ONYff96u+cRxLP2pjWfuLMJAo6yVyMO61gbNl3gcC3Zmz6LSMUkpwb+3r+zPeRzLBYda56II48if8bscX9RxrJey50O20KQqhTawERE55el4ZPRI76RoMkyHK3uHdo4abuXmJOVGKfZNe1gHfylHqVrCnWTaSiodnd6Sl9+aRGYlKbbCDxvKpa1nMMsWFy870Bwybg4ISXtJx7FO8yhDB/mMBXaLP4SS5QdwbFzPEZFtUCh/ecCCCL2QxSdkbV52aGuVjLYva53z62+uvMijVF/S76WQFzB+xZJ14M1yWWsIh+ZMHKiyLmrVTinO8p6VSjkKrvI6RPHVl/SbK6eoM56k9nKKusnXo0Nfjw6d7OiQf350KH/w6ND6DZP3xFv4sHiLn3N0yJlPHB1Snzw6pP+ko0P+Czw65D96dGhp7E8LOP2JB2rETyo4o85I0cgj40TC+fXzji9QwvnktD0jzSalTcj+L/iLKR5GI+U1YDVb/Rc6OPRCDv1AkaodVRiSrL92QD24oa2DEGVyBKJHi16JdULCSlyFKVgEjFD2ba420ASQbpXJD8FXTe428CDkZrWwHoSBzkwZLBY1Cf+GhMtOYigGsAWv+Hr46Ovho7/I4SMiBGkhO9hLQtXC3qTMjGhmBDCuqeCMuCvCKTmIGwoJUgwlZvZD3gQ+fiUPZEd2wcch38hI1wq6BX/uZYxaDLo/0Us5VQEbkq10Rp5Fl5s/+PrXQ1DvHYKCO6MEoMCFiTA9BDlEYT2kV7xX1mjxQCkKxIj2I2QVwWdaQEaiAvKLOkhFlHfRaeCyker+wPyI/TCg+arKuobsyQUJHLc38uDBkBQ8ElqI6lg2NR3EFLTXIr1lWZDg92Uizr21miYJGxNELRP9xMWUKhWCx8vPjW0LvS+mMHeKg1RGk5GSxMx0DCUTH9ylOjwIsFBSVUC5YeYxvWxYkO0sRYRgd2SPcaoC4dfDWM8PY1m04sDJwUt5ZKapJBhgVH5DhvsCFKKBZCKDNqua5clTEopxBETTVjCVjY0Oz5fH+ZMEPKMFTisKei0Pwmtyl8NGMudjyq5xJxVUDXTL3rSvh8L+TAXkkxws+/pbSSf/raSvh8K+/u7Tn3mx5CSH07zw+Alck0C9UEGF4qlAn4Zhk6Strg0fgpVJkkGREc2yjdfj6zN/gYfT5FRZLBoolt8uadMIXzNSKyD/G2WidGjMTIDJfn/IqBy8FbVjyraB+KUcLDvNb1id4HDaS/kdrCiED0ZIOiPltJbIcLihg92LBg7rcbY+VPRUEGeT08Xy6xhmunzIDV/Uot7Xg2XvHyyDpQBCKHrZ0WWRWB1h4Cx3xpAWCgylxCM1bEcURnX0HdsEYVr6dAfLqswfSCnYJKtZAX0jZ23qUHQDgt2q/BiTPOija8jVLDGj1wNpqGyHKU5zOE1W6BysREQYRrQQ3IDIrtIpCAQT42eu64dIJOvB3pC/cI2VzsIXdzhNNHoCltBaGnf268cIqoCanMrjyi77Qb1BQyC+SftK0InJkm3/yb2oA24v5Te9TrAIfpLDemRk1RVwW1YVo2ArORznIaxQY1jdKHJ4ADloyFRBAgJZh66E+JittnWSA3/CND1tOvlxFqiQkaNUwHbCT+SPJFxYkJesDxppJ1FBMHY54LT9ZtMXdWjwSzrwB30D+kAWP+FYoBYUQYY0JFgy8k40nke9DzkXIiIp8W1Ph2AJyr+ow3okSjDVS40ckSXP1SDqgQw5rWx9NyinnrqkfCZczj3J1hAEktNmIi9f1GG9LFUVJT+ERbqFwjQc0nrwDkwFwiQxSZ0O3gS1JauDyMhc77eNKeGLO/BHG5LEYDoe6WuIZ+itIt6CYBgJmuEMSfjMNJJXyY1Q1LgN7rdxhy/r0OAJDvx5RTL3SR49iVIDX2AhHn8SxUFqq6AjQ3RyugOEXQUlpDw8xgBa20FdORfc5KloWupb0KMxHSiK+oCSB1kXS1IFF80tT1hRss8ZRkYmZVh5y/tff+PsZf7G2Wk2v8lxozkHLmiFdcJh8BYsw3/KosMamI05QBcIO6yDSMJYjnyJN/aX9RtnXw8NfrmHBkk06MGJYdDOqB7c1Q84DBIqJ1lwJ03hs4iLwUSMRjAAswaRg8Hy14OHX/bvtckGvbux37ZT7u/fvSu3P2+vby7X7tL1qYa0Lo882ozKW8Kjbsa57G/jnav7y0vZorjttJNzUuEsyBNFCEYr5xLlOeHlav/juN2vHWfblXKhPtNS9YNz6mCgfs/2gn7sCBJ8+fhns2Tb5PGGOBFaQTYRvhvl6uOb4M5gSEpOKnkrBPvh+vf21qkzH4TxoNvkSV1y23flp/Or8eOhZdmst07jydvX93c393ePXz8Yppa79vZ8f/Gv4/xxh6CcypNv3Yzy/fmbm/vzd+Pd9e3P5+8u6jpbGc+koKfk+dnx8HhzMdyzXaBlr9J/2tHadR/f7O5uMTGt7785vHX7+u/+566P7dO/+e0/bq8xwa5c9d1f/e733z1sRdyvnYrf7H4st+/ub77Z9XJXeKs8Xvz+HsbXt+OGv2Uv4K5flDdX13ve3W+ewqXnR28+7sB83Pco2wwdHCNB0Z2S6qrZTlte/Mv9ONoMKz1/2vOJxhEr0yjvvXt3ffWxK+2nL5u3pUnHVzfg7D4gwJyoZwTm2iP8uM+aNh4tfks08KZsiuVFuXjz9u7by4ur8bjBdhn06hpbjfZ6/FAuX7dyebl/ffFO9tLuX4sRlsX2ry+vr2/2r6W9qzf75xtuZU8p97x9tuH2A/ttD9ts3+/tYff0CuSnDdE4r/rIKdUP/8Sg8+vt4z3WIX/g6KtsD726Pr8dd/e3V+8fepUOffRX77Yzh3j/CvRycXluzfnN5f17m3wxGu5zf3fxA2hDO/tz+/5V/r1+nd/9fHOwwO/W23/72NtPb0rO68zv1l+8VnY9v//h9uY23P05c3x/dElKAl8XjJi7rm2yW38eLpAp4CsXB7h6anrh6Qc3Q29N3o7S3j7g4doU/ejqf/y3B9clAkjn9b7LruD93fXNsz4Jkh9Q8hk4m6TymXXOoYfJdXqd+zxGuOeXZ5fiWUS+EtWyjW79sAYdurg6h9CeP2WHu9v78ctRTjmfF5fjIeF8+7R1e3/2v/eE6S+//B/4ZEGA')))
result['historical_bpe8k19'] = json.loads(zlib.decompress(base64.b64decode('eNrtnXtvZEly3b9Kof+yrRl2vh8aCbAkS4LglbDWriwDgkDks5seNkmzyHloMd/dv8hbfM00e2a1pRXR4EIaVlfdypsZGXHinMjMur97c3V9+c24KBdtvPnT3ZvflG9G393ux/Xut243vrsa12cfxsWNzl/s9uXD2P2vb8fFl/ub78/H7i9//dfp67c31+Xswiv19VuzGxftso9rXvWxvdJ8sLu96uVm7E92f1n24/zsYuwuLm921+P69uKrXbu9vuYGRu3a+3Lxbuzvvmt2N5dud3lx/v3Jmy92b/aXt9dtnF6P/e35zf50/74YH6TLOrhpVJo969LcDHlWF0pwzprsS08jWeNsVrkUO6u12odkpsm9O6tUctL4VblmcDfjek+Lyalog+Jtul1443dvrsc3Z/uzywu5XzXRuJDynCrmkZsxY3bv0hzaujqrycqGXqu0u4xzdvHudHxXPlydD2keW/E/GdH7ct33pxi78/a//G57g5dq9ejmvdxNurDZ+C2dvPlyffmEl//vdtzILer3N6tZE3TIMYet4YNtsk5Wd894S9GmR1t1CdWmmIbLmCi07jvjcFPHMdyco6TWTEqxRZVGe2z4y2/Xbe56L/8+3bdycbG6f/8+/drfYKpT3Oraa8NnITs+kEk95Y3Ty+vTD5fX4m7e6cQn+MS7s3p+fw+dnVr2Z2Q3p+3y6mzssWA7v+3jzj7vy/796ftx3i9vb9b3Hl+gMy1zUWltXN2MfroMeHpVztYE6+i8zj98sXswuf4Zk+vnTB509E9M7o0xPfhYdMGwxVs9dTXGKzutjw77exenYl68yinHroqJubY0A69DOpLJk3nG5Dbkj5tcmz/M5GYZ4nmTJ+efmNz8jMnNcyY3xrknJm9BJeOUHiWHUEYpyUavdCoD2/fcYvO+NauGM7o5XmZr2yitY/1m+/jlJtfRxJSeMbl6xstdUuFjJg/RG/uHmDwQp58wuVfBhh/+VQY3zke7OeDX5OMbAezdXY+A44sv5VZfHm71xXpH+rWTm+5Ku77c73ft8mI/2u3N2Tdjd3UmJtltGKZOTr7aXV7T2kU5X5AP3rWb3dX1mGfffbW7s9af/TmWeSsm+jNe7P7pt3+Tdmtmv9r126vzs0aa2K1R3H9lT5644Q3uVfa7OqbY9A5ZH2WBoF2y0/UGCKo5/TRA3ZyzON99tFn3NpvSpYZBktBOtTBCDS1m01Kz6aHN80u6IXd+1Hoz1priXcnG0mbyveTSsh5NFZuLGW6UHmdsZliTSh1tzKjM0N2qUEtcLkzyM+rrx5nLKgDQ5gAemKm0wh9jaNqQRFyfVmUdiqO/bpBf8NluwQwTnPWZ1DXuWhWLP2oW6HF6ZN1GBvm5/4hDp26bVxXnn9EUrSQwXMqq+amaJXhLcTkOF2t+3NltBk9vL7bkLG53c307Ht/4+UuWPYN7PGLjtQrRmapcIWDpGj3hHauaqj0Xq6aX+GaiU6jEKxMZPbNHRBplpWu3F2c4x8HZ791kxZm3UQLq3pfuP76POReMhMw35fysr1l+nJmDe/LRk56r0XEpR8avvWllmXk/JlnStVirKi1BJQbwngq996nMMRkITuCDVV5Jz8d3dPo+nB/dGPD8yccPty4Gdx152qH8TIPJGy4wUbYYnMJ31/IA/ho5pWk3UjW24TwFEsI/5kriF+Pbj904hvSjDx9uO5KJdnoP3wklNw97wI3mTJKsWuUT1esczfAFm5SAjfWTcOh2dlv7iqjLm3L+0RErI8Y+fCL/enxnnWJtJkdDdAQ/NQ7tcVqmNwyC20zQe6bii/c64uLe5eRdzcpjF7e85NAyLT4GiWg9sTNGLIREhealmXRSPWmFOcGL5L1VcTQ+DKQN5YbKGR45p41zgcQT5D8vVwdHPNzvgNvPXCSIerhiQmQfQfuAcX+zYud3b35qpgXpjw1UALjYlPI+91D18MGronpP+KevTjwmWz1GgnWkpmeAomYfGoaKsBEZxzw7X4x/deQO+L+868jJ/92TLe4x8cfjOQzi8rzfD+GZS36aemRMu37GxRekiPsc9Dj/PEL/eX35YcsyGi3wLX+/IjntuPPdxW9Xntry58rkvFwjuxvLjpyxm5KauOoDpOKMse0u527rGxd8ICOd779asmQAALdF3uVbws2RIbubW0TLzfszstHtxZsf7j1bJkYAcI+COSSNq/M7spDuSMSHUS4+fak+EYbQL7+9OL8s/bRc9NN7w/GKxNsXLbEnSflIqPmQ+V8QtDu7uJFhXmzX3g1+ibVDkn6QaHf2LXudvmJgX4+Ls39D64ldLi9o406ErTFy23n2bvnkfqx+OvOQevniYzVTy017v97dnF0McG+n/c242vz5cPXByqvLh8/+xayP/Pqv3q4z2x+r7j/iT3x0iRJ6M2/Pz08/1uJzX+ina8bFon5Rs023np6X7zcJKMM8yNCnb74nz69cI7edF6f97MNi24srjvNbInkO9CxU8uHjQxT368urA0ScbOoEJlXuYuM335797a/+afdffnPGf98JEfpvO9HK5XonIxv/9c3hlvWs7B9y7MXl9YfTFVcP751fr3ugWiRGr27OPsgky03+opcP/ywNfTvO3r2/OWWM5fvtYrm2nZ9dnUqLyyVX+Lb3o9+er2//7s0FTiXN/Kp8qL386h/fPLpC3t/6e0qUfrgF896PC3jtnjff/MQTDlPz5nDtYwdZniffun/X5e3tq1G+Pn0yOrjh6YflbPLOemsZ43SrOWwX6/HlYuAE3w3qQlqc5XwvppI7QF3OxHoPpjpULN6s6OrjOxT8iqk/1ycnv/3q8Ok/Czrsd9Knu7d+u701zy8vr7/a3dtutxohyG4vbvYr1s6HAMyhLvJmTdlpmcTx6R2LPn3Pq3+7fBiZBGS/+X6Lbu5QbqzZzHoI4jWCby8eV2iEW395Pr4Z51Kskau/gdxWCc2NchDhN0T96Vk/vZxzP8QMgin7q9HO6MTZcvbfkYDuFEm93B/k8livzA+P+/AoR6HUpvEVelIH5KCEAmOFlyqSVx2QPJ8UcrgXGFktAR4Yagbdigk+17jow4fy3ekVrnDH8IxyginvxsW43kK9rUSDmHiKQGcXhyiU8Zg1jQ9vvPn19fjyV//ArLTrIcUtEsP//N+7VpivN09Hs4nChzLY4rD5SWVHEhSwey03fwhBCVQ+v8GIZ/92H+N/tdW6dmsWZAq2zLO7v1KS1Up4W3tf7Q6ocIsD4Ub7y/OVmQSNndn91a//afeP//C3X+z+T/nmjM8/lJvrs4YDkj92Fxcnv9rgQwBjrOwGuo/29dUlKWMniWb0J8Wv0w38T78hO4i1tdJmGfzBAKcT1LpcvvaR7PGnT5wPd+8kW/rzJ9uIb8/L9fdO5UDmWaP/k4cU9OWdQVEdvCvZ6qut4virv3+UwDZ7VTTa1/snxcVDShOlThI/WOpxH2Al+9v9o46gQ6+vbvdvF+3ejQ919M79d49qgYcSoKRtefduGpendAmp3d/9j3Xn8PYvv9r9za+t2W3Aun+7kHbZ/C/+/tdPHKYJQSrvVppsXx+gBft+iUZZUypRC4aUd9tEAikkuB3I9W7cbAbboOQL8eBVDODFQTYDYCDBD4/vd6APh8rBKkY+pMrzy3fv5C1hHA/OsWLN2HgSdcohpSR52B1KFRsfuP3wARtur1FbKzLkU523+HiibqR6/KZeAceXe3nngoS9uOEdsXHqJGhtLUAviUGvStLF/lsm4fRKAnFdKRfGE+NN0tEnbRHlC9cPeCAVELHO/nSzyxrqBldWh+SeXHl7M9PpXVXJeRuCuWNqzzQS0kkwWqrHni5EG+6uf9KUseHEIUi0qGurERwHHBORtbV8D1jyNoTgCvZ89/WDZT4FZSsJvru6Pf0wPlxef0/6E0A3MZ0YbVD0wcgtlyP8iA6WvUp/eijSfyEOc7Gn7f0X93V7UHBjPV8sR1yvJSTEDf/iN2S+Q/Vpv5OJ/GK3Je8vdhIkC1DuLt5iisC7eU8onbW3BBB/xeWg/uXdxSX0vu0fRPGjN3/ksFst7MTZnJl4J7Q3mbAKX4dKwMOkruLbio3N7ZaRL1d17cMHvP2ZK+2nL5vX5U7BLNqBoLfaGRKcz84vAX2Lsc/aaaGNx/JD4HjhtQQjsPDltvixXb1ZFIAY3432VmLybSvn5/u3Z6ApLOWtWGFLfW8hFFf7t9LexTuQ/PZif3slF9G2wB33vB4Io3vySSr4flfH7tvry4t3m665Kvv9f9dvftxbtIhwlo3ebR9spUW1DNzHlNrb6YH5aSMmfZjX03F9vbKBtuvy+25JbWHVxb8nvX53f5mRqvuTDjy9xcPkrRuJ12xxXc7OT61BMd3uH64JG4O5q0meinX3p/ZHVy0veNyPU2FR24h/s97+67veiVP9HR06KLT793/YsiR2os+nV9/fvBfqcXcDYm7JhbluT+b5Zvz4w+3N63Fzew2yLD7/6JIYD0TzcN9DN+8uMHdVqQ1pHvl33mjY6fhwdrO56/1d9TLfSlQb7AhVemgy3fGc9fa3zOcq0Fzun14isHRabzuZB1e5fGhAHQD9gJVPMNoaZ068LK6FTAJx0svHMPf0aq11VCfJGB3ATpfXfaVDaEyd1CP1K9Tqh0ep5fSuhCF558sHUrjfihcHBXtzfXm+TfaSHOvVQ3HoxwWXTQ8+SlXqJJOWTLYpWkN/4rLrncbbkvKzaScE655c/yRV+KCDxMgF6pV8+tOve2/CSQjOmRS1oguHi8V+9y2tPprIVdFi+ZhdDk8wiSZvHo96WyA4XXLsfsQPXbz73t0ygllrFI+G8BDk99dY+6jmtV9k4v4Lj+tWuXer8XfXShg+2Tx6atmH3A1A6uaIOfjRtB9jzGZSL1yi04yoheJnePMkhd9h1ykRM8++G49uij3rpmaeoNpaHf0pqIWfApX7CZyFJbYPMSzfeoxkDzcu767HEhX7uxhaGHxv1gfwSnegcUaMTQDu9voBFIxeHSCHCHL0Rynpsf3vrg6Py14r1d8VetYM7b59f7kfj7LD2X73KIEc0tEDqn+1O3SHy25IJPBKsOkSnV1uRNg+Tj8PfXyUfvYfyT/oyzPRlHeJ6JOYarx+Btq2Tz4NXf4ns/xg9vhUP95xDeJPVDHkErbrghWscCupSUlAWPS9eW8/bLXDpQjG+feLc/d7i7+7vry92n+xxlx2sL7dnu+jh+6J3LLAV4tCvf0xUdrdkx7Js3e+9FBclSrpAo9/eQNfTmubQ4WVhlaiGl63FIo1aVZX1Iwt2JJz7Lq0bubI0QPL2btgRvVyB90t4TZzb9H1DHMMo6bKPwgVNUJQvXhbhkWjE7FB1arDtD0aBaU2ebVhYm6loJ5MKrHIJoxR3WhR+zqdlJ9z5NupqaikTN8h0KHAonrzdEitkZppaLWHplM13nVri1ERshdDqt5IUV84X4w5D1+iLTM5H/ow3oIaeWxtDGt8LdmgVrRGSfjSwyym5uwCFvAzjuycCt2XInZilHHaWlzoIa1iivXOuRRqDjY5U52vKruZdUWd5M6wtGm16JS6droOR4dKbz3QL2ZhK1rQWhG7DatHtKOoXLxTA8nCP5OtmYlqmQEWzSAD5gk2p+5515u5rZu4NCfdllUIgH8qG0imLs4ZfOxtpGrDdC1ZH3yp3ZRZXJRVn4Zhku4LKN3IyvVpwkxFe65wXY9WBw2iEVobKtlpG7MXWjBVOyBWmWH7zLG1ttrgjaTanFhuJObW6OEFuxOXjImtg7LMqEE94WExlVnxQvyjY37r59ZGaJYBOZvABxtKSLiLeIRzvs9iq821az2tuK7SrUXbg7fTa11bX6s5cRT6oeMcPcaeO0OqhdwREXY9FB9rzSWlSDu8HMSEYuIxqI1kRr8WbpLFcbCbRLhXsj/Axpy4uvqijYLPNziIJUeFUmZUvic3Rif/Bro6FnVOuak109XryTwMM0vTLUY3I9E2DaPxczDLjv51srZimkqevpPkwprbzHTpWgsDscMOV0ZEKcJ/ssJATHPJ0jdtpVfYUcEijO5qFAKk1mVTOMlInVidOGJLvZFB9TDBYn1Zx40zcRti2itiHmOrGWBlJFLXmOG1NynHQmR5U5zPM4/gLK49jPi18ZAdhVeBoZKOqx+1+q6r9hJxdsyQ14pm0S7EjtG9z7rJrNgStCvFGdwBHHLVtcl4Sfe0GmqD01jLvCSp/q1+YJ1k4kTbuTxijcFIjKs+HAhjbRaowLunG6riVGpW3VMCoqEItvhtudrq2RGGnXj1WjG/lYZ8Uk58rOleauquxqYH/yl2ujBkeNmpmkdec1u5X81SoZQF/ZGnYWKTJlJb7AmPNMNlB5wMTYu66UrQcClOh0HqwqBmADVjawsZ36gzNNWsKdVjQFl5nyVMbWsnnJNgnStYslhQK4I8paw2fBoWTIi9GmF+jKNCklydhLNWxeFlLhPqXafSHGPiRiNDnbVhuFs/AhA6c5r0npZUblaTIJTOaQGJw3esR1FbvM60gSG4n8lEYISsLXtIkGVZ2nXGulqt0qYkVw3z6AZ+DsxqRwAo25prZkD16YUCoZj1mpefDtl8o0kcXXc9Nb3Q3UseyKBp6MwVZjaYJFlAn4klDeRGkDU18Pk1FkFoHa3zQKedTdtpBjkaVOYrA09j4IQeHqiSa86WJKvUeL8ChWJY/Zh8RqsTj9GyBQK4cUwqg+OvwayWOHYqaVkBVnWtrebIpNMUGa+++dcfnq5WP7NQ+1Q4eO+t5f+wE1hrfw/lQEIJ5nnlYKyopU9IB5zB5pMcSKCkXVK1/4R0EJFGUKsM+V479X6pdDgM+nntkNZeqU9rB70Kab9EPATZeIkKktUCK5t3mDXAAocLYRj4hq9V6epbNWQgcniNgAzwK5MY8x8oHsD4j4mH8LEqx0/Vw+Kdj+WD/Xgdxf8CVaEXL/24rAhBf0pW5Pj7yIrV6c9KV4B4z5VMtsXDn6mJ2OeVxSp3PassLIEGhRVXFWkRtX/J0gLwI++Q9xWUvkV4FGQ4T7ASlocmTyMZWKDwoB4z1MQNWeYzDUmQbVvUQEGO44AIQch8DsiGgPCAOMfGS0de6FLwQbrkbELWsH0YnEsENKmxmK2NlIzkBPKyN7AqKGFSukgCU8IVyByQFocgSKvl3N1Usru5lerSSqdw3dAbtC01bF81VMzAWBBC1cJPNDSDjC1cEUZqUu30oBnZthuzgUWsNlIa2g8tyW9lGxJvJxNjItmNNKwlEcp6hANMG1kUDutNc6YlgHqTOCqTMgcDCTFw06ThoqRhWFNLUBBSl5ouVY10IalhPF9JjHMEhmZrWFRJFYgnzbpJsiPdVyiT9jGFTHekgtIq8yscnmFYqHqUjYsD9DMjQOqX5DuGbPQBTuQhwEF2N0Nes2sFJovdehmjImxgddwBMzJUJ3ueaQme0fzBpnBGmTrMjfYh3SJVukEnefzH4GAljiT2qB2u0jTs1GYdKnpjCAXc+lGwyMjcPYUBvUekJjpemqw1Q2rxqzqrTzMpuC43hGHqMiGgNYvEOJoEVhNbOhREQ5UiwdA3EM0sOmEq2edIoOATOY7JKDEOLbksC962lLZ83SAGPMywZ9m2bNAd9Je3oEFmEjKw+lKiRhUSmV1FE2TDs2nBpSiV1k2KG4M8I+mjl1Ud8DdHSiwevRHM9IQoLFm2lbqgmGIfEFWQOpNTgk/Wo8l51LlGurfBl+jwEK2TzJBd3Tpo0XCeITpEi0tKdr2nNHECxydgfXhRJQFcJ0M7YBMW0Y8QprlmOyhiGDOEGQdisGGG1QDBhmIOrmr079jm1upe3ayh+egicnh4wMjFYi2SHoAMsqURrl1Vo3GFghlFNZAONapMWXQcyocITRIzEylvsWqip1UOE0zZ6Fdk4x8qiezkaQ09TxAJre8OlBrHKm/gsKg8ATk9tGNMGRANAgZ60CvQdsL2RVkoj7If9AZXlHDw00LjVxs4G1DVpocegw7IqmoRZnC7wP37ENhogA8IY4tCA2dUIomg6lKyW/hhpw9EidJoZOYSv8hgeJLcIXzWKNyjyaSpBgygMOlXEFENlEUfN3vMPgYSiyw1mRB01GQ2a00ReRgBCuGTMglBm1VX8GCqhFEcsnv8iOWe4VUyKLkQpQwum8s1YpjZw65+Mt2ynQYXQ9OgwhSizvXgZidfmpGPVjLK6MZUxCUbaEdSGrNH3Mu7GbpKIHtQkRspMoQOUrIH+ZBmBnkdytZGX3WcpohSZnCMAIB3lCdAk2uBUZBsOjFjmBJdZdN2CCSLFD252dpjla7cLHJEy6SAAFfM5wCyvAgiEhSKl/l0zXs6Eh351M8QWvWk0Y7ItCm/pPKXhAm8Aa2M9iXKJTMjUxVsywP1CNZqY4UEgOdZjD0R/eQv3Jb00LexJAdyBB80fk7+B/0T0M8UkwpShEokG9HAOLcPUYpXNoGWOCl0gFy0tQG7GoHoggsJjiYFbk8iXbsMdjeDj1ugBLEdddFMmcoDaOIGg0hdbdDt1jRw4boLUnKCYJITydHNNlfwT9KzrbIxWzqZCUSZn4KRUexma2NoDBkBVLASlhoYL/CJh+P8JAOyf2khVrDbxkSKARSB2yFoRzBKGyT3ZjBlmoCEELlhBnE//RgI9ikwnHsnKfI6L7QC5KE3XQvtnCtXhoZjETIR1+L+IIiEIGTUAQbkO4AEJMQIHhMzXO8AoLqKlID7diggQp6tZFajIUdOTngIgWXqyfEwXiYDGEiNMPDkQvIj1FaReBoOqLo/WnnU4OdGaBbopnpF23dYGfHKoMGhSudUgEkw/6OILKiAg9SgKvm39BdVYlW4s5ZdP04n5gvajltkD18UJgr50aUSKPhLBukmbjLgVxBckihGelFl2jAm4VdDlIxcl/IKRATQqaFtQQIPyWKzQm110r4sF3st++sauqkcq9RLCm2km8kgUSpwFjSbeBTAhO/BSbqyGdecqCoLwHudQe2I9zWj3NjGApkHregpHBwVYuTwLDpqCH+DbJICmHZDSDQtMkiKwXAChINCb+jFHTKBTJxWOQbTiawkgnCGgdOjRGC5TM7w4k+TcOukBIQDIMREe2G0RytbA3RYHoE3UJNC3a3tEhF2qNhpGSySciix2gzRTIhrEmiHbYik21S4NR1ijvLhDUwK2tBgq9AB5ktSIrNKikxBJahbAjBhUOJ0KCHYyWoDjCEfGkgFrKbgoPCEQQLF2fgLCFkEgYgOLmGM2TIsEubQ6OVq7LFK8OAthK73SggB3IzSBzkDxQXkIRg4f7PUUrUkzRShncheZEMsQ4TdagPeXoSBdlQHsB/R89ArWLMB/mCV4vpWSaWX9A8+4uvOFdnct84ErTI+rBlJxA2hTOC2tXgfqgdzYeFECAVkKWK8tqYas5sgerCX3JyltYWnFW1iopw4w+rwThEQAQ4NvqJYtTBModtyuhCZBAPCDyyEB35OcAW3tQGnMXL0yxf4dLddZKDpfoXqrAnejEdFYR1RlgbkuFUKKyXVfliSOMayhtCrLH4VvRlhHa0WbGN4ktZRPyJeMA49VNLlHuwETZkdsv2cWxtZOALRADcg6cNlvKAMGRVXLwaGTxw7KW2Aq7MlD8hgIfI8SDGPtrxSXea+eF3H0ZF+VdhYYk5hiKBX6ogJOGqVZRHUh3LC5gukuQGyaeOnImYzl5DDBriASSrjNcSf7HeGhpPpi6okf5g3iS/hUGEMzK6lDLD5GMIZyAEVID8wbgROWVNMcm5IS9xOQD0ZN2CcTsA2YU1InfPM3uZjCHwNuDkjcArDLqQ0cEkL0deQh17Io5qYmUQPWFaF9aCiVSXb+c3HEBwgnEI1AdxyVpSgIp1OOF5nXgldkeJkeYscc7lIWQVRVOl6I3dubThe9Ww1oramWsg0yFMynIEZDeGhUBw8gGxCy7iOJv/CEqaB4sXFx+A5cpA1QD8HDNCZDIEASWFtDq8kh8o+HlJKiLKuM6fASoDeiJByh7iFJ8wy6LrvS+NDXRkcMWpAMpvQwTk0zR26YygkVoialQOsDQ+bm00rGIj6TAB2ASSbWipYgU12BOYbzE5+aO9dgKnXCbEwRBjdJ0uYebQlPKYRJ4KBGC8/s9CqHHStsmAWfIYpuoatyE6+kcBgpbIZygLGtKlc1S9pGVDaIDZ9A8pggoYYg4ohxMgSYDrJlU4zqRX4grFZJTeCVeN6uPCWs1sR/ORyUiFqCZ3mERJN+IzkWykJ+llhnz5qySh0h4YJf9BjFLvqliC5o4/Zg95VuCcYD3ZCJ71LTeQ9WSt5qUNFKTTIkVTXELRwB9XC4jDdSGLKwkcdFlwxM3FvopdAXlGEEAHnSEz4AJoOmYmLlDnhz235B5kBRJfqFZRUDoQH2a+WCJlVhIL09jJgmcQUfNHOSqhgHBgkKEmcrzaASeEmJGtMnwBlIqF4ZD+5HPElJQ/0L0QHtBZSoxF+6GcPV41uOwbdyRjooYJzI2h17EIY18FkGGaWgrXyTFYAPidAAiahpgwAgU+iDrZ+FOvIIEAear9JZJM/QBNkIlmy0QkAw5I0G+gmCQYMTeJ9+KGPm/boXTFreIQeQnCM4eqmM9hQINmmIwInhFJQHvpdI2Aa1RRmGSVE4tYGoYT0gxmBg1EcuUHBENoNie8Ej4UbWanzEeIQVvHbMqrQ2rlpwj5FQiEamACCQqAYxU9SR4HRvBRiIVBGDvui9jJBy5RO/ETBhIre5hYRweWkOUiQwwOnVBugcY78NBHjckTeImrAbKFyWY4+SwYJqHKc62jL5shKgBiOhHySEj6kpTg1+DY3J/zkAGgSHCxSQm0ocuQixIChEk+LSw06C2Gtos3hgaRsZBdUBIqP/ohpVohAcsLJKkhSiiU5Fyk4ByuF/9WGhxvBYMWWCooF5qSIX8pRfeDYQl+niG0thMtLlUOEVgcuSwAptrEkp+VUlwhTS8RAGgi8Qm9IoNwQaCUxRZJfj0OW3IUkaccrFC5EYGsDz9cId7gQHeyysC1wQUIk20MDUClTftMgkjphfBV2h8aT0919Tue2rQiAV0vw+jiQerAmzMOUBLhRRb0rP9Bjc9Quv00Q5VC/xQHpeAPFnDrWdoYBrEHlFAZFYjW6Bwo22PIgsmFRlQREhlbQD1y6w/rpixSpYTTI4ZVfUK6d9KsEaCCUBhrU8WIlv5IQM64LszQZdWfFbEQJ5tbQO9g1VtnODU455k4uADiYWN8ARlqQkpojc/aC3gPCMF6D4Uotv5MB4KxqwMrjxnGPsjVDjv6pBMrKjylB+2V5YNYiBV9xG6WkNo8vSDrPzE+GowD9HcmeDjgGdybgTEKTxNGlfOZhiIqGkAYN77Cyw0PgRlWonCuS+YWRODBvFru1URR0FR6NLMpGZsjINeAosV/QLKueYXOAY2hgZtX/+QbJGzzyR9lmstowqwznZSksT9EPBMosxK1FsXuJIeJcshZQDjX0ChiHUSfx+bTiZUJx8G+8W4tS0g1W0WUZNMLAZ3dS/CGYZRlQUiEXZJiAkt9Jgaxh03/9QZbR17np33/DPBoiG3gWo2JS1gGoP9aOeRdSOPGSVhFWOq4DEs9se5EfjMFfRACB0dm+bpl/2PXyo4M8D6bZzuB+bB9MfuZc0Ed2x/yCLS7roNszG+ej+dQOl3Xa5xfvcImf38Z58+zGeffz+1s+sXFef2p7iztxSn6ZCN4tB/RsfNm7W8gJEdFF5vakHTQPFFrqOsgICNeYEGbyVlZScuoNMo0AVV5+XEpBTI+2i8KibkNA50uJ28jaQJs+I72dVEI6oonMIZoftTuEzykR8Rr7QjU2xasdCSLLGnRIKkg5qnDHEdFFrpKnLbRRyTJW6x6l4GEF3GCuIoAt/fPbwM89jAgrkqLijh3hDp2UNUzo5TRSwOLm0Y1sUJBz6h5g4zCLWEbfVutfN9//aPURKQYpb2gSfF3qRbIYFbivbE8SxUrrCWKlIvIXPe0CrMtUhT1od80t7BbP54PC8BVX56kRjFGEKcpZDn9LBVcMpqI2xhTYrkxctQ1tu63YBSPLSS5KIR7hlOTX1vz6tU3D/KDyRD56pllqwpBZ5GxUSaimiNj+uR0kQGgnho+DwAe8B49kmr2X1TsP9yZqQ0yYfBp4O5y72imlMqm0NL1VREQGdvSIslo2VeEk1VtR2yr6Cc03rqG58EKgKctCZ/RS+fbIGhEm+VgrZbgFtwB0hOOidcbQTHYhpgpEHvMqvAaC3hA5AUhFjAaJhdGk0Nn0SzqMcJQK4kvZwK8sl2dR2EwrcQlOYE/QAxSpJhfFnMgGTvwNX5ffVkTAylqVn1IQelGHALxMKWi16qwdGCUOvG9dlj707CPL9qPK5HSjFc2DruSh6ECaEE35zztIoMigShnZB0UK9X/MgwTaG31CRk5e+/CpYwRBNmuQElJEfen0eozgeUFl9RNFtf2i7E8UlX/mpxY+cuBgkfPf93CyXj938O86RrB+QvgXi6w13M9KZUm+fP4cQfxZnaXj80LLh08JLWNPDBrLRniG/JzBZy+0jnMU4QjHCFyvsLcYZCXXyQ+DxmSQSbmmTIKHhaO65CdL5cfgDXIEITAHCmZG+Qk0d7SjCBEhNOAis8D40QLkLS/ioMpPVAhj6jEhtqBhYZAy4SVOg2DYuxnry8s6zgDmcmFJcCnuojLjYoplPUPN3mwKi5DLVjw3ZdO+wRQN2ZqKhTAd7UiEqrxdmA40GYTYaaQG8g8pgEZWEDTYEWIQOSGHC6YsSykrG5AqIiCn4xUEqtQuDdQCeSsF2SmF3tFhw/KTuvJ78tBjOVSTUDgD9h9gbkmsHvK2lfilFBVezvEOtGWLDAQ5BEEmwzfIPtZDD3VZI0FlZkQweg2GO6TDw8uaaJNfF5uvRzNej2b87NEMj6qUnQsJGZZl2wf6z0ihCBNock3JctJH9AhaUtJZkwXQKltB8f2yFa58487oWz5OpXiHzCBZlCo7oriX/DC6BUKJEG0MoY3UDWriJaMc4uUYxa/P6XjHZ3WsouHQOAMh6isIY2SjejLO4k8kDXQs+QhCUMCuJvAum2oJ9jqB1xw+x6MZ3LEE2Xak13512W0f7OhZwR86rdZipfTiGS3GwkbEQkmyA99sxcAXVGAlpZPKgQR4Lv0wQukmnKzAYWExkOYmu5oVwSwFJxsyzAqiNULeMOgoRdpjHBEp0SfR1QZK4X0kqSTjjeQ5YKHBomeMQgNHmznIxkdhs/h0V7S9rVe+lCMiRylaH+OYiewxgs7irn3ElqB7QEmKowR4z0AKBhhsCeCF0FwIjPgTqQGXJqUuTA5kfTIRYwZ7NHTZDsmPUiJMBnBCRch2bLjSXAxDRZgYk4CEUNDefqwCvNRYqp1Oqslk3p4nrkM+lc27DBBARtkwGvg589wq5EO2wEHLSUIgw7GO3UAiBrynT9CaOfVyukVBtFGgWdbWgWydhqoBeVaD7DorPQcSrexf7tujIKLwHpJvIQqhW1kOTzo3AI1MM9BtsgMyDESz8iacaq4qPUQbaVOX9ohLXoE1AgcoSjAPvoG4lcAapJAsslKkEElMyqhDDl8jW4ecJt6Od0SwG7pedPJykNm3CQ1EJ4u+Ju/iH1ErORgBVRqy8x8iSW4aolRM2bbfHmNhBGSZ9I203QLZDs3T8GFQC0vDVGSH/ZCNr/RQjjqhGQpBD3+mL7JRZy1IaGFveDJ0gds4xbQAYxXAAUwa2RFr0ZSZzK3GeeVnBXBgLG4hX/XzW6CxQVAD6ITNVHnsCfqPvEqb8twes3aw4nvwCIlTCx1qfYBx/JsgPtoiz0s5UnWM41Cf01Gml7JwdozjUK9HmY5/lIm8Ig93hFgOVGFB4iAQkP+qTuIhefEeGDxJiZhtKBRxomw9np257HhHmY5wDOn16M7r0Z1PHN2pstXViw8qOsMf+SEXQFA4N9pFxiabm/G7SDqXyr1FVcuj8pAKeNmLOv7zUo7ufE7Hbl7MkZnX4y5H32TyemTm9cjMp47MQBDLRFkS2Ig/nMU6KVkoeYipGTavXzBLDAheCKVwRkqestEeelaqOt6xG8AZvo909CQZmDGS36PX4GWIWFkwA9qtPJ+7w3PAJoJhSqUSkK/bw1lfzNGdY2yoej3+81KP/2zPS/x3nP+RwTJFQth8TOaPef5HZXuiRS7lsJ6F9Px+NXmSsvXwXINWx3Nez/88t13N/ej8T/7IBrSPHgn6Qw/6xE8e9FG/zx409/kd9PHPHvQJP7sBTX3ioI/65P6zE1wR9RdjkgdxoblfT/r8ksdsKCOPMYcXw8Jga35tQnLWw+Xk526rUJsynYZ+V0Ldyi+8wJQL0jlthb7P6ZTO6yMyXuYpHWixHllUW8EA4mtWtl26YRU6T3YMynYZZGCfUrtCviqheakg5XTfFnA+qxM24hurOCIVn+IAEFs0PiujL7gWotPLxrGK4CUJr/0JuFm1vcl+zs/tcR8ZWp5lZyVBh66BZlepTRMZs2f0RKHv+CbaJUVwo6fqYgFmAIuIYptHW2wg0LkrlqwDQC9I0dbgcM2ZOJBnXeSznVKl5T0r5W4kZeV1iC2Noy0UHOU3rSBsFWYekGGyAiAFXWYNVwU25NHJqXhiRVbEZZ1fNj44huIMM6lw1pd04uizOi30n3HSR/aLetl1rQF+lf+YJ32IRqvkaYMaWWTltM8ntJMLWebEyjJOeD3q87x2MunpUZ/87LMGfySe3HM/qKD+PUd9lsWeO+qjPnnUx/0+MmsN9zM76qPds0d99M8rrfSJkz7uU0pLh5NM7nUQ23XSx78e9fklR33g3xqJ5WvyqUeUSUSC6Ai9j1PLelOFhQHmPc8uVVuopCXdNZvkrMR2xMapBoXSCAf5lVVYSBX9MmTD7pTdszBoUrtU6KpzFTnhvQ6qxrxtqP/snn7yWR05ej0u9OS40Gf1JJcjVFkUgW/UzKIS6dOkrWjbkFNCQYIXtPKjS1Ehr1KLR/+hlWcnkNRWVXh9GszrkaP/8CNHr09heX0Ky+txodfjQv+px4WO8DSYz+i40FGO+hzlmM4fXsU/yjGdCCSAtwx8YEbSaCNLMyrQ18levGlkkZpIIHSB6QrWhqr5NMl223G0J+y8mGM6qQKt4L3tCbAFggiuCEuRWpqWKhRIgMTJwB94hHikjaDcEHT6DI/6iIitTsvDq6CVGBVlTfSGIco3w3CgHIAeOpS8XeRcQCF7GAQ1cnbb3v5ijguheKV+KDtgMX6SjfSYnmBz8Oq+3DrJBrlKcIBSyjsU7JC6AHxiW5HAlSFrsDnSKqmvoWo1BFU5dLaGegXZeKR8qGihgC85SGKFADE4lw856sWser0+gen4T2A6wupbxkOxOLw0NHIGCT0pOd4YSm01IuFBsWBkdzxDk8UXUHRIicdJgcOZox196r0TOqjHLpIbegGAyC8SQBjQGXPC0RAJ0h84eQKPNVcQK/QclZA+uydBvZAVzdejTz86+qSFRTH5iniTUEUpk4FBl+T8KstoZUW0rGMLpBUDrZ5CQTEkCLUdn+LmRCjYK6f1SFV8d6KkwXF6F7yBXZFfCPRV/IVSuDBkb2pMIdX0sp5IdYxjXEdYNX89gnX8I1hH2IlwlGNc6Knaob7NRmQBaC9WU7Jx2Ygvkzchl4QBOMWcwuA1iC3P+AKefH5ZR8FeyjGuFJUc8+iIaMnOaLIiBmNchCBfJX93EYHkKfQecyNnuDKhk5GF25Hnz+opTmZtl+b/x/qdE/5Jt2Bc5JShNKgRVmGc1GAgJlIVIZOiNHkV3eH3dY9xJE0Oq0rRWEttClqCUia7AYhQ4SBrOIjNJodcvYhHlTTmBM6MrInm7QmQr0+CeplH416Ptf0HHGv7vJ7iJPq5y0omaMS9kcelyNktlDExAqOTpfiWrBRNEJxTqlC4rpMlhPGyngQFrOUpagPNhBPC+OU51tmAgcL05dGhU5aRYDgD8EZuW7vScyXnlNenSX3ex8l++OH/AxsTKpY=')))
result['historical_bpe8k20'] = json.loads(zlib.decompress(base64.b64decode('eNrtnXtvZEly3b9Kof+yjRl2vh8aCbBky4LhtbDWriQDgkDks5seNkmzyHloMd/dv8hbfPU0e2e1tWuiQUHY5lTdypuPiBPnRGbc+7s3V9eX342LctHGm7/YvflN+W703e1+XO9+63bjh6txffZhXNwY9dVuXLTLPq7N2z7WH+6r3b58GLv/9f242P3Nr/82ffv25rqcXXilvn2r+Z/d7VUvN2P/9lf/cLL7m7If52cXY3dxebO7Hte3F9/s2u31tTSud+19uXg39vf32N1cut3lxfmPJ2++2r3ZX95et3F6Pfa35zf70/37YnyQ7uZcbOylpRiGaypUG81IanqVXNOev7TJaaYwdJp2RKdaCyqqxK/SMF0avyrXjOJmXO9pUSuvVdKBz+l64ZPfvbke353tzy4v5IbVRONCynOqmEduxozZvUtzaOvqrCYrG3qt0vCajLOLd6fjh/Lh6nxI+8wN/ydDel+u+/6Ume58/C+/2z7gT7W6dPNe7iZd2Ob0Lb28+Xr9+IQ//+/tuJFb1B9vVrMm6JBjDlvDd5Ojk9XdJzNL0aZHW3VhhlJMw+XgXGjdd8bhpo5juDlHSa2ZlGJjhkZ7PPOX36/b3PVe/vt038rFxer+/ef0a3/DVJ1iU9deG74LGTt5I6t6ygenl9enHy6vxda804lvMIp3Z/X8/h46OxWkLUZ2c9our87Gnhls57d93M3P+7J/f/p+nPfL25v1u8cX6EzLXFRaG1c3o5+uCTy9KmfbCkfndf7pq93DlOvfM+X6uSkPOvonU+6NMT34WHRhYou3eupqjFd2Wh8d8+9dnJif9iqnHLsqJuba0gz8HdKRpjyZZ6bchvzpKdfmj5tysybi+SlPzj+ZcvN7ptw8N+XGOPdkyvHnZJzSo+QQyigF3/ZKpzKY+55bbN63ZtVwRjfHn9naNkrrzH6zffzyKdfRxJSemXL1jJW7pMKnpjxEb+wfM+UBP/3MlHsVbPjpX2Vw43y0mwN+Tb6+EYDe3fUIPL74Wm719eFWX61PpF87uemutOvL/X7XLi/2o93enH03dldnMiW7DcPUyck3u8trWrso5wv2wbt2s7u6HvPsh292d7P1l3/FzLyVKfpL/tj942//W9qtlf1m12+vzs8aoWK3RnH/kz2B4oYPuFfZ7+qYMqd3yPooDATtkp2uN0BQzemnAermnMX57qPNurfZlC41jAbaEQXCCDW0mE1LzaaHNs8v6Ybc+VHrzVhrinclG0ubyfeSS8t6NFVsLma4UXqcsZlhTSp1tDGjMkN3S0QqcZkw0c+obx+1qq0CAG0O4IGZSivsMYamDUHE9WlV1qE4+usG8QWb7RbMMMFZn61S465VmfFHzQI9To+s28ggP/cfkdjXbfOqYvwzmqKVOIZLWTU/VbM4bykux+FizY87u63g6e3FFp3F7G6ub8fjGz9/yZrP4B6P2BBbQ3SmKldwWLpGT/jEqqZqJ5QTuMW/WegUKv7KQkbP6uGRRlnp2u3FGcZxMPZ7M1l+5m0Uh7q3pfuv733OBSMu8105P+trlR9H5uCefPWk52p0TMoR8WtvWllW3o9JlHQt1qrgH/CNAbynQu99KnNMBoIR+GCVV9Lz8QOdvnfnRzcGPH/29cOti8FcR4a8KD/TYPGGCyyULQaj8N21PIC/Rkxp2o1UjW0YT4GE8B9zBfGL8f2nbhxD+ujLh9uOZKKd3sN3QsnNwx6EQs0kwapVvlG9ztEMP7BJCdhYP3GHbme3dRGrm8ubcv7JESsjk334Rv7r8Z11irWZHA3eEfzUGLTHaFneMHBuM0HvmYov3uuIiXuXk3c1K8+8uGUlh5Zp8TFIROvxnTFiwSWqihZiqJPqSSumE7xI3lsVR+PLQNhQbqicTfZz2jgXSDxB/vNydTDEw/0OuP3MRYKohysmTPYRtA/o9nfLd3735ufTtCD98QQVAC42pbzPPVQ9fPCqqN4T9umrE4vJVg94sJ2p6RmgqNmHxkRF2IiMY56dL7q/OnIH/F/fdeTk/+yJFveY+PF4DoO4PO/3Q3jmkp+HHhnTrp9x8QUh4j4GPY4/j9B/Xl9+2KKMRgx8z7/fEJx23Pnu4rcrTm3xc0Vy/lwjuxvLjpixmxKauOoDpOKMse0u527rGxd8ICKd779ZumQAALdFPuVXws3RIbubW1TLzfszotHtxZuf7i1bFkYAcI+EOQSNq/M7spDuSMSHUS4+f6k+WXLj8vuL88vST8tFP72fOP4i8PZFS+yJ1iYFm+xq24r3nl3cyDAvtmvvBr9k2SFIP0iy+ym7/HZcnP0bEu96tLOrQWzVmSA8PpJea6jcfZ69W6a5H6u7zjxEYFp5LGpquWnv16ebzcs83E/X/mZcbWZ9uPow2avnh+/+xayv/PpfvV1ntn+suv+Kf+KjS5SwnHl7fn76qRaf+0E/XQsvE+sXQ9uU5+l5+XGTgjLMw5w8fLjhVul3f895cdrPPizSvSjjOL/FoedA18IoH74+OHO/vrw6IMXJJlIgVOXORX7z/dnf/eofd//hN2f87zvhQ/9pJ5q5XO9kZOM/vjncsp6V/UOovbi8/nC63Ovhs/PrdQ/Ei7jq1c3ZB1lxuclf9/Lhn6Wh78fZu/c3p4yx/LhdLNe287OrU2lxWeby4vZ+9Nvz9evfvbnAtqSZX5UPtZdf/cObR1fI51t/T3HWD7dA3/txAb3d8+Gbn1nCYWneHK59bCDL8uRX95+6vH18Ncq3p09GB0U8/bCMTT5ZH63JON3SD9vFeny9iDg+eIPIkBZnOd/LVMkdYDBnMnsPU3VIXrxZTtbHDwj55Vp/pU9OfvvN4dt/FpDY76RPdx/9dvtonl9eXn+zu5+73WoEDLq9uNkvKDofgjOHFMmbtWSnZeLOp3dk+vQ9f/3b5cPIxCH7zY+bk3OHcmPNNq0Hj14j+P5ipWS+3t/8CLQKxf76fHw3ziVHI1d/B8et4pob88DDbwCQ07N+ejnnfsg0CJHaX4EOdOJsGfvviEN3wqRe7g+qeay/zE+P+/AoVCHYpvEVllIHHKGEAnGFnipiWB1wPZ8UqrgXiFktAToYak7KFxN8rnGxiA/lh9MrTOGO6BnlBFPejYtxvbl6W/EGTfEUgc4uDl4o4zFrGR8+ePPr6/H1r/6eVWnXQxJcxIf/8U+7VlivN09Hs2nDh1TYorL5SYJH4hToey03f3BBcVS+v2ESz/7t3sf/SWZfpv7n+bTd/dUSt1bs29r8ZndAhluMCFPaX56vICWI7Mzuv/z6H3f/8Pd/99Xuf5fvzvj+Q7m5PmsYIaFkd3Fx8qsNQgQ0xgp0O0bZvr26JHrsJOYM4urh3mKaH4iQmMT+8nq/oqJIvzXWLR7uJUe4zcB97ux04/+n3+11WjFCwX7dM1dkWUZotNuA935mTydweLmM+DAlBKWrm/crJH2zJRvvJ+37s843Prz9JenHt//t1xbtKat7iHZPp/eSCC+T+dUab9m9u6XbuMUQpoB9lXZ28+OuXt7cnGN37ds3h/ygxHQZ4d3irjt06c3uv//XvdwqvP2bb3Zy990Gt/u3C3/XKvz1//z1EzNqwp7KuxU827cHwGFyvkbArEUWXwZZyrttaQEawt4OPHs3bnbLZDeA+UrsemUK+OOgqYE18OGnx/c7cItDWmFlKh8C6Pnlu3fykdCRB3MRD7ROuRPiacxBxHWK7pDH2FjC7YcP5frH7W+k2PIX+VbnzWaeSB8t+ah6BUhf7uWTC8L4Io53rMfrk+hcTDoi7eG8cq9ysf+ecHx6Je65rpRexROVfApee60DUi88oISkR2R29qfbvKyhbiBmJDn15Mrbm5lO71JOTmUX7B2Ne6aRYE+csy4obZHq0dhw94MnbRltT9CoGqmfvcVC4wHeRIJtTd/jmHwMT7iCW9/9/DA1n0O4FRvfXd2efhgfLq9/JCoKztucT0LWKTvuaiRPt2zhI7pY9ir9xcG9vhKbudjT+v4+7f8WeNy88qtli+tv5mBZ4l//hpB4yE7td7KWX+22qP7VTvxkoczdxUvq7mDiN+9BhrP2Fh/iX7E6pEF5d3EJ/W/7B9H86MOPbHbLlZ1IRley8mIhOif9kCh4WNaVm1vesRneWtTLlXwTwHvuyt9z2bwudwJn0RH0vtXOEPg807309S1zfdZOC208VicC0QvDxR0Bhq+3zZHt6m1CgYjxw2hvxSsBr/Pz/duzD1eXsJe3MglbSHwL0bjav5X2Lt6B7rcX+9sruYi2Bcy457Xg+z0pJTwAZWP3/fXlxbtN9lyV/f4/6zcf9xapIlxmo33bFysNmvya4D6mhITTAyMMZn16f/dly+LLPxJdfzgd19cL2vXy4PvFv/88/vTR7Z/e4H5B1m3EYja3Lmfnp9agpm7395f4jdXcpStPZWb3p/bpRQu1HvftVIjVNtjfrI//9r7HP22BjLmgZ6dXP968F9pxb0vKL6kw122Izd+Nj7/cPrweN7fX4Mfi8g+XpJwPJPMgDg/9ubtA3SWmNjh5ZMN5o2Cn48PZzWaS93fduO4KRxu2CE16uMDecZz18fesxsrRXO6fXiLYc1pvO/EFc7i8etInge0DIj5BYmO9OUneBxWtBAnxgsdY9vTqiCueaBOcAZuCl9mS7qAudVKP5K+Qqp8ehY/TuxyGxJavH+jg/j578aD31wBPF285FezbVnn7b0aRvtX50Z2cOglaW4viEIWi1wLeXeu+1enxteEkWAmFNi+BKpB+2NU8vaN12y3M48F8Kqo9heMF0TjtO4YE9dlYxpBNWaxQdiP3Xx1c9wz33v2dsAO8+z2x99s3Px2U+8315fk21iW11l8PubGP802bDn4UjNVJ0iGmrLKCrvkte/ogbjfe8WxkDcG6J9c/CYY+6CATe4FshzL8/OeWWH7iQo5ogWDuLhXTuW9nddE4rN0wlT44G1iGxzBCizePR73tj5wuGXo/4oce3v3ubhfFrC2aRyN4QLf7a6x9lPLbL7p0/4MnW9W9E/ohS62E4ZPNo6eWfcjdECjcHEKnRtN+jDGbSb1wiU4zopKKn+HNE5Jyh82nGPc8+2E8uinTWTcV97uP4PhROFpCSn8Kxpfff4zZj7H54Ubl3fVY4ul+HlZMuZ/GeyhZELDucwaeTDD79voRALrVD2KioGR/FGIfz/fd1elnrnKXN1wrsvv+/eV+PIp2Z/vdo4B4CK8P8QclsHWHy24IjDBlcPjyA/r7RgT843D60MdH4XT/iXiKjj4T7XwXWD8bP8zCvE/BuFm48HmYNj9bw6fz+kgn31En3E3m8cQmFVIS3oo2v+OFSxfcT+/thy1VuqHP+Y8Llvr9jL+7vry92t+JJmjsbs/vz5n0O2a6ZuCbxQjffsz7dvccThjBnS095JIlKbyw4l/eKJWaj9kNRKVvJjsbWxmua91MjDmNOafXLWRVUKa95ZRyt+Bqb0GN5uQOQHoqAaJeDQKhlagGP0mhWJNmdUXN2IItOceuS+tmjhy9dzZ7xMSoXtowMbdS6ugmlVgy143qRova1+kk2w5cxZKaop85+e4D7RNBevM0ptZIzTS2xx6aTpUV7tYWo6LjviFVb2QPw4QUGVYevkRbZkIo92G8BSXy2NoY1vhasjEEKCEZvvQwi6kZIkzv/YwjO6dC96XIGEHJOG0tLvSQVtKIv4rqxQ+rR7SjqFy8U0NHz38mW4lMo2U6VzQdDAwNdE3d86k3c9vicWlObikbJqiaqSy2FFycE2XY20jVhulasgT2Ursps7goG1SNQSXdF6i5kZXr04SZivZcwbKOVgcNYpatDZXstI2ZDy2Yqh1wqMywfebYWltt8EFSDSOIYyTWxejhBWeRBG1M5ikoy2qYQIOdYFFmZfV7NJ2ps35ubYRmGZCzmNOwoYTEUstqOuf7LLbaXLG4acVklG4t2h68xfB0bX1tPHlXXY2mhVGxsRqxP0aEFg16jhD1IEalVMdQcXo19XAhpGaqYj5od61t0DWpyY3sVG44TwAYMRqx9mlHNroaZJxrGIiV8yaz5Ea/WIaJDtrmNI7CfOg4R4+x587U1kK8QX2y/sXHWnNJKfI7/hz4hMJ4WFgbYyAcSRvJYnysn08peSVHKiyextXVF20UGqcZjUl0GiwzKt+TG6MHtDdTNpacSLmpjL/F6vXEHga9bZquuxnxtmmYVT8H1uboX4+IYsyl5Ok7gTEsG0ujJZ8Vi5uFH+LDskTeD9Wrd/iLiZCUEMo0zEBUptpZp8Ywp2k6rM1XHFz1TOtWz6o9C1y9VRXfjH72GJjOnMWCSpsAR2zR2yKJy9wQ9HHlLTLmq2stTKgddrgyoEbWGLyP9jD7kmWOtJXZwa4UBMjorkbB2WtdNpatHamPmlhfBgpQdUzVBIs1yhZ8nInbYDZQSu8xPjXpm4EEuIbFr2NlORZQwpvifJ5ZLKrVMqBBFYiIWCZeRjwQKlH9qAxCM2RBDzsmCLnQVgEWDeCy/A4iOoY2Nhb8sgAELLMyKhXVmlasgi50IYg/jWa6022tbdEuxI4BeZ91EwuzJWhXuBgXA1PxBSZ0augOPQu1eW+sxcaSZH3XWFjpZOI0ybk8YmUhBPNUH04b5ioLdLLA0w1VcVTF4vWUYNxQJBZoO63AmnZWr1eV5fCeV5WGfFJO/LbpXmrqeGXTg/8pdrowZIqyUzWPvMYie/YKPwSwlSsJyIkugSI+QtCcuJaPctIwtDn8BL8M65q6YjDWa7PG0gwgb2wlCmHnlQlTzZpSsdQqBy/Agalt7UGCGdjvCqtRLCgevWPaVhuBcDBzmuBusFHlZjWBSpIaC1gdtmP9Sh9BetvoEgm7ySBShGiusYizZ9mVd8a6Wq3SpiRXDWvgBv5GyNAOR1S2NdfAFp0G+Ahis2I1LzsdynJ5xn4mZg7IgHmsK0gIIlaTi8I2spxGMQwjyqma2mjfJQ+o2dWPIcuWh60xNSZCi0Wt8ak4iBYgkSUomGBS663XEAX/h81gRbQs4NrrImLpaJ0nHNnZgEQzEmQ5a+LmoEcO0wOiwIfkmrMlySEFPEiB7DGssfAtCJG6RIWu6C+RG/PDJwnj1TJ/YDloq2cAYfhvOUIU5CeqBIx623NjVVuugHgAtl2QA5SGcXTpSUClECKdg1ZguTOgWTojZiqiagwnvvnXn54eeHhmr/+p+JIDP9okyAzsYLHEXyq+AmLCPC++DJ1b3z+nvnzAJk+wN+NYJ7eyCc/JL+fEeCQUoRbNHyK/DoN+Xn+lddzu8/pLr2zrLxFgQc7uZptkp8nK+S/MCcDB8EMYBg7na1W6+lYNzKBm4jdARTjCpvQWA/5IAbaG+6DA0kH5fKTA1gb7RwrMfzppZn+BMNM2PqvMQoyfU2Ze/SHKzLgvTZpp9XyKTen0e8VZeF6cOf85cWbSiVJoCcPiiTpT6UtXZwwXogVlIUpAv6CiDhcEaiOEvSA8RjIQEKGSPWairxuyq2y4nc+2LVYDjnsNx7FxyBn3BgPDdQPMiKBDLPWqSQDF/w2wzSdZT+8iKsVw77ZUkXKqQbs0giXUCdEwVXTTUBL4TVMgAuLGoIJ8da4iY7yHZNYIF8vcd2sDxYYUgCwbxpgi4szAcOkO8aoQHYRB0HavjdAGMyqlWWSSlrnb1AhtJHgUnC/D9SDIqIykYGJwACVUCS4B73Pow5QgYTBJN5Wc7W+lurTYBPIp9AYDTy0EXxlsNxA2NG210DPUh4awCO1H5JhUezKZKE4kjNlAolYbETU5IGOzIJsQVLPD0AhLBGQmDcrYY0KxwoehVTpCzJjvKKRZhrj1I6Wh/dDCQVbAhgh0Rd+Re1OkrIWPaAtdieg1yAzSykM2TEvEqU01K1GilUHmulJ9mEFORO1SoHrZo9xjm6xLQekhN5EWQSYac0MUwgkObdAFJjTEwOCTRt5gEJDXlmCCMAg1Xaq4toZbuBh9NVOEG1Nsa1gsTyGeAheWBCGlpyozN5grg4Ke9mZTWOrKKJ8dspW/mM5WOgzTwva2NuA4DM9NeItMJewXqpmgGF1JMhA9BzcRwt+TRclGOYI8CEIGEdnWmU41M7q8CKEV+gVz6zC70kXgoyiGRnoNLTkMZjiP7pkqOeiLRBw6l3ysjAiMlsiPetAYEbJXeUiURgciKeQkqJRBoDGggTgF/A4pF6CdSWY95C2LwHq22D36MEi9BJoqu1YQWNhiL2NUyBVCgV5imiybkyoKegP9bf5gp8gQcQdMONGSkoxCN44BgQ0G8ChxJFnb2kGAphFNNqPQFV4oqmIbS2FlUNjZMRLUbywhMfjS5NgK/m3oMMIxTZghnJIlzEOXaWWVJROw2ugB2BPyiC6FmUMsGmqHkSNMYYtAFhhoc0U4Q62HdHj41FNtgNTGWOHbtFDMRIRiFbgq7D8HulCTR4MBJKqDqBPt6qdC52pYaORHDGqGlQEwaFQPye1ZCiEMHJX+8hEc2CBgLGIT65GsAajb6WyQEgrTAnqXgJO3bBcdKXKmHgdXdWBADoZUPDI40DsZSxYLRvIrzATx1DRawwDwWHz90jJmVveK0AkEOxcVXN8DrA5haOOcIFKQw8lIr6paEQDQaH/VQG3WWpmyVJFlYnBvsMoCtA51icRAl0JLA1qnDzHxhrPhDbaozMIjkgkEFW/Pbtm6nT6wogpxLn1nHjIYzpwTW3RiqZiOBnu1SFrEKFJmqiB5CVw3+ji2NvBQwh5+ibYh1gKTuGuVSjoWioFDhaXjQZuVIvLgkCw5ogfuvGX/UAdYLxCMzAsglBelDfwQo7oNARMEWhvdlQPUNWO7Odrc5Jg/UTEeLws5vEqmaROi7KRIeYbO9MKKzfuJxchJtDADshAxjK6zqDY3u0vGjHy0TGZG9qUipgCHYObdmD0Kc3AzdJVAsqAkGClQFYZBXxGTqFvDvIWytYEoH0oBBU1OzJlQndamF21ZVXyKiFB9maj2kkBF0KhL8q07ViXii4c2sGGkte6+4tm46oA+OQs/AKwbpq+hWaPwY4KEpZ/EygSbARrzhh/MmcQBCA+hiliE1CUmwYe5b65FtHziIzsM5qWrlF8gg5mLKLhk7bEyu24WKbaUc8+mKGxzABUek1cEhjY6tukYUJfiTuKpB/5Q3ITRbidOlQ/ZYe4InzMRK1O9qiF2bokfirjdabUWK/kaz2iZLOYIlC4QBGYx9ReVYTaSIYndGSIHOCUErhIj4BZEGsQqjjCAYWbXzCheHmYFciGbBcXbDllq3J4wnImHSvJcRijdhJMVuDQMBPLe6D4QE5Vk3GzIIngyZpA3HDtGpptIilPQd+hKFSxOKoEVUoPrchdGjs9bKBxcBnvXmI7KQ8H5Uhyg32oDc2IiI6AM3jIaYCcCwXgrjkwwIHKXFiI39jampDDjCWSD9TMDLCvjDpgYVzpaoYoGmfAn7EO8xio9F5frHefj77zQW0eWBSotNHzWY2XtAwg7kiyXqRAb3F/BJCuwJTVdhBaLGcCisCkBdywOY7Mx0iBgtmV2w1CsrkORAZiYBIzIsT6+ayIRURWdIuaNnWOx2hj0AWIIHjJEZell61HiayHgsNr8JhthVqCS9TkJA4dHWeg+3mflQwejWGn5nj30ty4bi6gfoASMtD0BLrgchhCJyp5IryXJguVDxzPujv9VF2GBgbkTb8xl28VwwwpLMBqi5yQdJgIHV4CvwDt9EIhPYngeboD2QPrgxKXhkKr7o+2EKDFNwAHUxO4YeGPtmTWZx4gBtNSNFPD2FVjAUOROh6sRb4vdOAwxgtgD0hPquY1DZ4JIpkqmLco+hW+tolHNRFrokqX2yRkALEEWcdSj7cigziS1ZaMoCpShFFtFQjMuB2cKVSPPpagNmV2JKqh0lBIgArxPWPHRdnUw9RZB2AQ5xZi41XSKwVsJjWL1SA3CF8ED6MZrkCPcLY0M6YcjvKidIYWCkwxycDqBCchcoAcZRMBCqcjBUEyahZgsZIsM0g/4NyKKljC8rQ2pw2DEqnVBHOEbjCKmPlE4TkwcAohKhBEgPePKLCO1C00A7/ZF7VCFMTHrShjA1OvK3ITi6BucOqUg4J6QzZlhWMmgO2eS13IovvWD7x9jlwvq26B4k0HCOeHncgoWq7b4g2oOXqhwUAiwlZDUu5wiBQZAhWYAom0sCEbsgZ6i87AwI498UEUYkxVBA93BDQxQ1bSkDWQPCy6POFVoWr1yWzhkx/1QOp1+SwHnxFnmFNUMv58Tg4WcQ9QNPDZhyJor8DcAHna+MCgTtIhJVQpA+1BAAEaFQwJoKGYoHgtMwMbPp3e6Y0NoQYIlBudFeb2oXT/GjQUkDKtrLzKVIAFVlnASOy0Tu2VHSp7WQSTseLWGPBNHrKRitoxkgmPAdRADKJqCkcPv0RcKg+VfM5iUjAAHFglrWL6lSxDMoXUk9thj7T7Kllaj6xU/TPgHsB2k+pcLoEKoTf5FIPNTIYgpzowyyRKIy5AExGqDoKiqVoG7OAdL0lGCFmwem2oVngoAByHCUjut1qYToQUYAyHsFhtobZQmCpIVjogTAjXSCqVqwA6ilbigVbJRBuUmfuJzzmFoyHWpqF1tAK9qIAAlbeEFkVCcfEuIlV1LTMlI/JIojW1HIdNYM84HGHXYzdpJRe0CjAxc5EeTzKuDZ2cEJQ4IsazBICtRmg2+zW0SBggFB04tPVq4DqOs8kwSyCmsFVIaLECU0AsyYhZcrBglAeEASyWCiTFnOYc5iGhbaTyEC1LOxKL0BTJQ24hLUC6BKpIOWQUogIyFrxNmDXQW2iCZPKThmg9aUAWuRTwT12Ho/HaixgP8C9rpjWRbbAZwgohvX0SGOaSqZG1q2tpAExkpAvcldd1tl21o0/2Cv1kTi4p3RFEtUXaJpfA6hUVHJV90tB1ukWe50hckILxOHrIi8SIS+cDqTACLjAVlYbuSLvdgGSqQCCNVW/70GLvkVWYegHRGIBllXKAJYIoWga7RLshZiBY+M/EesKwKw7do3QqD8AtPIb8DhCNK86WXpyTgVCzp1E6ACmaT8WlhcBaAdblI+q8R4aB1MLftgQySo4AQWt2IWqkijqKHlCBMUQFDNBeSF0EmW9NwEBYVTtMILiZKCn9ro1cnSW8ooyV4m47fOjwom4B1riyNZmE631r+V5J5ABV9gUWgJlYbzKEUugecAbXjDG5XQVI0uiuSdI9yDJTQFqKCVkjQyFZotSRi3Oa3stEwy2D4vq+cGHKACcK/DGhogXU8AVCHsMM7MwEeYWPl8Q9N6NDmc9BVvFgnALtAZRqaww+Yb0MnB9AFzE6wRO9dEP08IXwGq6T7RBqz2QcKRYyjNdmBQX8nmTVUkZv0KEhC10queoKXrClKQoPMpjGdBm2xtTG5txdfUAyIfxhcIziJJgMbZX5gWRbbjdBODBrBKrs+NEVsmOpoJzEwSe4LqwMJg3NtoVyVsxPBE8m9I0AjZCTZDGvMTs70WgILbaLA1trSCydGSjyEcjXJB6soyQ8GQOgEcRJuCjQreBXEF0qI86SE9wCWsR3rRAgTDONxTKJTdl0t5z8GRt/hZAFqzsQo2RQxaI85Mp2q4IuBuqGFttMtIioxJw9607kxwXgwSw5xSMRCLRM94d4dLJsiLpKQ/ozeU8z+xrW7kQCZRa84KNTy3Ymb0Q8vHcGbEcbEHwJkXrhsxRpAqYlGa8s+ukONF9EMyA15JEuQI9NwprYSv4iijmxFk8aOacD3cVnR3khyJn8sW+dudqU6Yx+ydYPZspySN2WIElzwCk+YxK0ki1Ilz9YDHN44UYzbKRuQuifYQmkFHkVLwMh6NAhsOcuGlwIPbIDbAATJ9eZyRu2KPaFAt34UiwkIzkfUtPgKuI3BzAB3Zq6JnKlYgmYDRgTY4Z+0hAV55+Omb3tX9MhNJcdusHbD1Q2qKWKBDnf4A4oGUiB7OAq1GuSpK8KSo5h33NrADVzxsCvwOBp5sAc0LozUcguuSZ8KWCi5dWCCiRKbK6MKRZ/ebKeORKYjCFmAViTJIpk6grqG7CSC21QOlSyP24gaXycY4a7YiYKRFb2tLQKRy4ePWD8LgucawjHu1wPY22XTYlhEL7FD6GCWh48YOcmBAsK4XtTpJ0WnuHxmeLVs30F8CrGBHjAAlDKOE5LgcUFgOdk+bF2tvUzh0fVYJ6gGk0b8rZJTgxejj1EjUBFkE5ouivwNKjnhqPAcwM9mRKVsFAUrm36rDWT8FHWNqbWC+kd2rC0r8HdI0gBygqRkMWGbmCpADPTJjo5DzfuV22JRQVkmgClVkEpCQood+gHJITxZa/gPMQYhj16ypCKiO7BdQue71UZyWorDJZHDrWBvMLhKhzrhlk4D8QT7CKHocgIATQvR0o6/EO9Q6a0N36LG4+FTDLJnYUUTKIZQwqpkzQkOCvoJHYG9VoBCCjsT7j6nWzYmOSsDF2yyZUQ/amSKMS9oYaoxAlBD5N2oXZ50FCWbYoNAKbMHc98eB1R1hsqpIckwA1lwzHJDPchDClGYGJQwFtWB9EmolGRDlQ2qqGSjdMWoASjNKsIvSQAEcoXzweRcBjecJTRJtjVF2TiVcwOifAT1nEK1H2x9yP5qYxiiFTz0JxIUTZOnG2IniIohEnfKfhUxoPFTkEr8V3x8y0lPeWgOMQkQZGE9jUmRq6T1CXTgMeQaOGbyGvoQfoK1EsDnYko9BveiTvrJUwhUIurI4x2RUiCv6ALZhBPTU0r2B7EnoUiZNc6EUUJh98OlA64T2gEgk9CKcXRJ4XuklKIh5FbDwhABjBH4VXWw3kXYlLA8kQ5wha0NqLYWzRHlDIFYipFriCtgYUFLrvyfzQHepoFd5HcH+bEjAt0W52CtkQWrclgFuSgaRHhpwfclA+DFh5Ls1odKWIJue0VIMgNmgM2n5S8Tyod9Y91alKNusJsux0pizbKBJslSnFmOEUhY54Lc5RQAcR7yepiPP/70pBzTWo9w+cNr2DCwJIcqsBz+SP7PWMMmG+gn4BRcKCb9uSI2E7zgATyQ/0+rmPu1iO2TZyifHqE0+lMlxp86VRl/wVHJ9dzQZ2rYovrcScn0hxyUdF9eCZv705Sw5c8dkownEEkvWT0UTU7rYTavFWy/t4LNNniEbDKojLtm2diwRR5UC7VEcREgJWs8iXhEFskzwxmbV0n2nba9itcquNcquM+eUdCYaRMj1nIWdECFJqKDuDBlX73S7yabcEX2pqpCI80A+08dNs4qteOdLwimGNmdJrCyKrQNl3RadxeDbIpjSbPITMjWNozSQNgS6yW59tTidjbAq8J33UiAlkNdQI3kDKLkLrRpw3s5xUqwcxpx3iVJCI3FWQbuYY92NiAY2ex1UbZS+HmSp8369bRxdGhEY4t49xEy5oqGfmv0oEpCjiWFsOw0yIlo6KZFdDSUSe5VF8SAbH2KTJE9koTVuwLDloeXFldlb7XVxoi3PUZ5fKuD+CHz5tRetIpkIFoWIRjh/p54r3wvZYrb9QFhnDnbVjLsd645FcuzkObOaG2BAdesUdvaN+wEaIL95xYN9LvnigqB6KKRwYUse+HTv1Y4vuAKx9fqxKfVicbJw9WbYCOhLiQ5Pdc6Ub2h9pps4CF1FCYmcyVPxwOynJL8Sgws+dH2f2DiFdEWcHLZ+JO9D8cc60ZIkKfzJayBtcCllCFOSL63VWQaK6kw1qPl1eWFCURlCZQEDlxGYpGCGMnGN7OCYTNag4oFS7TFXuiwkiflMylDvagqSb2eOtyzz0lyX4Ofei/5StmsIWAnLdFRziyupLvsjCaZcxyN2BxfVJUk38nJXCxXy3PCiVYu9sQE8a9xkrAozamk5THJqq4HEGfJsUlxXJj131XhCKh5KcPQRk46uT9ngSOz5U9YfSzPSK7meWlulby/QShdAp3da33jZ+obw0f1jenn4jx+SpzbZx4Ktpbw8vDgrF/4IBqiyfP1jiF8TsW7P+hJNGu0X1q9o32+3tH8/meGfUbKO/U5LW/tCaQBL9NaxHzWr/WOv6De8bVW8aXWKn45dYaDCCEVYwhsiL6ZE4trss1peoEPTauHWBVKQNLAyErFNPlcsSHZTjparaKCvbaCWVgIYVWOW0ryAJVnS1dwTUiaCqJ6pWJvyt61snJasqLvcnpR9Y4vpFbx5dQIvpCcJFCoZX8SvNwQEiEs5TRK4duyEdi9nDyRDfuk5H1dKU0W0PENMXiruxR1KAipRanLTjNOJwVbqBdrNU1jriaIuMbrpFpMko3oEoiwcxsmf1H1jsk3kXVaYUOpMExUBkCPpqk4d1dSDGiBPyVHBFHEBtUe1MRKRjnY+kupmUQMy+t3KmFFagpdzVU7l3oC3aH66FdMMqP+9QSlp6ndKZOrHYP1zHO81kx+Iu/9Wu/4tN5xVG2Hg6NU+g4wy+EW4DKA01NBjEIA97uueL2kvIaUm/NzeEqMdXtL0WvN5J9gP+IYdZfAF5wwA5vQCVeEZXglSb3siWiKgG1jzXLYvmUxPnmUicZA5fHIqeeXVbv5x++tHKX+s8AHJN1htDwjK8o+gfFG+Adho7leZ4xCrVniHOTEuSgEcKIreWJIOF4N6RH2iY5Sh4r7YZBFyBrBJLdZ65Sj20mvk21ypA2e4eUck5P8nYCBxT8QgHPzuePUsuqICmTeBi0hQYhTNq6sazLQHpSZ1MEAXnOxNhVht41ojNXiIv1oe14CtdbI3gf6R/LnskajdyyhavSfWkaOlRgNAYW9DAjMkIxz664d9t6GbLk42TiBERHKA8gHtCVUvaL1gOJUkjrH7phIjFheRYxMkR2wWI+19/Ziantfyh7gkt/EEYH6xBKNOCUWYtlCP5LwGVqQcpoQZXdkyJNmuuJukPtDjfFR6pT/+L1Igg6S3AO9Rg7AypHKFYuRiS6uE58iFlEUDrrt5dEx8sgcOdUqB0m3s3qvNcavNcafrTE+xv7ua23va23vn7q297Uu92XW5UrVJ5KwEVph4fgfZg6NqhDSQlDhcis4HvsSzFYyiwMfC8wuxPCF1fYe4VzGa33w0/pgC45KAVxBmQepZGJalBzF0CwzRkp0kwpJVk7kWmxKHi2BDIJNd5jxasONxkoCWngO5j6gESgfZp14p4FQVA9cGyGEoiJWJgwbFGO4hK1hyxdXY5yg41ry0l5eKw8Tq15OyWjwFsnUDfPRU5dwjTNLiaOcikGoYbATXf+iaoyPcAapyqvq/RTYcRB1yaQrZHPp3F7KIAkwMSl6ByEFHaW4lXALy6A/dN9utc4WmUqnU5LcoMNRCzatpVRG6pWL5EUDuppIM2QLEPEidbBeDXGw+pLqpV/MmayXUuscRFEO34BUb9FDUvk26QcsgpgPcWIihqQw8B49ASNu1Im9WWBh43Svtc5fcK3zF1WnfIQa4xdyhvG1TvmjOuWXUmPcmG6wXxLVRgcMzjpJvTIDSvau8npUc6IzaDFomjOyHSYlgciZUtWhTrlDaZSAFcTWeBwPb1KECJQpLgTDNfJoZCul2QAGU0W4UygnVMv2ivMJWvGTMcE/gh1KEv3lMS/0AwxfDlMQHixCJmOskEnGKRBpiOUoofyi6pS/qBrjP/Lc8LHqlGUqgTZ0+EC1YSgg5trBtqw/Butll0p2t1H/IULa5GEu2DOanni5ra1IClgNCtCqiqLF2fUkyMsuXZZ9gUYktevpj4IIcGmspUTnvRyuNUepl95qnbX6972wkynHJa3UtBMp85/zhZ0G2n8iDzXWITpGHZ8/Ug1VdhAQkVAEKOdeq52fO1HtP3pl5ydfGLOO3X7q9PQvKXk2n3ltpzneazv9l1fzHJ6tefa//6T0Z17baT93UDqc6LWxY+WwjRxLe31v5+t7O18rll8rlh9O1YyI5dN4YRkUd8xTo7aiCPPE5Moj7WA6snAqamNMge0b2Zq2DW1vv7Sq5yj7coZYzqLW1EWHmhyCHDUQ/oGBC8MVSQc0SRoTjT5kY8WXEXGb12rjF1xtjGbKUs4AkCEY0UBVNrhAm9kzQq3wG4wWUZgg/phAdbEA3VJegBSeL2rn1KELmmRfPG7tuZ0DG6x3QjQlD66ynnJADtjNemSRlKJNErPlGNRqA6dg9MxkHVhzwfYbF8ieVhzo7y55CTtle4TPrOy7NbyOv5FEadgXVfX8Ut7JepTsOJNLHI7YpxzwicFXLnYVaGRNWHpTXMbkOoGoudi0PIQxBxZaXrA5jpUdHyysbEeiqRs8iChYkO/E0mmtEndTuCKLCFsr6/HyjmWVPGXRshvWvrj3uh4je/H/492wuAKUFYKrmXK3HoL1ZyudFkM5wUud7HzAPz6j8yVVg61aeEbI/rV0+jOl0+pp6bQJn6yTTs+8B/bfUye9nmj2TJ20/3ydtPmD6qTVl1cnbcyzddJG/171bz9TJm0++15Ye4IvCbxoeRiisf61TvqXvBcWBSaRO2vZgG9WDgw2yRZLOhz5kSO/HkZO3+HoTs0EgcqVCCinxMJrrfUX/l7YL6nWeowhekwobNVDcansz0M2lerQpwhVxfWCnBqSQmPoXdOyNRTbkL3m13rtL/v9tAaKKgEktIYgBqZKjQOUkpTCREzwdwUbpEBiTnl9ThaJBuHJCZ/a3pFtjNQBOyWvsWUmjFRW4oGi2rVHuOCSYzmHMXLESYNJUbPone/nIeN2nNrz15rv13fc/onfcYsohqxEK6d2FZgTgYcqR6glB0ngZ/2kkCT0Ll9IAXIFk9AN3pXDO16OU/PdahqKtSSsNqmO5OopM27kCKcTja0Ry4R6jKKKtJcdZ2Mlh9O7eX3X7kt+166Uao9qAds+tfAQ6AKwC5/wsEPGPIR/Wis0TEJZW9WgBEovZZWv9euv9eu/oH4dEmigW/L2h4JVgfhEswBPZy6H1BhA/4q8ZilLDaY2+DYG5ZFX/bA79UW9e9gVkMIjvZgU4nWGr8DA3Aj4X4S+J7sOJY3oQ5S9FJukrBREDlIQ8VoD//Ma+Bez8/hS3ud8jPr1CFQTj+k0l0GfJB3OnBl5ZYsczJ5GThuBLpgeYbwSi1eBbQTh57ZzwLqiiUyWFyfjJwhzVgW0lAO1xH4YHzqtAo4oBlSn1Bspi0ZjOhiQ2fpREkapW4EpwejayAmZmyWT4yuLApOb3cdVEdTFZwAFoEeKACPuou5q4IlL2JxiKhHrRmiAxe2AaKADgNYZQxQTYGmddwLGa081Se3P0Wrgj7Ej/FLekQ2pBT0LABl7kecpIdblhYOwODShlhAMXqIGlv/IyxLkD+DVTHy8vaj6dSVZJOI85AhvKF0qZcBnMQcmQmtxBZiWgoTYwGK1ht4EiIqU3Gy1Gy/mXd0G3mBExqMmhH4Yx9o4KXGyeAA+A/XG5KB5ogyYjSpVskT6ir4r/UXV4hNE5V1TkQDgpIgtgDIQX+5YnO4sGj7cW5FDy8SmCGSMJJk2rA8scF/cyYUvqRb/GHX0RziFkbFQZovYGED2CFlLrIplYWqDckr9fA5GXvnJ0HSzkvQekmZ2klR05rWe/090KgVziTAYWF1ViMs4pTwV4I46ApgZ2op/pIZEBRmgM00qUejYKCY70472TABITw2mTxQBSCXvo6RByYnQZycyFitH1iZIjRTrA2V0WhxIy37NF3fC5ijPN3h9Z/jrcwU++1yBYzwTIIueZlKBfrMe3OolCkHZMIcCy5JKNCePSiZmTgnwA+3gpcwIHqiP9lyB13eX/wneXX6MZxMw98wQriQpISm0KyKQsldQ4yn80keCBWwXI0NYw2C7bHAxH4zUHu/956/PBHiRpx5fnyvw5LkCDUQrkrYpkBhLbIoaUwF05pTHETKDI0AtQkYweDnWAEYUyLVREFBA72jPJrDyJG4t+gXlnKSVsDyD34I0ftasa5/yzmApyXYqwpSRnIR4NPKm517M8w0IPcLDwTdMIBEPCE5EVQggsxKLbFG6PhdABiHwwBdQKG+SntEp+/o++E+8Dx7WRlxzCIwB9/cgl+TkJCuHwfgOksxhxe27qH58jbVGkGBO2cZtP/ulPKvhOO9ylwpt5CNmjFFUOSctJw2Ac8iIRes3YinLQuRAWsPSaZRFc4RdtFt/Sae3j/KshmM8ZwHSCLZmojLIbAEgI0e/q7yuqWEUWiMlg5HXcdAHRAWQW+GVssvHSo+tjY57FKi58DpQUKMiC2ZSwCYCDFOAy4UulDLIGXRJNiKOlDzRoWzPJng5z3tQQJ0cumb8KP4CLYLzMcO4CDOI0ixJYUEdAg9w2OrksEqFYjSpY39Jz4x4faf8n+J5D7iF5BGUHB2RWNskdVC9bE3B1XFrSX4I+8H4wpAtYHQLDBWf81uO7Yt6ZsQLebf9Tz/9Pzv79y0=')))
# Manual answers are used ONLY after generation on heldout; never in tokenizer/train.
result['manual74_protocol'] = json.loads(zlib.decompress(base64.b64decode('eNrtXdtuXNd5fpUNAQYkUiTX+SCCRXvRXrTJTdpeGamwjtYgJIedIX1AEMCykziGHRstWuSmSVu0QG9lWYrPzkVfYPgKfZJ+/xqKoma494jQBGBVyfBwuGfvtf/j93//WmsPf3rjaDJ+sxyGw1Ru3Olu/MXo7ZK7g3B4Eva7MBkd3zsox6PUlTfD/kk4Ho0Pu/0Qy/60q5PxQVfePiqT0UE5PBasCyd5dLzbjfdzF3I4Oh69Wbp7ZT+PT4678eH+O7vd4bg7noTR4ejwjbNhdnI4Dl3BhSXT5ziz1lEa4e5HYTr9U75943Z3Yzo+maRyt93g7vReENqQsD5LrgpLUqQiRE1O1+gYV14zH5LPVeZUlK/W+xiMkJZxoZTiilnjpNVt6LJfEumFAX8KY5Q3R+OT6d06mkyPpcBBKXBWyLgzTgr7d6He3TOt7v79SZnS4SnOM+7Z80aHeZTK9O40HBztk21fp5GkxP84U+G95rc7i/fe3O44wxuucIRrfMQdvXOKXvCpYPRC1wqt6MXSi8cL3VXQIMLhRQpOL5ruQzeiu0iFk6WhY5Y+9RhA0XiKM5JD0guGUpJ+1ThPGU4vdJ6lTy2dTEJqGlnTeJpk1ySGNvSBxTvDcDfDIYZxjF4MveBXyzC8pRtZuofV+MCSLNbjMkeyOBLcyfZCx2g85zGeZ7jM0908Ce49mQku/jGMfTQZvRmOy91pKRn2FYyRvjcQsPfGdODGJBzm8cH2j9qPm+3zW9tzf9zEZ2+Um3AJZ0Ldum3crdvddDxpYQjXlaOCl0MKx+6NclgmuFHu4POjk+MpxU2dlOm980A4mTYRatiflp/h08lJnIwSyfDDpVRqiXMeObgrsmG/dIcnB0il1IXD6Vtl0nLl/MZbaZwLkq2kE7pou/vhyf7xaGsuzs7BaEpj7IwOEwYcTY8h9044iKM3ThDK3VvjSaaEK2+n/ZNc8ja82oWUytHxFAdDOu7eCm8U3Dh3CZdOtybjEyif7bbU23Dyk5M5cvxgdHhyXKbdeCK6e8jKaRdLClD/zDbdyeGIXqZHJY3qiKzZjswN0k1KHR3CkLHU8aR0+AW2mabxhAQkAcLR0T6u6o7HXdjf7w6gN7Cm/D1suP/ONuKiw73o9lApTN7BtffG4331Nnx2XCZI3+MGUrsIHhxqCk+7c2EcjD+CY6cNj7a7vz2cnhwdNa9f8PKf/fXfQGj4d7z/Zsm3GyqFk+PxAYZOJEf31mR8+EZDpuPxMXL9Ig5wxghWoFPJz3xgEb83njjhmU+EoSsCbnZ3enJwgFgp0wZH3O1ohn/tl/mQTwbCLxMAF92Qfm0SEQrRzU8Of3I4fosQjVEwYhSrV46iL4xCiX1hFHE2CmdXE4YQ6sIwvA3jn0clcVElfukoz6GSvCiLe2YUeTbK86gk+u3bLCPY86jEL44illXCKM+hEu83zBNZnkcl2a/SXBj+PMLYi/ZVz9r3vLQuxPo8PQbSII2PrjETeQXs/x+A/SwFnieTXH8OiFc58CoH/u/mwM/OSclZuJ8H6t17YXpvmaK8foMpa22RUdkYtTemou9QxmVuE96qZFNGG2Y4N94L47lVPjinnPDGJR4ERRevBVYXSjubo6qJ0Wkl2OJZZakatA+Oy5KcM8FZJ1RSIgUvckA7KFKmMYRTmqPHS0XH6r0t1mbpRIGcghue8auuIqssuHIsM7QprurEFD4ptZg2RhUy22wSdxFdV5YyCAaBPcaNWiirJLov6Ot90cHKUHFTk4vQUtXiC40hdQ3VcnSm2SvPOJfaC1FclLiMc+6Cj97JWkXQgWEwnmJwPvIiTIRBaAzDyBohu2oiTtC1YABrqi5FMl69YNrnHHLEe59TzAmaa5kzj7byGmkMyFpTqj4aaIV+mHslMFisRhmNW2UtWGSKi+JdcsFxw9BKi6p9MiYnRmM4mFzDB7V6bbR2wVQlndaF5agV7CGssc6YQHdQlokoa4QAeBEJA9IYQcNzzDAucymVB8MY9OewrivoyHEspshj5olpJ6MuLiXJPUtcmuqzno8hovLZyxBZgEjBcCjEvdcJ79D2F+F9YCXGyIOFdXiUGRYWgSXjTfNLzgg3FbQ0QSucFGtJKiqD+0FjlQKLBhaV5NfotBQYDr+UiJurqlucwlSFIyyCw1EEGAsSl9QC18E4sHYtQURTCwuO1VJ9Yk5zivZUEC3NHtUzh+O1QkuNSzLnymbc0OGnUJlLKKSYgxWdZrHaxJi3LqPLrtzMfYsIsDgnala09JVppk1JNXAdpVBcS6uUQwddTORScC80y0GKgiiMBkPf+PGzLcI1ymLNJLKTG+RvVLBQKQpRUL2rITDOLPIHAa+FjxRzLAsbk9BIQSeMrK6NYQXGNd4ypCuLBbdQkXEEYqRk0LGI7D2kQyawgCAyCLbglPDOJfhzfUig4AYTvZEYPCodmYcyPMKMHhYqXCD1ER6ZKx4LPOChWzY0dSW4zWtDguwZMsHLDPOxDA/jBbDDAUy4yOkK2yHGXaop51qcC1AHaKmRp67KhgTM11ygPqKdcxayTwgsBXchF2xADCMrheGsyoTQQUpheJFzCbKYwNW60KSIqmBeGS3BhEAFR94ZuIfZgrtLbpCzEeq4BFNGY6sLvEjIra2EZdaVxcUjJZBfCNXMK0cu86wRcB4GjSYj9zzgp3LpJO4UeUY4eViFJVYEz+W6IcE57X19PWl8TVJwLekTKXcQ4wgCKkipCosSIgoCGXEomEBWCYoTaSAIstoibyAHwFKEEq5T+rwqxi9zGvt10/KMNKeYT1ZlbwRkii7iF+6RWAZWgwdDkQYJYWBEOBz3yfA5l1x4vTYkuDbU/noU9FfZ88fJnqd0+OWK2hdvSCUOoxwJAcMx3DEH3B1YgsjIEFZokALcHDzBi+psrTwbVQS4gw0lJ3GtavEa6uirDPzjZOAFGrqOAvYypfG1SZ9c0V5ZCSYZswnKK2jDebFoHRHCMKGhGABCRPg3gUMgsBBIIIGpKv6K/v2x0ufCkuTrL1UDthbA1lXKGCFtLraklDzIHHowJhUTKJAJF2hXJQyaiqhRMI/o5iEKhRi1fG1Re03mL6rWsBoYAUccM3go2GAVVI4lw37SZsYTqAKv6Ps0fkepr4YuYcGEKNcStW0M5CDEjozBnZkVMBWHRrgamUq0mtIRaC/BL9B02ASGAt8YUziIhdLqSeS/nNTtumRPRBzzDBPARFoHWkAIrALg4HweWQ2OTsV9QS8LT8ZBnCAKRQZDAMR1ZeC1yZ5rhPkXKNNLFbbXZMbsuoT+q+KzWHyuSwryi8toPKLpBc2uocBszpaYNZOISR7AoMmFsK+Cxi5UHlSMXDgXkTkIIoQwSbWWxue6LKNdlzmQl2n+4pog4ytUu66kgD8zj3JNIIkZT5EdkkywO4wtIIm3SmdUUUdxBqcjFr0uxgeEG9AGoayQVWT5eK1W9l/BydrhxOskKJWRMBrUABnCMm5gkkmJwZ+ZBQYs4FVCqeRtDZqJrAPawOoSK68g6WWe36KNifP9qgRqP71xtquVHuShp67K2/TgA84536749ImxGIzjLAIbJBwRnC7g85xpSi6fi9Kg8EGCuCN3HZM20SNmPCNCkLE+tiSZbxgt+e7Zpta7tBm37YZ8XW0zAt1JCdP2QNkNzrbElqL9xJeLyS8XM2lX4HuQX0SjQAgbHZPXKlawIeSlqiIoXwFj4G5Jwa8Umt4o3x5wKyvE5Nty4d+C1Ds3+Y7Y5DvqVr/o6nLR17EjYEB04RZNLDYE27jJt9i2ZAPi6svFzSkBMSNIsZepukgwx6rmgZO5DfqNwgSCBomWakRLZ1HNbM0EoCWVVQEhGFsKCc12mrhCD4hregIjUUlKkCogZWMF7CGWkbhITIByzZZpZBCXPluJ9EaE85ABupbTBg65Qlyrl63L9abc4GyTb+hNzvoltpdL7CuPQEoNVq+BxE4ySAkoBrl3SqWoYO0khCOQgS4OdgfqSoGAZjG5uCqU9bJ9Ie2O3uK6e/o0Ju0T/0mZTPvFdz0GLyx5H52xOit0SoBhwJRJnnY4WAA2A0wlxVEOEBUKZZFVQxETY1VcmlXiL4cHY1uSbekNuyV0v7i+R1wwIJdFiuBi0UQfwNLAXlCCJPop7rxyxbKI7JRWg/VUkBCuikrFaYtsXRXOdls/K+5NRVHRgoTRG30LfHO3C2+WCW27PyoT2oV/MICAPRAIAkNkI0nnhdTaBEa7LbIHObKM4QWkkeissDYZaFNAwgISNnlwGVPUSggEg8LI7OzVLnjBbbBtp3e43OWiOzqZpHthWnJ3tH8y5V2dlNKlk6MUflIGNBOXa7aOnTRDPjKLIXXTbIpNdWuTft7aNO3pizwJb027MJ3i+jygg7xch3UQgaE6upQWhm2IHTkgaE85WgfrGcT3RUH1xlC574F1xIN2wgWIIxHWiOSMA0WrYiurGQxAVuAOiC+SIqJ9idKA+ka0aRy91up6v5C1bFtvSLPDxYCoPXieIB+6ArANNGkCxF7x4tEORXQlSoHOBSCYclbwqq0vUQcfSzbOzCumXSWqk0uIKOSm11uCAdIH5O0B8HXsYB2Q1y8FgNKIVN4vqOihpuuYCB0QVC7BAqr6gP9FDy6vo929Uj0HA0E5HBC0B2bXsdN9QFDwN7Pw71m5HdE8wTYd1cUhZip6oCtT6wxmmpwtCoSzygITJvRKaPo9LTrACSqii3Q5G2ZYJQaFptdWnCjVlaGLqMdNyzYVMZBbt6l40yNzu1C1G7Xn1Ea4nL4UoU7C/Dsz+pXqQbq1bFy+SuFQjpoDvmOGPNCDdcJYahRZYDIomFqElIMwATzQ6RhoEszhvWEpgf4x0Chp0NBkVXSQwL4Vwurl5qC2Bz7j+PCEnuJL5Zx6/MkeTePNH9c7mozpsb5GbPuVkj04s445ryt5AILP+zMzlAWyB2+s9A7GNbSvqOoIWLEBCuQYXWCxIDM8K4TcxaNPV6rNW6KXVJIUCKvqzRIqCkV5a9hg7yt7MGcdSy1DKSvBSRdAvNmWekn83Nxm5la3107rZo9m38++wX8PTt/Fu4cD2sg+7vTis06DpX6J43Wzb2dfzr45/Xjn9APIfb/bkN383exBt4cLzk8Y0KaHCFhAiBTKO48e3lvmDaioiCha1JeyEgNaC47CWsEXjIg0M8EiB18UISvNrzr1I1Q3+8Ps+9N3T38Jkb/sdszF37/qZt9Bp9Ofn74/++70vW5Pnb+fPRiYdOkJvHVMMg4p557VbfZvUOT+6XuKbd5E53drTznE2+kns6/gn8FYs300zXgqoZUpZrUEuUw6B5PBeZmP6CZYIK4pJQ65DJqZvbWoizSb4AL63VWxhszcZhf+iQWFfkfqzB4gXUixj4V+rcO792gxF4GHq6+QT76n9tWs0JzWULVkXKN6o01nQRkuRJEapN6rpEHsQVSqkWikgMrCuope3krgyqpycgnwdrOvZ992O2L+BlnV7eknGTWoQ/sqqksL+Bq2fA5BHKBswTX/DtG/J5uTi04/5k+V2RWWoYlhe3TVOVAM9Qh986020ZaykEtVxsQYCPkyWgaTogpaeQb+hXQSueRMnERH9LQGDCagk9dpVfTxbbPEvbfBDcWG2NZsUyLEcMpVYqx9Ldil/HwNz6UM+WchccAV3ZbZUqg6gtzyPRDs57PHcNRHQy1Qj/AkS00qFk9r/sUkMI8IJ4RKRD5wyRz6N+1TEdFneD5FQDiHe4IB8NVVPZDsAWnezb7kor3dtrrbk/JZVwzEVPsutsunSF58R+SQIzwkXQgq3bgA18QB6OMrxFP7NrnL5/PX8JTgQGXhS2o4qc/LfLdzU1xgBbPPG0R/Q2EGJdvF3cWDg57qqz3onooWhWlVJJxBa8kBSqXEEhI+gUiy4C1YQlCyoEuk3V20mG3gN6HrqkbLLc6+zH43++z0w0YCAGkfnP7D6XtO79JsI2oNzr4KDrSv/bs0/JLgDvyzZmsiE9bT4rENCDmblXPSp2DRcjGVInFTEzSt4ksbpQG2rKQ7bFvaJbVQQb8DTP9q9hg/v5w9bgk1+wy/PmrY0M0eAqAfoyLh088pr2bfCrVj1J68AzUA4F/ic3LkQ/wP6/waVz3EuQ+bhx8TO6L7DhHzHoRfx+ajwXn8RWDRNF1F3IGxK1HxHi6eJEormhxUnOgRkpyjG0YBirjCG25dzApknKFseRzwEdzOKIX6pQ1SM6ysUEttG8qTQWkiFRCXzfyPmgt+uUKFPo66jk0OV6E/s/+a1yHY/HHX6F1jEdDgo67F1QNQVsQg8b47Ag1f40b3Z98O6dYD9gE0zbdZRbxBN0p7FIygxQg4C6/ciVid4xU2MBItH5wnUX9NQsiZ4la6B3RhgYPfBHHQ6haIBLPkIzrjKpFmelZE0R6hJ40ZPwWLQSWW0f4ENLXcWRaVMAkx7QUwsNDSGFpXEFtO8wjSa716gndpBXf2rw265/EFmG/g8TV9LR3wYu61dsQisODHd9uZD7TZpd/eR3H4dvYAww7oavvILGowTyDdOiXiSbhIRdrKlBFvhbmYcMwK0hQUPMkqIjp2Zj36Kly3Cvn1ElcSdstRmD2/n1QfVyros2NB/oAzBEu7WkwNSH9wcCtoKxXkRvNnqnIVHuJKRoScZfAV+GxZRcQ9bWRd6ifwbwvQBlBon19FDyH7qu+L77EZijelltYT2AbI3h59QoXoi3ncUT0eEr+HH63jq5WG+NGzsv8gnEwCLc38oIy53KUfbSJBEUcicP6Yqu+AGn2zghA3gneWYKwQptLOLiU4uE6k6fuYdPXgriivxttYnEGplNawlAHNEaC3inkvZPxvISyYwekv8PoIlv8WmIzCQun/EfGGB/PMxxnvI8c/2k1ltH8TzZ5wt/YGVgHb1xdf6qWIzPW0gO8AVZk6Hc0c6rvzFaniSAd0EmCBKKWg9KDjqFMZ5dMHdLzqyrXnn07fBTR93iG6WgMBFvT9fIJBMLbb/fnBaP8deA7n/J78Jy525bebSy8e6ehygsYV3jU9SwlG5uoZMwjVIjLtHEEjmDVDN4JwVSYWmmD3VYL1WhZcBXPItHiLHl8wt6opdGJpJq9tBkBLuOfElVDC9qAER6ih2YhaWfpebwf+DRG5oAkVBpZWnfK4Vc3FIpAZSpEMMYgsc1Jo0VdVJfo2QOWko2lBSyvxtzs6tKDVk1YEbqMpO8l3nk5P7lqzc5OO3EIn7HYkX+ggUcvOzt2df/IHCm3wkAeNpdw//bTDwa/nYTMnu0ShSYohc/XNAkTGtAbgF6ezrAVelNX47FESigyAVmZwjCWjPTqEGmivkIKFdUbgx1UTGp6+ZhEeEJwB4yRfnrIhEv970Pn30MNBVXI7Ie3Deb4jE55U+jarg+57h7tNviPYrjRsRzqa7N3h/jkmdVTfdJtEMBRaOqdlXwNUrhWcn3aEAuZsQGwzHmmnlOEo6rEoXXSifcOommjKVwU9t3KJxtDs4YeNV37sjNo9T/AFdANUI5G/nn3ZYA1n7uhbexhvYPea6AG2dTySNDhvusTVKPZPP+bgMA/PnIew3UTPfvrJ7PNNyxrkofGDFWjm9Gn3Drr2azqbwp+qLc1/tyyA+xvkPW7sG7b58En0Dxikj4mv4xsuBunr4kahLbVptohOuAbbcCsl7+PZ1wPC674udQ172oe50LPS/+X43iH8qLpWor4gh1woNHek2VCNJ7V+aZsWKIaXJ3TfZNg6HhoYDtNtEBux1MNewGvq+lrIzcPxk90Ly00XAvUM28U58mx3s39EEzlnKhiAgPpxIy0IUWT1g7PQfb9hOcU+TrozB712DrUnp588xfjPaN3nkpkNgonTX1D60F0+mOfA6acD1ra2b+/ci299HyJzywsN89USmix4/pnT9vciLg8WljSzjHHT9jswsLKIoGBor6UtAkUqJFFA5Jj3RooQWdZKM/RyBq/mylt2moNReGF0+LDN2DwJhfstNeC65n+aoPPLfOaJsz8DvAPZBlTu26XktZBgX0mhCKGLLhEpAAUTOm8ljUXjYKxntBjPQqY1PaAAdaEwj2Og7qunpxZ1/lFIkFAha/6qHB6+g8J7Bs/izJm4ZkAR19NH+ELrcd6CXGpaZqjKUyvEeQpWQ58Qaad98V5GdNgygZmgCCPwgjYp55Xwu8DItm4KWn7kZ9k6JHBfG61iqLQNhWmbUAiSATF24EBo6Yo2RJZ9AEGQ0iqReTYgTxrGR7RJCaFX9W+L1OhfaMKJSBGSXFKcfdP4wq+IF8DqBCjK3VG6zbidvjewksp6OlIZCtoz9GiQXImSTQTx55LT9Jo3SUMXbsEMS8FwLipdBdypq4qFybAy+2miaUEpq1/rGpc9X6YDodNtUUvoxTX9AY36pgiUY9Wga9aGqxwdsoIexoBWrgadvM8AOVQSFkIlUKtQAxWTGoFYaDPXqqWUhao4+w1NTJ/hL/nqmcnD91XznbsjGEq/Q6lQK2YN2x+8ubwrffFHtgb1Wpr5mHNPFKJGRx+2CXpUPuhwlS6t/d2ey2HsxZ/bGoQxsa2GVvH/GWT7MdqnC6X6D63vfgx4//QJsnWt/fga6E5E86s5NUfxhUfZxrb7O1qObXcaeFqgb1bOReOEhcuou5aOhypoN2+1tONQmsRoJZ8zeNnQU5XVRRwPtFFRCBPESiC3bql03SfooOZqHrJntGGu+f2zeXtaUXlwseu6w2ndmW3R3spNuyloGZezTaJ7dmAm1fWtr6Ak1UhL/pxHGKeKSFNeieco2lZgIE3OmuuY0F4L9I1BohKkCCTlOemwcuV2cVffFhdbbnO+Vts2Brx/tjwxSFBd3zYzEAqr0EmVEGldRdHDB5qrCApKM6oK1YBVkRxCmSvOitUAUkFb/9BYS+dXym+WSQe88sUZu3jvfz78T5peBY207LU7ALyNbUu6mSFd+si2TFoGI2kBtjh6EIjWnX0iCqhjRjBqy4D42bmAO6GPJ+IRU5bVRFPq1fnTb+e926NGZr8U+rU9CT5xttqHdOS06fnpCX4os/pI7TqeGR6aw1KL091tmfYjFC2kC+n3q7a94Sx/2uJmq3YftLRq7arqjkcHZdod0J9/aO3A1+q/f7MrAStqw+7RLVrF7+at1tMlj08G7OFFH8lnCW0/mtqqmStUFgGvqBSaI6+KtPTYhYVNHEg+7UeF+WIKqgZe4PeV+wsWjPEfT9ub04/abi+aqfqCABTM8f15bafCP3t81lXO26TH8/aoYdBnNK8LdT+lNeA7bb8FOij8N7BXpP3Zt0uRdg3PKg/WmsWNluRRcvPnTZknLfOj1qvRBMY53lIrSUXnUesFz7q675/dYDb0JF37I3eXpsAaHt8fcrm+BKHmQX7/SbTfJ0o0X4UjX34HJgGGRIwCFjhrhdupYBewzvezrygmwGe3Gs0YeoCD9T3D88LfC3bVZ/DmRZWAC15vZOhR897DluhtLZIq6m4XJ6Nc7rwxGY8P9sQdlJb2lnz745/9Ly+wTPI=')))
result['historical_constant21'] = json.loads(zlib.decompress(base64.b64decode('eNrtnXtvXEly5b9KQX/to5vK98NtL9betb2Gx8bsPOwFDIPIp8RtikVXka1pD/q77y/yFl9qSTPG0LbcywGmRRVv5c0bGXHinMi8od++uj7svxtX5aqNV3+0e/XL8t3ou9vjOOx+5XbjN9fjcPFuXN0Y/dXuWN4N91/d6z/7+Z+nb1/fHMrFlVfq29e13LS36fVf/Nyas90v97eHNnbt7WjfXu8vrm4i16ivGOrieHNx9WbX9lc3h/3l+vjr/6aV/E8+PN6Uq5uf/UKd8Xd9tvtfXL4/XLRyuXs3yvH2MGQax692V/vdYRxur85efbV7dVx3Oz+M4+3lzfH8+LYYH+Q5gnZ5KD1nH1lXW0ZJzpU6zfDe2dhnGm3omqOdxRenXarOzz5DMcopxmDw63LgiW/G4ciI2qgUsjZ8zmTnxRs+++2r4xidH5x8vAxyfrz4ZzGkX8/Fp8s469PzdQG/S3Lx/qZcnh9vxvUa/O7q8V25vC03F/ur+9/9g1m/2qyot+vM9odV97/ij/joEvWPDDZvLy/PPzbip77Qz9/t+7jkEqwos7lq/P1wflm+34zg5KLxkQ/fjtLvfp7z6rxfvONvIckzHcblLSs0x4HxxvnDr7Uya8DD/np/e8MH6gwve1XazcV3a8bLId9f/OXPfr37T7+84L9vys3Y/Zfd5cXVKIedPNn4z69Ot6wXRWZwc7gdfHK1P7w7nxeH483DZ5eHdQ/ci7/sr28u3rEq8tmrP+3l3d/LQO/HxZu3N+c8Y/l+u1iubZcX1+cyokz6TJ7piH/328v17d++usJNZJiflXe1l5/94tWjK+Tzbb7n78vh3e31+c3bcXXe9kc+fPUjTzgtzavTtY8dZHmefOv+U5e3j69H+fb8ydONm3L+bjnbFk9iIzHG+e11x4bbxXp8rcJaIGLvcCMjznJ5FFPJHc5vry7Eeg+m2m3flllfXHUi+kqi4NXxT/TZ2a++Of327wU9jjuZ091Hv9o+mpf7/eGb3b3tdmsQYv+WwOaPd9eXRFs/fen4ai3ZeZlE4DlQ8ObiCkO95ad/3j882Q/iQTffXy9Dc4dyY81m1m/H1d367t9f7f73+3H19fHm+8uxq9/fjK8vx3fjcgeWydXf7VupEpo6nyL8BrQ5v+jn+zmPQ8xgxSzXo10wiYvl7L8FIST6V5jv1zKJ6ddP5ofHc3gETCboaXz13dWhVCihKDWcHmqUWYc3xicVne4lmFhLiGqGmpPyxQSfa+wy23flN+fXuILEyLqZcoIpb8bVOGyh3sq1ONOCq0cIdHF1ikJ5HrOW8eGDVz8/jK9/9resStvAFuj967/btcJ6vXr6NBvqPmQH+bbOr+5AEL84n4f9u/NjO8jNH0JQApXf32DEi3++j/G/E+uL6XcnxHGvTyjjdvdX4xgy5u405je7EzLc4kS40nF/WcR5BJGd2f2Pn/9694u//cuvdv+nfHfB79+VG5IJTliu+u7q6uxnG4QIaIzjN5JUHjLWDi/qo39zd29xzXf7qx0ucdwfjlx8syUseVZuebMnNZI2jXqSNc7b23L1Zpx/d9RJFiMmlbZw/cgVWVYkxOy9+fgVDI5/+RhUXtB8b/vzCWDul5ufrLfr4/rmrWFe7puVtHd31nz9/qLzGx9eSzTsPkzkoMy3d9H3+me/WBn99Vr/XWEC33ywAPurscz91bIIPGI/dzdvy83dKrIo/J1w29/cXOKb7VuxD6OX8+vDkGe8c4B1jy7z2f3V/2SNjjq8/rNvdnL/3QbJx9cLo9dK/enf/PyJqzUIzKG8WQm2fXsCJczz9WH/fjmCxDvoU95syw8YkRp3YN6bcbNbbr2B0Ffi+5e3rD0/7Nb4O6APDPnhNPEV9Yfx3cXxNPNqonEh5TlVzCM3Y8bs3qU5tHV1VpOVDb3WJxMevymCdcfHPAGEOPTjOXMTSPmH324fnNDlutxIEK0pbOv1GvPdfL2+fMaP/3QLRkmsf3+zhgVlQo7LUx6gJ+tkdffJzFK06dFWXUK1KabhcnAutO47z+GmjmO4OSFNrZmUYosKvvSYcO3fb8Bzmr38nWgvV1dr+vefM6/jAiRZIuBISEF2C2H7OOcDgB3ScViUyWlBsXEJ0tfL+3vo7FSQsXiyG9b6+mIcseC2Tif7vC3Ht+dvx2WHR6zvPb5AZ0ZevKIRF6NvPOz8ulxsxC46rzML/GBy/TtMrj9l8qCjf2Jy8Nz04GPRBcMWb/XUFYxXdloPzJfgXZyKdfEqpxy7Kibm2tIM/BzSM5k8mU+Y3Ib8cZNDd/8gk5tliE+bPDn/xOTmd5jcfMrkxjj3xOQtqGSc0qPkEOD9JVmIrk5lYPueW2zet2ZJukY3x4/Z2jZK61i/2T5+f5PraGJKnzC5+oSXu7Q4149MHqI39g8xeSBOP2Nyr4INPwjLP47L0e6Qd7FCAf/d3YxA2Kuv5VZfn2711fpE5rWTm+5KO+yPx5UBR7uFq4/d9YWYZLdhmDo7Izec+NpKKeBdkwQx5sVvvtndWeuP/wTLvBYT/TE/7H79q79IC6lJx/32+hLpB2pvIHz3lSPS72ZLuuW4q2OKTR+k12P1l+x0vQGCak4/DVA35yzOdx9t1r3NpnSpYTTQzqkWRqihxWxaajY9jHkJNVl3fjR6M9aa4l3JxjJm8r3k0rIeTRWbixkONRRnbGZYk0pFac6ozNDdqlBLXC4M7zDq20ejaqsAQJsDeGCm0gp/jKFpQxJxfVqVdSiO+bpBfsFnuwUzTHDWZ4TguBtVLP5oWKAHdpl1Gxnk5/4jDp26bV5VnH9GU7SSwHApq+anapbgLcXlOFys+fFktxVEF2yspD/wuvsbf/qSZc/gHj+x8VqF6ExVrhCwTI2Z8IlVTdWei1XTS3yz0ClU4pWFjJ7VIyKNsjI1NArOcXL2ezdZceZtlIC696X7X9/HnINjC/mHXvaNNz/KzME9+dWTmavRcSlHxq+9aWVZeT8mWdK1WKsqDarnYPUlFWbvU5lj8iA4gQ9W+cUTx2+Y9H04P7rxJkGe/vrh1sXgriNPO5SfabB4wwUWyhaDUyArWh7AXyOnNO1GqsY2nKdAQvjLXEn8arz/2I1jSB/88uG2I5lop/fwnVBy87AH3GjOJMmqVX6jep2jGb5gobmAjfWTcOh2dlv7g9L92BNvhYDTb+Rvj++sU6zN5GiIjuCnxqE9TsvyhkFwmwl6z1R88V5HXNy7nLyrWXns4paXnEZmxMcgEa0ndsaIhZCoKto0k06qJ60wJ3iRvLcqjsYvA2lDuaFyNtnPaeNcIPEE+S/L9ckRT/c74fYnLhJEPV0xkeCPoH0cx+G7FTu/ffVjMy1If2ygAsDFppT3uYeqhw9eFdV7wj99deIx2eoxEqwjNT0DFDX70DBUhI2sAsrFVqlYE7kD/q/vJnL2f49ki3tM/PB5Tg+xv+z3j/CJS36ceuSZdn2VBkkR9znocf55hP6b/pAso1Ef7/lzCQLufHfx65Wntvy5lWH2W2ng7ll25IzdlNS0Kg7lgHK82qFctrlxwaqBLU3IhLbqGZ/yLeHmIvZubhE+N29RN4fbq1c/3Hu2LIwA4FG085Y0ri/vyEK6IxHvRrn6/KX6TBhC37+/EiF6jmQ5vzccP5F4V/nB2DMkJWTHoD/kf0vP38hTXm2X3j370oCnHP2g9u4tdqfrSa7t4vpO651E3JKRGPvy+6/uZeTuAd5/OFGWrRp6++5dOXy//QzqrrqA/FbnrTrwBOW0TLheD3LsUT65ur28XD5y94DRnUWlCWqrtDyfWKVcHd8P3P9ayhDrylWtPIP9ZNA1kg+j8f6hGiJMSB7weL5pu6W/tmINSJyfXnl7M9P5Hbt0Nqt0v2KfGCSkMwWLjhYm6KxedZh1/ZOhjEln8L7A4yBmcrKnIo5g7TbwfbVGPmZVrgmiu2+fDPO5Os6qAL65vj1/N+CZ35+/u6irLO24LcDt7KLjS8t+4BblqNIfnRb7K/GOqyNjH7+6W//Xf/13p3X/amnp9bPU8UVJ/+kvf4WjbCT0uJN1/Gq3VS5xFlj8qqTcXbwy2o6Au3n7btxctNeHcc2f4qogQHlztSfK2/EhNz768M7tn1DiM+cjSiJbKYNam+0DH3hY0kXBl7rfnG7ZeL84thR1PnWl/fxl81DucGyVXEnrLD6RqH12fqXRW2x90c4LYzwGISlDrTqVBOKbtzdfS3F4d7p6MyiINn4z2msBn9etXF4eX1+8u94fbo6vxQhb2e/15X5/fXwt4129Aa1ur46313IRYwtycc+D1LDuC++7d+V7KPPu/WF/9WZDt+tyPP53/erD2YJIUq/dStvbL5baWbaTctsUBn5+qnpDf9bH97eXyF0q+Huw7Tfn43BY1amVux5W//5z0YFP7v/0Dg9rt+4jPrMFdbm4PLcG2Lw93l+Tt9rtnS45F9sez+3Ti1b0PZ7cuZSPt8f95fr4z++mLA/8V0znBNL3n/+w1egwEjM+v/7+5q3UXO8mmhaqbA+B5+4vvxsf/nL7EDFzewBU1kbGo0uSOVXYT/c9TfPuAnNHTDeQeeTceas/n493Fzebr97fVS/jrTrbBjlSI34YMt0VeNfH71mlxdH2x6eXCCSd19v+Zoif7B8GUCcsP8HkE3gWuX8G/dWQjqAFjYQtPMK4p5f7FOMZSkzQHH1vxV4yIZgH7OxRAhRh8cOjrHJ+x2Ik5Xz9UA0/3vOXh4y/HvF8lW3PBRa35d/+znOkb83jO3l9Fp2LSUf0JERrVRwertX50bVOnQWtrXUQUKkV+cfXum91enxtOIMmqwiErb08USPt9nCQsu5dKfz8Luue5qXPP58hn8L7gnxA4A12+AaOs6quQ/Z2cV4pYh5P1VsICcjxl1It5c5vyePfvvrhh9NeyGF/uRlobRmunx4o9Yc0dds7fJTYFUlIZbi0syE4KXMtX7zbENzqsJ/M0iFY9+T6J5nVBx3EwleXl+fQj48keQtPQjvrFLxhBul0sbjc/Uhrksbib9YJI8WoZi3cPS4x5s3j594KK+eL+N0/88Mc7753V34xxj59hge4vL/G2kda4bho3/0XHvP93LvlcaJrJQyfbB49NUIrd0PqcXPEHDyixY+BJjWpo8JEq0Uiqvi5trYf0tkd2p8TEyj38eimGLRue1+//QDfHyW4VR/7eGKwj5H9YdTy5jC2zXyJAFnblZLubXYPWiuPrkEvQJ0J4N8eHsHklnZIqYKl/VGGfmzce9D/UWTcqYtl/t37t/vjeJQsIfeP8ukpOz9kr292p+lw2Q159QIKM67279AVN7LH+TgbP8zxUTY+fiQd7+e8kO3Fu7z82SyzkdyPgf32m8+Duf7Rgj38yj3dSrxjXkSXIM6ZhuBYWKwAFpi40UoJuAfz3r7bBNUGNpffLxTq9xZ/c9jfXp9wp+zgwLsj30f13dPaZYFvFqF8/SFt3N1TQOETd770oDhFOi5o+IdXiKPmY3YjKuebydDgVobrWjcTY05jzul1C1mVrnNvGRHRUQVSOFSjObkDUJ5KcM5Wo3VoJarBV1Io1qSJuFZI72BLzrHr0rqZI0c5YpK9C2ZUL2OYmFspdXSTSiyZ60Z1o0Xt63SiyXM0saSmmGdOvvvA+OB6b57B1HpSU9b+0rB95Dqlfg2Q5jgqizj1LHo4niWWztwiUr/PEmasfFjndNvWpJnG9thD06kantLaYlR0DBNS9UaqJWjJiGny8CXaMpPzoQ/DgvNcYxtjWONrIT+T3DRJzpceZjE1ZxewgJ9xZIeDdF+K2Im8H6etxYUe0qo2WO5hemeEXhR37IW7dx3VHD2OSeKHT5cZ3chmpsjz9eCGaSrHMnpbYzBa4et+WD2iHUXl4p0aOnr+mmwlUY+WecCieciAiYPNqXs+9WZuBSmX5mTaUt7RKk9lg1LoxzmDj72NVG2YriXrgy+1mzIL4tK22TBMYk5rjJGV69OEmYr2XIF7jVYHA6pgWhsq2WkbHhBaMFU7MFgZlnHm2FpbY/BBUg1njGMk/MPo4QXcyZRtTGwdlMUrTGDAbmIqs+KFHRtifuvnNkZolgciw4ExNpSQcDnxKuc8zmCrzRXPn1ZcV+nWou3BW5xG19ZXmcy76mo0LYyKr9dIHPBEgfvqOULUIzB6qmOoOL2auFwIqZmqsAfjLv/w0ZXGFLrTtdo4cTYydDBTF6ZqdBlih2x6VHV2njKRwhkt1jnUVCteWEKih8ELJlTcMU+tBxo2OVJdEXolVT8xukKVGFMU4Y0DVdt03J4lwL5KLqkbL6pP8V8CHovHUBjNtOG9n9OQIJ1mXPy2Qdr89GkQptsYuiY1MZqdyg3UpjEDzxUEmRbf1NVYbVwjYKzs9M2SGzbGpabzJ/9gzSxQ46Jt3vL1JLsCfu0KS63Cd0LEM/SYzhWttdI1RJVY2lgsobHGaFI7cMUO5ZpyJfeqS3fKapwjxprthA/Y5Iprq8hcHEGffauNJ95KoLH42HJRFSMkb+3sWUfcTnYvTCI29GAFTK/g0RyYYW1XTAe8OUugrTFGwU91lDCNPXdcvhZWJFpNbHODWrF4ilWWog4wUwEMBJyNMcBNZIxkARbiyqeUvJKNOTEPV1dfWCYkdIP1W0hOKECAwkBujA5xxNXcWGo1ZaAAPI7Va/yABZulaZaB2YLG0+Dtfg5QwDG/HjWZqhFY03dYUlg2XWbIaoqdgvdgvISO90P16h1YaGKICRechtXEqBimArT8xzQdFp6SABRmVGshoCvEqreq4rbRzx4DroHTEdmlTRJLbNHbImd/cnNA4CqJZWCFSGEZux12uIKLZGsMyMp4wFHJYiNtxTrEO55gje5qFIC81hX72dqR+qgJX+VBSWQdCDGkaGDc5Rhn4jaEM2HmPaCgJnMzMEInCWUdTsDdyADeFOfzzBLprZaBaKvAfwQxQD9SjfDK6kflITSPLJnBjkkGXdlYkQhwOS53Gm4whsb5C3hZAGiWWRHNRbWmFaugC1MIgnOD4HW6rbUt2oXYcSDvs27iYZaAd4WLgT5yLhiFQaeG+zKzUJuH10twJTk4tZ6lOPwWZyHCtUme2zkwynon5Ni1GtADU6XJ+jv8PpPUDUShJSxODG3MguAEr1mNOiANJRrmnVJozsShSENa+WInqGj5zLbEcxP9/BwYaKyYw+OSiROEcXnEyniSV1UfThvWLEuKx9GmG6oC5Aon6tzCyl4SjrLtveFbHS/qFQmlCVdVGcgnuBe43jS5M3VQuxG/sBs7XRiyVNmpCmFYNpUdKAVOQyxAj0RKQpGLwiHPZidw5QEbIQYS+eQ3g3+lrjCqxYLLpg0LZWMrbIl4qyycataQuHEG2UYUuqFt7UFIF/zCFbwCm+PZ2L0s/GiEiCUPx15xPyIg+IpyAaxIoVqxbLhfxl26TgUka9wb3oFmBGWdW/mlBWjLzGmS2xlJ5WY164f+TSt5O+IAfM3eEkGgfBfW100m60UU1LKHAFeWfSqHIiRLofZKctXgT26AHRATDd+AEICe4HXUiVkoWAHeRzJaJUq9dph69jlZG2VTFn8LTBb4lqSftGTYUUaT6TiUZRIf0UQiwbk2gYcsfR62khUwphwxLuv5VBwwElDVQjxMMKn11skLwjGGzeBexI4rVw7ooMTpANgKuCCxx9IN6Azsy0sgIKALYZ5L9xkgEcPPBsiwmHXlKGFnUpfxVlJJI92ZkVCSWcMRB5HHAgCrRKRKZBhbkmz9GQkb0lRY9pj8DlieeK6WfWWypoudTJP407C8FmhwKmnZViP1y4ZVjqkHhtJh1lf/+MPTLcBP7H49rSsYeCb81eDNpAPzL6grBFSy+XRdwVipPX2msEAoKX8GCCinvSKWw6cLC4Q7Vg/kp+Vz8V9QWDg99acrC2mdQPl8ZQGI+j1LC0GOs2Wb5OSplSMRLBuoheeHMAyCxdcKQ4FZQOCg/JBEUJfcKqu4JbQ/sLSwaq4PtYX08dqCtz8uMOeP15fN71GF0OHTZYhgP1uGWN/8vcsQ6+l+UnUICKv5VNVZafW7KxHu06WIdezvk6UIp8+UnIYlTqV26n7qlQjwEuSGfpEloJLQamdEGZEn0aaI25EMZEpocY+ZFOCGHDJH4ZCjbFsMjdhFRscKQxikkST1A9ALZaHReYhFTRoMkq8RmRC0WiFQJHPhfSG3dXpEyeE9kjakHVxvKfrSDUw7yDs35GIYPAwCEtNrA/BgaKU0CBJSiefe1CpjIHEM3DPDOSHqqNCkuCH5WwlVgkvAP1FN6AnIIIzWwdgirL1UlxabQBuG3lACqQWIBCKyG4ijR3ZYaCLqlFxrRX7wXCbVnkxuRo5gxmwgUWsMKJ9G6kjuX0mOBNwhIphZTpYMFCEGtdCEiBaHRKAfvYEDtkR+2CozSqoMlQnmmrScOak9JzRjKdA0RBgspU2YckHFWx+QJ0GMxDIj+MmjawzEU0DblQQR5C4qQxBZYiYELezNkspFXaFjs5uYBvFJCiWjI1VhWdsY8AKm5oTXwpgqrBOKl1i3rqQyjDYdXUoPmMIO36IcZBvgtkHcn9Z25qJLESIZ5ahZrVA1U7oUb1AUQ5Pjh5YaF9bJo3seU46LIXeHzmUlAK0qQxeWw040E/kRDawrdJSpdgUzhFKpIDoX0QHlTA05LAeZKqowp+equsFGEeWkXo0DhIFhkXoaLYkskTNJciCX1Az9wqHhVchB2DvcHdkR8lYhQoaIG+I6qXsYi+uzwzmGJ54MAVfiSC5VXTu0k+CxDSEdqsL7hc1v8yiszGBBHbNA/cYi5zV4cnnzA61luBnCMc2EFxhuiFjQZVpZZanQrDHgSfDJQLI1wmbJxYSxzBphigQlzMENmyvCGQU5ZMLDp55qg01uDE2jFkqPdaK+I1aDlxusX0R/o3eRUxNfEErtBk/XmGBbgoSfCa6lAowxUkAxEyGLZxFmsrUTeIyaPDoOEFAdJJvoXz+heSyfgyngEDbPsNimQed6hAKElIUwSHueeUhtiHVs0yJY8UCpooB2EFoT5DCvaQHNDMXPW0WViRQ53emQpnXghEg8XTxSOjA7sUeWKICOK9wEtsbDw70AVqKmPltVFijULEsbfEnKsSLjE+6tFPigpUzieUSHHsdScqI7ITsbfhQjKTg8W2UXpOilEZHCNNG4cvBr9MjKYIYKDGlGz6CJHEoErazDl2B5Wk6bm+VjRnSpILWWekfULQMgvsyhOjpQMyTha4KUKEAQ4nNKuRVV5YkftymaL6bCTChliCz81GonrzMwXLOdDGFYPxG7pDxI7QxrABxfEfWuahLU2CrMVneMjKL3EXsEnp9E4xDKNmL3WIMcPURGVtWKADMAhsomixGDyhT9XJVui8MC3eQQS/JyKPZSkYzkhBrQbX2Q0VwDBNtItqhMQJYMTMnrHSW7tS52+kCkQci62JD1yORFYoF8rRMhxLI0MZZqwFEyiDoQGlAGUqOPYxsD5MS5wEs0JdyD9AWMQiLCIIBYANiFPHzQZpX/PDlGQhExi5TYKu5KM3hEpCI45dRvBHJRGWS4gJUIjIFDzhh6l190X7EzUEOCd4TtsikZDh8fpFekcyCDeamAkJ7gDt2GALxAQBqPLMc0awaXcrSiTcncrMfz7R4MrxLYaEKUbVc5BK4zs7CCZyjvbuVlP1wMmQy3UgUhzvLO7pIxIz/bDkTDoaVMoMVcRuBvQAWdhS+RABvho6GMo5APSN4oUKlmJVcnKStvmMxcJbc2RZTiSWOwAKMrDSqmXIvUJRIf2WFwDV3lcHUImTlEwXprn2snxM0ir1JJQdoUFaR6oqLHXVUhuxAztToeiIlEBwB6UkqrHnoD1yAg8mk3hTuWUA2xL+VciAU+akfPCj7VGbUWK/Urz9NiLGxETJaEv1ezVdy/mB0ZI9We2J0ho4N1QmgreRfOBwNAdytJNo3vDjOjRGiYlTQG+S6EVTvt6hCyhFOGpyip+8FFMxRIAQ8Wip4QIo3pAw9RScK3IaPdUsYN8oZjX8zOUEIFDJ4fKlolJyQp404N+rpMTm0MA+HLHZ4Kx9G4n8oDdlmQOqDfGqNEn6S6YTTEYB0ySsYb4Q5AbXMd/Ik8Fb43M5mWdR0mE1tdCZPfduxwaxYU9PLgPlYFdiKpgGglkElsJPHSQsQAXo6GKsJpkjrIWzMDLF/ULhegZkDXNFlYET7wfOJFUMQqPZfm6F3QF4azMhGJ05NLtci0WZ9tp4xFDUgVQq+P2BI03C2JWgK8eBjyJ8oCUWWqSBhSgPj1VI0HxxWWRg5ClfAEEAc1pRXsEQ6McII6wkdZzowSyxhxRC2lW3IUHBmkIxWksu22edEkEkamNrI768MAFRiXN2lI+ZbwRDEQ65IwQQJAwJLTgANWVz3brh9avtrpZOsIJtLhz6AWsJS6xUgkFzuZGquB7wKDBIIK1SK5IEYg5RojAtXkMBaRW0EXpHyLDxkyGhyU4DRyKgiEAlJJfZX8FSrUL4Lwc6u44+foKpPFTMR8jwkvBXGD4vlkm7Ci9SoAi3JBGTh5tcii83APFthsz1ISQapbgRnAYBoUF3KepfrhK04Kc0EwxMzqZRZfSBGoCODLG89xe2HmOXZBIYiss+mTLMpT+EbWV9CahMvI+amMzE1DAZ0gXrCImtIzBkHO5dK3dz6icNpSBa9EFsArMasDZHAnqQKgB60r8EVMyodw/7m2F3v2yPi6MDmuUgJ5RKAePk4+w7Ei3LZI2k/CI0wW2Q+5kL2dob3EBmuEMsjl2XZ0ye0ZCq8TeRyMa9MInzdSj4JjEnNRKwj5hEpD2AfMLsM7hqhyA6V4rl1hsgb5AG5OCihT9H0DW8hIrBbYn5nxaKItfcvy3krD80khMP8OEKdlUzxKur1kyRdEFAsneyrOeKmq8IhaQgGWpCAhNshATUPGOwoYsNduG0NSCsQAxkS+YBEbOCd2bYyBoxF8Rl7N7YvMwZ9YmI6bwJOL3TRhwltk8wTwY6YwB+XQ7uRrSGSv+BLKDMzjyxMWkDI0d40HXsNHUt3GEKUC0kHzMZlTuCnptnqCIcr+s8d7MIuRXR7CW96McgYCJKfhSfTPttMuuZoAsFEqRZlFRS1GKDUgi25bUJHklbcK4HS0tgf9SdDkfHjvtgv6k9qtV9Wxmng26RSAIy3haahrcrhH3QZ5qRQVzvTAy8jq+OGAWoLFVwLgi9rxD2PiTiQejYvVVT0OBSlc0dMpBSFUycjbFVF2D7VzzgAT0qej9bph0HOcGkBuNiTR5CHRaOg92KegxSpRNIeOUgQXmd4KlYTC6AytgODAhiAb27PggjgM6OGKuGnCcYVoAFqwXnkVFCfnNjwQwhF2rKXwg1+jFKpNpzEQOHA5krKV6qORZg6qiFqyUiBC6uDCJg4HcdWICFyGr05Ib4t9w4+MTCAXQC2jRktrHACI4KbTipvyzYbYB0kQ1ggsqHSoUupXuAB0cWnTDFGEf1WpiPahCF+cimACWGGoQdBYyDoxOj34h3xqNkNycTgvlawv6iQGaRajJ5yTVC5lP8tzT1GSKnZGhpTIzrpjKrCtTlSTf0cnjwep/K8xICw1kLNRJqBMgIkyIHwHbmucSFA8FEmaICTeuQQMRXSXlSitZLEv6kRIQivB8yzMgbxJ0M+AAyFMyZsD4s0C5yj1Ii7B1sJpyJEkXh1bNc92qgQAhu+3UUmNQQRrXm+TK6j+ADcKVgzIcwifTivBSKwhj+vavdhwTI4qkMKgf74mMIcUFOQ9Z26CLMzG8GeWrX0tYjnFCeEb2fcWy5Ai+RqjDw9ThLcqWToSUhQykUlt5ExoUSeZBCkKyFviKiwYshlYB/ftlucYbZQmlTA8nqlHSFjo8A5S/5TtNIE1edeP9bUNReAkEgu0wa13h9cYk3QP9R1SWudKSEGW30Lc5USL1EglF7cgb3hDcyCPBTJW5ZX1jrJZp2yUnPGQF1ekFMPkIckDWilEqgZENk6sbba1tqbQ65iE25LjcBZm5LcxXAF6oBkwEFvAxanNIHUgvuDWisut5NfYV1HGShV9gFsB6yKktjHglNLBBbE/ekHkB0uCwBtFVqOEvCCDRtx2OSMiGV0AIst7KNDytjUSQLAVEoSXt4oFyr0lxYGyUq9dJXumJwUcY2FtSTYQEINAv9ShweA1BoIVEsYfVuAI8/HdCUcL6CZYG5JiktoyiSBIIRLq68JwUhlMIdW0jTGJdXll3iNy4a9djjkZOchC0M+a8tIbUapAUU4hyWvqkGaRs7Iv8mwnqKTclQV3IlI0rJY0kscjbIIcmiEFkWeppHRiRvY2EWdLzaAk1bbHV5myAPuQoi8sUwiMZFToJy5VYHuKfONAiwCHmEJ4BjoIpqfJinObhwNT4A4kMZgueEEKQ6GDi/AQ7RUP3VAEKFxEFRwm4ZSgOo8LnRh2WxfuChDDcT0ypcq+FRTP8EytkzyBKiPrYGVfjBVDNRJ/3Cai/XCS8Vwnyqp4EQnUGUn7g8lAAck5WoqmmpzJbJgkGDJBE3JdlaoL6g3FAyCtfIsAH2RAWBy/9NIfA5DBPSe8n0QGa82yRSPq3IL5uciWXYNFIT9g9lsrDqkbI1wsJJ3UUguMyKcGEyMp1iH1uAFxt1mOYAWsXXSVMgpQF2W7e42RkHxa9oK8NPuQkoNvOKWWZG99x1Kt8/zQOWxjhGNkCzZNMHv2tsUtNpQWBwFw0AW2BgxVMq2H0eGV8EV5jwf6JAfMSBxT4D4gZKXA7jYck035WQaP7/vaczH8pnfwBsXf0VIS1WQ/xCmaIkMix5CUgTWELm/zkOYfXjplVocukx0fpch4na9qNwEEHlAxcpCOUSHA1Sc0PstY3HrlqIokAhkRNA2xgWUVMxqoqxZEbI0ML0ioCC/uJRu/ECMMgZZwotq2eGlgJQ7Wmo0jWgJCLK8mmMhTATKDRwe7JjkIv0B5w6ogeSyJQYvP7dQiNEwoqLy4B2HiQcD4IQcICeeeED6AspTOIKP4rWRTw6UoPwPKzJ/ayUcZAwDzDUj1Fo1p8VD0tZLnJt/Iq9Q8pJbaOWrNKrkRTIEwIhw3jtugLdAAkL+BWcF2tDtPLhpCeCHJCj2F7nIewQwZYDoMXDW8tqK1l1YnOzrmmD3rQ3Yck7wp4QJLERYQg5w6Q98RGqh8gjeJKMwBPoz1Nz0n5RpCQvSsw4Ir/iehioj0iBhBBIPiAf6yrB95yk48EdSeuRDfawxnyCyyQ4sclYY+Qd6bS4T/2pxENPcyUHbggxyYRP9bOSyCavMSxCteuJtd22ARN5PaK6nUNdlT4xEl2RJZHhMSmlKlr4Y82gMCBJiFpWwnUslcnVRAgBS4OiMBRauxDKouyyEVRdq1Af48pxSIm5x6BOys8QUCtjXWs67Klj/JBPclVshjIOPE1xAz+DfgZyEijUQqiQ6ikcT7slQi+zZGV8wIj9BDiLgxUkzWGZwrMuEOJ0M1Q7Tk3IVCxwTp2SNqjgGVjtsYcoyoeCslToJR2sIgFYKc80CfSX4SDm9l/xeYwFDitwUu6uXsitlO6NqJF5KuYVJFiueyEwRR0hBIfA2Pg6IaadYSNbFOci5p4icKplz0trYTtZk96RaSzIIQubJNTlDBbWfrctBg2GQj+UckR5bWNZJRw6gO53q2k8KKAVGLM6O75MgNRLA4Nfg2Nyf8cPqQBI+LHBOQIz+tq3V2SOKpflGnjTEaObzKfgt6BQqTEeQoBFyMcaTEImVh4f3wPsDPMpcihzuClYM6awy0ufQzlWIhjwKLhEkiRMjbTBhUR+sjwCADHdyH8Qrh046fCkRCr3iBrnYDn2xyFIAxamRa2ARqmWpEmfkR45xQY2HUUSpUFgfkpg0Uc1sDJiAJOqikPudNm1jHVakMogiFicH5hSkoqBTu2J0UgaocPIiKJL+dQRuIsjK7kQ0VowMLbZ2U95V0wyLPsBQNMcVk0GLQNGdku1Re1kWKlLrNAxIPlVACNBBb43F4KWBKl6tIRkLwNCNHfGyLxB88ORH3XaF6lLze/GynwEErbjsm+EeyQ52jvzyugX6A4cshKFKMlWaROBqsFVsJRBrEOEoob2OAgOQkAAxv8w2AZkjZ8sUlwVKEAlA6HbFCuAbxNKvg8Giv3GNw22l04DVPUbRyvsiin4UYpGzAYlFxQqCnHHWAJWHiqqKwythknzeo8mwn2qX5riKwjBzPQVrKcR40jhyuEPdVSs6f4JNCbzK+luGOpLHuh0snTIYswHKgoLgTCwLraNVNOUApM4VNMJAjDtAqhADRtLJPgOMAAGM7I4CmAYBMSnL8p8sWscf+islI5RtVb+VwvcCvqlB96XIu5xB0FSk1tz2+KXRdiwaLcnZQvN7INbgp3lHQ56s+bHOAt2lgd52TkZNAg0S35bk/+IT/GoMIRFd5OUKHc6LrCPpZwA+pVHnBAzmsBArjoqCXV6Q1MwbJE2BZsT+hjcQ70a5F0euGlOpyFDTWLIcrZHMBh5fjg0INuAB/BbHgCpBo1uXDNw3OP2gVJO2dH/cJ40mEOvuOTg+oWd1nkzKwbKlrLefEmpy5CsKXM16kJroqgayAKaxl69h8dVsuo3voY/2kV9Lq1bL6OayGC4fDaOtsdLw7lb7ao66D0t9e7d9f3bWjPrWsetL4Tm+N5E9Hk5/8yjzpD/EXciR/t83syanm+47xu8tSx+Wp7dajfwtA7dZrC99I460duuB6NYG8O1QtvaJWE5u7frenYV6vjkCDL47+0YPmq6P/4bZuJ/9f/c2PZrbmcf9AX90d1r4/Fr96Q61b36/o19I7Y7XTuZUvne3+5vby5uLrrfvI63cXRxnj9cWVdI25OErT79flXb14c7u/Pe7e7w+re9GdMc9ctLutwebx1NTyfXkzVnejJq8SfH3Yy1H2Hs+sP/M+3l1Mrt29u7i6vRnH3f5gdm/3twfpX9nK7XHstsnspN26HNiXHuPzQmy0PtkMIif2V9/LrenlqYOZOM5de6VyfX15sTUoK5eXpy5mu/FP2PDy+zPQfmvCvh4Jx5M27Pv9pfvNbvUMu5bGmmKhb8DZuzbIx939ZBLGvxhX/biW92z360cvFtwbW9oXPXrN4HSu/vZm/46hpZHR99trBmfrlYIfx8TJ8z9+gv5ZTq7HGOW8L3zA5yC7VOjPdb6cHx3Y2IOxQfMFOZWLNJKNS5fQUFKgWplWevdCOzVYHCvg4qVTcu6iy4pAMAxQjuvJfrnsRZHFZctMjpAk2bBPz3Z69TlOnj7Dac3nOOH4HCdQnmMn+1l2kJ9hp/JZFOUzqLBnUQwZ8YprFaFEwoPlJQDZNDMoHOBRuowbROrAxkqWpynZZCTqYFxwvC+GeSzWsP1bKf/yBkjRydl9E6GxcvbM/Rs2QDLOhjMEQ5T9WPRH/vR7ivi69VYBgdJVb3uj8aUB0u9ugKRz+vi7h/bjfZF+/O6i/n1aJenPtEqKn22VFP8/b5XkPtkqyf3uvnefaZXkP98qKRiPyzo5h6i09j/9VkmMAR9pSeo78BUbrXJm1VHlXYMgrcaNl47oOgnUyNYI4DYSXGluh4S1/L3Ifqm8c4f4lBcbqpxNdqQccnVaWxCEtCrkrz7lZbhaZAMUhavCs73Y89Iq6aVV0r92q6RERBg5HspqYVE8oNvhtO6wBQb1eNIs8hRytrQkLT0EEraWAxGpxWc7RP6ltCh6aS/00l7oP0J7oS+lNdBLW58nm9tfzCaZHCMLg8eLNWWSi/wrLjwRmUpkaUNgNbIGinSYUaq8fOUHyjPK8bS0HeAc8g8VkUVU111PjTV098JXMsgqj6jzNIalSRZygnNAVzJeBgcaxO/4oloDeXFxYt5YvKgnaR4g/whTl9NSWiou8iYrWY2n0HgoLKLDl9DMcMZoyr9be6GIZiaRyr8QJi+VuX/L9kJSIzxTWRpGJOm4+hnVTrQA4Xi+Tlvv15fmQp+Q7fZjzYU+otvdx3T7+kd+P9Dtq5XT/tTI/ffU8TqmTzcb2v5F3E8Jef8vajZkf4rNhtynmw3Z391s6DN9j13+nJi3nmCMxssx4izyzb+0G/p92g3J4TrYXE0+9SgyHS2hI/IvTi3ntCqsFDDveXY57QA9t9IvxyYono3P17JINWipRpzKKwEwuyo6eyghRvLWOyotwDaVnHmEt6AdPCS8xrxtrXxRbY9cr0iViOYdzEg6JiTmQa6Ha8AkTNDy7mBc/xCp6UHUN4/pUTvyjw67Z2udhPeYOeQZvTTZgD6PmkmVHVmWRNYMRocYeHmL3Mq/YoekT44Iknr/Yp8qliYvV8vp1lQQ8ZABeTddGpTgfFD7HpNjIg5Pw28gvyiGKOJGTF2+rBZOP6X2S89QyXuO9kvPUA38clo4PUP7JQWIGbxEKhHMSf594WjbkO5PQXaxQW8ASIpweZU3vbc5KzlPhGzaqnDP0sLJSCscp6TjVHZN9k890SMVBUR7FRE3Vm8yY+QUtwbXeOTOuErPUwXtWdpAOYB6puVogm52va+SpOVPlW4XliVEAQlU+Jal7Y2WrKVlJ87lbZPzi2kl9RwtnF7aL720X/oP0X5JXt9U+BRx2KRxC1dPWTUjb7o4qaloBVdIcpAUJeitHO4zVgoJrOdLC6d/pRZOZM4ufVtsZ9mZkUxJTsk3EipcZUhpbPA0ATOarMF7gcZACvIxbq90vrSBemkD9fldL1ccwYFsQgdAkjM8AQYGo5PXI6HNya6TOtIGJsr+ipUuLAQtNByO8Gw7Zy+tpD5oJfUcbaAIHRyyCFkjEeQ2a53ypl8SFNWopwqjF8rj1zvmWgLZFqK3spjz2dpAfSktnH5K7Zeqk14TcEPwhhkHYk0aA6AiO9EYUCKOBdLyIth0cJwiW2zDTbiWgag/WwsnxuDmVhrgRjleYyT9WhGpQu4F5HUmqKQzGu7vvBNAX3vGSV6R/qJaOD1H+6UqXWI7Y7AQWFSpSlgU8IcrsrwkXAa4DZNfLeECmlB+AOahvOTNbefdDStKBRxrXmSUvHAzxOXgKpVvCCVKAvYebYHPl+pInIX4hoT5lzZQX3AbKIMzGSlroASEBhmHj0CtSSdEEfHv5WUjXcmFQ2pP8jpul63aisttLfG+mFZSUpgrJNGqB0xd9uFZabxB2u1Ix4YEPI4pPF+KZKny2NJYRuyMHNqeJRHq5IYGUeJxgWWcnjEN+pXZ5QISNiBBppewL0sn55CtAoSlGfOznRBp8p4o+KqFTAotrTBz8ta0qBQchMRUpyhxrVeFy6zOU9LLsY5T3fKlLdZLW6zPt8V6hpZWvXdCOCNqpPwIpQeUpVQCOVZgrfQHBzaTvPoP9cQJNVcQszAlN1t6aWn1kZZWRrrZSOcneIgGQ9AQWJ/ljyokiOyQhuoMJy9C4ITVCLRDzTWMOI/60o7qX6kd1UsrqS+0lVSEJSKPg+xiWmmaA48IWQolqKAm58YQVWaiaFCf06nV6kneiAZQ/TO2kvoptYF6aeH0pIUTgeBkkxlZgnjCNclZqzGBkRfFVgVfd7ekleW/snkGYDIXGNGpNv7SwumDFk7odjkpKB1NWOcsrYCgOs1IVRCGxlxAwyLlBCt1RGSeiDMcwhG22+nS52gDVaVNgK+rRwR6ktzFU0EapMaFGJB1kgYVxHKEystJCnQDPsNQUjpRL22gPtYGCv5OgKCglF13lEY4g+Dr0n0Jl8IwChSTljlyZmFKRyKfoCwJ3V+/qDZQZr0pyP/HKkjzV9l1l7I7rE6DXmFtAJPmDORXqt0wnIxIAJPd6R8VfZZWUgrC6JO8mo7kHDxIA4YxD1nLSpXUWszkpF+HxK2N8PIuPNeQ8DZcf2lH9dNtR/XSSuqDVlKuwnsBXtykFWmhPPr6V8/A8CFFJThz0YAYzynZGzBv0i0YemWhVGE7rd95NhRZEx4DciCJSdbkYOIZjcj0cdPQFQ8f5Gy7FAoRA0qaVpVT+6XnOPH/pbTFeo43D1jVJJRPQAuljVITKUbUECJSfIPbJpWHHPq0SYZx8u+NVVJ7m0zspT3Xl9ye6wt5I+RLac/FYuK05IHMDNATSUF5iAU5LUjiiawrT4pZpf4KEhTZY3SQyAE29P7S4usjLb5Etka0pjhILWbKpumUhuxeejInRaDVZNeuDGataK0uuOG8l7dIzL9HmzDZXMPpC9oTjVHkbHYe4n0i46RjahdemPAOOWZmovimkHMUxJi1zz+gTVj6dJsw89Im7KVN2EubsH/zNmHP8XbEs5xMfmnx9UW2+JIdDOFg5FYESyatqc4NQgutKdazq6JkZ4PMJ8w4TjkkKYRmuImeGC9twn7cJuw5lMsX0mpMgO8uL96/iXizj37rPmaRbWdJCnwOrZzzahD2kevv25XBV/VZQjpLXZAvuB/+H0USilM=')))
result['manual74_augassign_protocol'] = json.loads(zlib.decompress(base64.b64decode('eNrtWV1vW8kN/SsXfl5sSc5wPtqX5g/0JX0LCoMzw3GE2lKqayUbLPzfe64SLNxt0vpBCwiBX2SNPJdDHp5DcqRfbz4cDx99b/vuN39ebt7agy9z94uPHJcH25/sfrH9+smPK/6OxX/p96d1d9ivPy9vTx8+3PuD7x/t+Hlph9N++FjevP37Yqexe1xsjHX522bwzenuzbru7vbLYX//+S/L4bi72+1h+ujTj46zSymLf7T7kz0ejueTftvz1Yu1H46+4olH2+194Hj76EMYm+3+87pbv5z607I/PC57/7Q8HrFxt79b7q35/QqDy2HOXd/B2Adb17/yzzc/LTfr4XTsfnt++HZ9b6JpA6JViqFwbmG2zEVC7dqKcsx5jioSS9YSVIhGE8EWbJTUagjiM8Z4Nr15eLueHh7suPMVZn+94fKnrER0XpxjGnibI7ZjcfT+iCUrlp+Oh/0dFlqxOO3/uT982mNJT1jCChP9XzPxmZlE3zBTX+JM+L4z8tXKS5wJ33eGNzNCL3GGn1uRb1q5BDLCL3GmPEdGv+EMv8iZ+n0z9PT0G5G+PnH7r5Ovj5Dg7Xtb3/83rd7dEEiaPbSYW9Oa0gxJYyqDc8fb2HMfSUJiTrVKqpxjtVJikZpKZ5ONuzy9eZeoJY8WZ6dtm1v2SpP6TFlT4eC9lGQlF4k9Srcqw+oI0sdmQ5RCocEpe24xsLrHWcesZZoRE3RlozeV2qbUSUNy66LDpUgKs5xtZIHdVDPkKNQcR8RGbMqtJZnaXEat8C7FRFarJkIIJUotpdtoZxtTwsgjdS5NNI4QTAhB15wKTo8ZplMBZrW6Wg42S9QENzTE6dU3G0Eh6gKBpwDjLWqjimC4AcYKhJylN+MClCM3RwYqYhsJNptwPuORaEPURpmp5cw6XbzkNIFLIJ5VSOsYcBrvK4AZnXPSMAa3PHmeY8mjkhjVMAAfDWQYL5k7axU8VHQCO4mz9NnHmCishnCGTq2FywybjUJ1Dkf4bToz2ag9CEeky0rLxhyLmiSmGTqo02uHeRnDLXgyPte2gtQruDAnIFctlibqpaqjHmoEppJTLinZltmYSbZCiiDwIp0TbTZcZgS8oeXSNw+EivWE9FB2nB44BbWGcEoHlC3lWYw9wG/NAcicbdTijJTCc1PPmSwAwOkuCU71PqebtDQdsdH0WTuhim9s745Mf/GjQhLsCVQdPLkH5qEgXAWgLY2qXKfI5FACTmo8QKcKVKiTC48zP2algk/mxNOKYwdQzEgHegiwiIPDtB6pAIGi1GbuRDWXkZAUTl9yCwZk7GlKrgF6UNLkfRprCxJZQwYF4Y2nxoCrImHD0G+8QAswffOP37WGd5eR8ZVI8CLyaZt2wHGQAKVI+5Rs4KuDyOChkEBVsvEkJDgCVWfoBn6gWIq5XZN8kIDglIjDcJ9siajBWaBbPJWAz1pv3AZ3wqjS1DeVcUVyQtpi/2IDtayOGqyRwSVLPDMqWtWOd9aro6oYeWuNDfAg52FMqMGop5rOeRkDXSOahgQksQmw9Nhiwnm1o90YtYSshC2vGKHAId8WDsGiUKm8yvhbMq7PGvpFFAhBQQ/ASxv0gZacRyjiOEM4MZoImoSMiJE6omlTwkg+tVPEfzClp8s10mmg2Oxl1FiJOaBzQcgt4DFmjAQVLA7gmKlR2Xpbs1IbgwoNUZ1t4GMISgTAEU4chtMx3oAZA86KoqzhcFS6KrPkOXmk6ILql81Hl6uqJheoBK8K/GMU+KyRXmKm/pFkfDXyGbh2Sw7ohW0kizUiGmbPGH5BYUCYNg6gQjTkt/MIIBaIhDbWZ+RzLM0wBFED/AEzqBV1hp+km/91eFQrYiGVrboUCrljNgKzW4c4em32KsE/SILPvhJ496qe36nnEo2jgYPgekCYXdViUgx2E2lG4rjRBPGxFeeiUTt3SADToPiWVULyzn4AsRBaQ8TDs/feK+7KmIgpRBI0+45DFTdPJLbjxteEKpTKuNhFaCXzNd0Er4n5z5rPD0XbK7k9XaLoX4t8LtJ8rkWCqsgeLgeMdkRgm2XLESE3H8hjyIO449bAE5dYxRpT/0zbI2TJWrguGfPzr2W54QqCoWeaA7aSvQ2lAF7j7i62pRD4RkRcDNf5iCu3lNKgPpAIMti8usgYei1fy17LjfRHuk3+QNX1tTJe63DC/3EzvpKyRqlu6rAeOnAH2AJPao460InLxlUkHXyu6qkaKIuKBTlEKHNDvl3Vr02vJeniJalql03KEIxivIBCaOCA1FPvhHwOMkIt4BkQVK95mpIMNfPt1zTy17L2+uvb/y6NT0//BuyWeo4=')))
EXPECTED_SOURCE_CHECKPOINT_SHA256 = 'cbdc205be3842f02ea7eabc024d2417934aac2dd3de47da1058b1ccb2ff4eaa1'
EXPECTED_SOURCE_ARCHIVE_SHA256 = '976670053c88a1f41d416595fc6ffb5be916ce7377cbb169cdfdeedac8a9c77b'
EXPECTED_SOURCE_COVERAGE = {'unique_train_rows_seen': 349535, 'useful_target_tokens': 39586597, 'target_slots_with_padding': 63842112}
save_results()
print('RUNTIME', json.dumps(result['runtime']))


## 2. Загрузить checkpoint №21/75000

Один исходный ZIP21 или PT75000. Проверка SHA выполняется до загрузки данных и построения кеша.


In [ ]:
def load_source_artifact(path):
    path = Path(path)
    archive_sha, member = None, None
    if path.suffix.lower() == '.zip':
        archive_sha = sha(path.read_bytes())
        assert archive_sha == EXPECTED_SOURCE_ARCHIVE_SHA256, 'Нужен исходный ZIP опыта21'
        with zipfile.ZipFile(path) as archive:
            member = 'step-75000.pt'
            raw = archive.read(member)  # CRC is checked while reading the source member.
            manifest = json.loads(archive.read('manifest.json'))
            entry = next(e for e in manifest['files'] if e['name'] == member)
            assert len(raw) == entry['bytes'] and sha(raw) == entry['sha256']
    else:
        assert path.suffix.lower() == '.pt', 'Выбери ZIP21 или step-75000.pt из него'
        raw = path.read_bytes()
    digest = sha(raw)
    assert digest == EXPECTED_SOURCE_CHECKPOINT_SHA256, 'Нужен именно checkpoint21/75000; latest.pt относится к100000'
    state = torch.load(io.BytesIO(raw), map_location='cpu', weights_only=True)
    assert state['steps'] == INITIAL_STEP == 75000
    assert state['config'] == result['historical_constant21']['config']
    assert state['tokenizer_sha256'] == sha(state['tokenizer_json'].encode('utf-8'))
    assert state['tokenizer_sha256'] == result['historical_constant21']['config']['tokenizer_sha256']
    result['source_artifact'] = {'uploaded_name': path.name, 'archive_member': member,
        'sha256': digest, 'source_archive_sha256': archive_sha, 'step': state['steps']}
    result['phase'] = 'source_artifact_loaded'
    save_results()
    print('SOURCE_ARTIFACT', json.dumps(result['source_artifact']), flush=True)
    return state

from google.colab import files
uploaded = files.upload()
assert len(uploaded) == 1, 'Загрузи один исходный ZIP21 или PT75000 из него'
uploaded_name = next(iter(uploaded))
del uploaded
source_checkpoint_state = load_source_artifact(uploaded_name)


## 3. Те же 500k train и heldout1024

Исходный split, порядок строк и hashes сверяются с №21. Fresh1024 только резервируется.


In [ ]:
revision = 'b2724689ff079e9c22efd548fe134bfb29036dbb'
base_url = f'https://huggingface.co/datasets/SlavaYus/tinysms_qa_12m/resolve/{revision}'
cache = Path('hw02-data')
cache.mkdir(exist_ok=True)
data_started = time.perf_counter()
# Pinned public file sizes / LFS SHA256; download only until 500k accepted pairs.
SHARDS = [{'path': 'data/train/part-00000.parquet', 'bytes': 26169796, 'sha256': '91831d582faa12d73b1a6b3878e496446cd5df074f17ee4ffea8cc2887c708ec'}, {'path': 'data/train/part-00001.parquet', 'bytes': 26166175, 'sha256': '5222d657a1af07a531f1b22503f35741a6547f0831509897d0a279bc8f67d068'}, {'path': 'data/train/part-00002.parquet', 'bytes': 26162244, 'sha256': 'c6082401ea966aeaa8375018ae1f1d9c7c55cc30e421c45cc933ceacd97dc3de'}]
EXPECTED_TRAIN500K_SHA256 = '61483f4dc6970ff5f2b1afffa45d57391dcfc01ab6ec4f140c6e6b6c792c8c38'
EXPECTED_LOCATIONS_SHA256 = 'c2332a54a923ffa85da9ac91ec0a39a2e4ead7f7c2e328abecef702e1d306ba7'
EXPECTED_FRESH_HELDOUT_SHA256 = 'eaf5f7c00559d6b1e5650a0dd879e5b4a214931ee803f8c1f6efd956c54971b2'

def download_checked(relative_path, expected_sha256, expected_size=None):
    target = cache / Path(relative_path).name
    if not target.exists():
        temporary = target.with_suffix(target.suffix + '.tmp')
        urlretrieve(f'{base_url}/{relative_path}', temporary)
        assert sha(temporary.read_bytes()) == expected_sha256, f'Checksum mismatch: {relative_path}'
        if expected_size is not None: assert temporary.stat().st_size == expected_size
        temporary.replace(target)
    assert sha(target.read_bytes()) == expected_sha256, f'Cached checksum mismatch: {relative_path}'
    if expected_size is not None: assert target.stat().st_size == expected_size
    return target

def is_validation(question):
    key = hashlib.sha256(f'{SEED}:{question}'.encode()).digest()
    return int.from_bytes(key[:8], 'big') % 10 == 0

def eligible(row):
    return len(row['question'].encode()) <= 512 and len(row['code'].encode()) < 512

test_file = download_checked('data/test/test.parquet',
    '8bde59c8d8bb7edc4e8a503be22fc132c4a47773025d333432e78c5489e6522a')
test_questions = pq.read_table(test_file, columns=['question'])['question'].to_pylist()
assert len(test_questions) == 1919
test_question_set = set(test_questions)
train, train_locations, validation = [], [], None
heldout_pool, heldout_questions = [], set()
fresh_heldout, fresh_questions = [], set()
shard_audits = []
for shard_index, spec in enumerate(SHARDS):
    path = download_checked(spec['path'], spec['sha256'], spec['bytes'])
    parquet = pq.ParquetFile(path)
    audit = {'shard': shard_index, **spec, 'source_rows': parquet.metadata.num_rows,
             'rows_scanned': 0, 'question_over512': 0, 'code_512_or_more': 0,
             'eligible_rows': 0, 'test_copies_excluded': 0, 'hash_heldout_rows_excluded': 0,
             'accepted_train_pairs': 0}
    offset = 0
    for batch in parquet.iter_batches(batch_size=4096):
        rows = batch.to_pylist()
        if validation is None:
            validation = [r for r in rows if eligible(r) and is_validation(r['question'])][:64]
            assert len(validation) == 64
            heldout_questions.update(r['question'] for r in validation)
        for index, row in enumerate(rows):
            audit['rows_scanned'] += 1
            qlong = len(row['question'].encode()) > 512
            clong = len(row['code'].encode()) >= 512
            audit['question_over512'] += int(qlong)
            audit['code_512_or_more'] += int(clong)
            if qlong or clong: continue
            audit['eligible_rows'] += 1
            question = row['question']
            if question in test_question_set:
                audit['test_copies_excluded'] += 1
                continue
            if is_validation(question):
                audit['hash_heldout_rows_excluded'] += 1
                if shard_index == 0 and question not in heldout_questions and len(heldout_pool) < HELDOUT_SIZE - 64:
                    heldout_pool.append(row)
                    heldout_questions.add(question)
                if shard_index > 0 and question not in heldout_questions and question not in fresh_questions and len(fresh_heldout) < FRESH_HELDOUT_SIZE:
                    fresh_heldout.append(row)
                    fresh_questions.add(question)
                continue
            train.append(row)
            train_locations.append((shard_index, offset + index))
            audit['accepted_train_pairs'] += 1
            if len(train) == TRAIN_SIZE: break
        if len(train) == TRAIN_SIZE: break
        offset += len(rows)
    shard_audits.append(audit)
    print('SHARD', json.dumps(audit), flush=True)
    if len(train) == TRAIN_SIZE: break
assert len(train) == TRAIN_SIZE == 500000, 'Pinned shards did not produce500k eligible non-heldout pairs'
assert len(validation) == 64 and len(heldout_pool) == HELDOUT_SIZE - 64
assert len(fresh_heldout) == FRESH_HELDOUT_SIZE
BASE20K_SHA256 = '1304183967f02f01097d76c12fb24df30916a4ec44e54833cd3509264359300e'
BASE100K_SHA256 = '7a141e91ce958206be7e18d3c50b5ccf72a10f1d94890c5f0c3927aa497e47b9'
assert data_hash(train[:20000]) == BASE20K_SHA256
assert data_hash(train[:100000]) == BASE100K_SHA256, 'Saved15 train100k must remain the exact prefix'
assert all(s == 0 for s, _ in train_locations[:100000])
assert data_hash([i for _, i in train_locations[:100000]]) == 'fd49ac534874f1a49777ea2277b8c940c2a3fbea908ec05027bbe1245db34d86'
assert data_hash(train[:64]) == '125106742b04a01ebe70f106730c0bd9a30f52244bef86b4215f27591d200203'
assert data_hash(train[:512]) == '3801627c7dcaae0c82c6694e0aa8514ca8ced044c07c7f30e779bb6a3130bc35'
assert len({r['question'] for r in train[:100000]}) == 99212
train_diagnostic = train[:64]
validation_extra, validation_new = heldout_pool[:192], heldout_pool[192:]
heldout_all = validation + validation_extra + validation_new
assert len(heldout_all) == len(heldout_questions) == HELDOUT_SIZE == 1024
assert len(validation_new) == 768
assert data_hash(validation) == '0edf2b479ebdc103ec05efc704c7bb0ac8084e41a8a44b58afef224702563050'
assert data_hash(validation_extra) == 'a2140e9f3e05f8ea10e4647b3a28335d4c9e240c083c14e8b23cf30a6dbe8bfc'
assert data_hash(validation_new) == 'e8273f55b276a9c5d73e18ff89bc8cbf550dbfec2a25380063635fa39d3fd3bd'
assert data_hash(heldout_all) == 'e187bc297218365f1a1452a15376e0ff2fceaf8a5a5517820549854b90564743'
train_questions = {r['question'] for r in train}
assert not (train_questions & (heldout_questions | fresh_questions | test_question_set))
assert not (fresh_questions & (heldout_questions | test_question_set))
assert all(not is_validation(q) for q in train_questions)
train_hash, validation_hash = data_hash(train), data_hash(validation)
locations_hash = data_hash(train_locations)
fresh_hash = data_hash(fresh_heldout)
if EXPECTED_TRAIN500K_SHA256 is not None: assert train_hash == EXPECTED_TRAIN500K_SHA256
if EXPECTED_LOCATIONS_SHA256 is not None: assert locations_hash == EXPECTED_LOCATIONS_SHA256
if EXPECTED_FRESH_HELDOUT_SHA256 is not None: assert fresh_hash == EXPECTED_FRESH_HELDOUT_SHA256
FRESH_HELDOUT_FILE = RUN_DIR / 'fresh-heldout-reserved.json'
write_json(FRESH_HELDOUT_FILE, fresh_heldout)
result['data'] = {'revision': revision, 'training_examples': len(train), 'shards_used': shard_audits,
    'selection': 'first500k eligible non-hash-heldout, non-test rows across consecutive pinned shards0..; original100k exact prefix; question<=512/code<512 UTF8 bytes; duplicate train questions retained as before',
    'train_sha256': train_hash, 'train_locations_sha256': locations_hash,
    'base20k_sha256': BASE20K_SHA256, 'base100k_sha256': BASE100K_SHA256,
    'base20k_prefix_unchanged': True, 'base100k_prefix_unchanged': True,
    'train64_sha256': data_hash(train_diagnostic), 'unique_train_questions': len(train_questions),
    'duplicate_question_rows': len(train) - len(train_questions),
    'validation_examples': 64, 'validation64_sha256': validation_hash,
    'extra_heldout_examples': 192, 'extra_heldout_sha256': data_hash(validation_extra),
    'new_heldout_examples': 768, 'new_heldout_sha256': data_hash(validation_new),
    'total_heldout_examples': len(heldout_all), 'heldout1024_sha256': data_hash(heldout_all),
    'heldout256_sha256': data_hash(validation + validation_extra),
    'question_overlap_train_heldout': 0, 'question_overlap_train_test': 0,
    'fresh_heldout_reserved': {'examples': len(fresh_heldout), 'pairs_sha256': fresh_hash,
        'file': FRESH_HELDOUT_FILE.name, 'train_question_overlap': 0, 'old_heldout_question_overlap': 0,
        'selection': 'first1024 distinct eligible hash-heldout questions from shard1 onward; no old-heldout/test copies',
        'scope': 'reserved for final comparison of selected models; not evaluated or used to tune this run'},
    'total_pair_presentations_planned': TOTAL_STEPS * BATCH_SIZE,
    'mean_pair_presentations_planned': TOTAL_STEPS * BATCH_SIZE / len(train),
    'download_and_selection_seconds': time.perf_counter() - data_started,
    'intervention_scope': 'same exact train500k/heldout1024/test/fresh as21; replay75001..100000 with lower LR only'}
for key in ['validation64_sha256', 'extra_heldout_sha256', 'new_heldout_sha256', 'heldout1024_sha256', 'heldout256_sha256']:
    assert result['data'][key] == result['historical_bpe4k18']['data'][key], key
    assert result['data'][key] == result['historical_bpe8k19']['data'][key], key
    assert result['data'][key] == result['historical_bpe8k20']['data'][key], key
for key in ['train_sha256', 'train_locations_sha256', 'validation64_sha256', 'extra_heldout_sha256', 'new_heldout_sha256', 'heldout1024_sha256', 'heldout256_sha256']:
    assert result['data'][key] == result['historical_constant21']['data'][key], key
result['phase'] = 'data_ready'
save_results()
print('DATA', json.dumps(result['data']), flush=True)
# Free selection-only objects before building the BPE cache.
del train_questions, train_locations, heldout_pool, heldout_questions, fresh_questions, fresh_heldout


## 4. Точный токенизатор из checkpoint и прежний кеш

BPE не переобучается: загружаются прежние JSON/IDs. Round-trip и совпадение online/cache проверяются. Подготовка исключена из train timer.


In [ ]:
tokenizer_started = time.perf_counter()
# Exact regex from Qwen2Tokenizer; keep our byte-preserving normalizer=None.
QWEN_PATTERN = r"(?i:'s|'t|'re|'ve|'m|'ll|'d)|[^\r\n\p{L}\p{N}]?\p{L}+|\p{N}| ?[^\s\p{L}\p{N}]+[\r\n]*|\s*[\r\n]+|\s+(?!\S)|\s+"
# Restore exact bytes/IDs from the source checkpoint; never refit BPE.
TOKENIZER_FILE.write_text(source_checkpoint_state['tokenizer_json'], encoding='utf-8')
tokenizer_hash = sha(TOKENIZER_FILE.read_bytes())
assert tokenizer_hash == source_checkpoint_state['tokenizer_sha256'] == result['historical_constant21']['config']['tokenizer_sha256']
tokenizer = Tokenizer.from_file(str(TOKENIZER_FILE))
VOCAB = tokenizer.get_vocab_size() + TOKEN_OFFSET
assert VOCAB == TARGET_VOCAB
tokenizer_training_seconds = 0.0
print('TOKENIZER_RESTORED', {'content_vocab': VOCAB - TOKEN_OFFSET, 'sha256': tokenizer_hash}, flush=True)
visible = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
byte_chars = {b: chr(b) for b in visible}
for index, b in enumerate(b for b in range(256) if b not in byte_chars):
    byte_chars[b] = chr(256 + index)
char_bytes = {char: b for b, char in byte_chars.items()}
assert set(char_bytes) == set(pre_tokenizers.ByteLevel.alphabet())
TOKEN_BYTES = [b'', b'', b''] + [bytes(char_bytes[c] for c in tokenizer.id_to_token(i))
                                for i in range(VOCAB - TOKEN_OFFSET)]
assert all(TOKEN_BYTES[i] for i in range(TOKEN_OFFSET, VOCAB))

def encode(text): return [i + TOKEN_OFFSET for i in tokenizer.encode(text, add_special_tokens=False).ids]
def decode_bytes(ids):
    parts = []
    for i in ids:
        if i == EOS: break
        if i >= TOKEN_OFFSET: parts.append(TOKEN_BYTES[i])
    return b''.join(parts)
def decode(ids): return decode_bytes(ids).decode('utf-8', errors='replace')
def length_stats(values):
    values = sorted(values)
    return {'count': len(values), 'mean': sum(values) / len(values), 'min': values[0],
        'p50': values[int(0.5 * (len(values) - 1))], 'p95': values[int(0.95 * (len(values) - 1))],
        'max': values[-1]}
def audit_tokenization(rows, encoded_pairs=None):
    report = {}
    for field in ('question', 'code'):
        byte_lengths, bpe_lengths = [], []
        for index, row in enumerate(rows):
            text = row[field]
            ids = encode(text) if encoded_pairs is None else (encoded_pairs[index][0][1:-1] if field == 'question' else encoded_pairs[index][1])
            assert decode_bytes(ids) == text.encode('utf-8'), 'Round-trip failed'
            assert tokenizer.decode([i - TOKEN_OFFSET for i in ids], skip_special_tokens=False) == text
            assert all(TOKEN_OFFSET <= i < VOCAB for i in ids)
            assert len(ids) + 2 <= MAX_POSITIONS
            byte_lengths.append(len(text.encode('utf-8')))
            bpe_lengths.append(len(ids))
        report[field] = {'utf8_bytes': length_stats(byte_lengths), 'bpe_tokens': length_stats(bpe_lengths),
            'total_bytes_per_bpe_token': sum(byte_lengths) / sum(bpe_lengths)}
    return report
probes = ['', '    return 1\n', '\n\n\t x\r\n', '  trailing  ',
          'Привет 世界 🧠', 'x = 0.005 + 123456789\n', '<pad><bos><eos>',
          ''.join(chr(i) for i in range(256))]
probes += ['500', ' 500', '1500', '12345', '12346', '-12.50', '30%', '1e-3', 'e\u0301 👋\r\n']
for text in probes: assert decode_bytes(encode(text)) == text.encode('utf-8')
digit_probes = []
for text in ['0', '500', ' 500', '1500', '12345', '12346', '-12.50', '30%', '1e-3']:
    content = tokenizer.encode(text, add_special_tokens=False)
    ascii_digits = [c for c in text if c in '0123456789']
    observed = [piece for piece in content.tokens if piece in '0123456789' and len(piece) == 1]
    assert observed == ascii_digits
    assert all(not any(c in '0123456789' for c in p) or (len(p) == 1 and p in '0123456789')
               for p in content.tokens), 'Digit merged with surrounding text'
    digit_probes.append({'text': text, 'pieces': content.tokens, 'content_ids': content.ids})
assert encode('500') == encode(' 500')[-3:]
reloaded_tokenizer = Tokenizer.from_file(str(TOKENIZER_FILE))
assert all(reloaded_tokenizer.encode(t).ids == tokenizer.encode(t).ids for t in probes)
cache_started = time.perf_counter()
train_cache = [([BOS] + encode(row['question']) + [EOS], encode(row['code'])) for row in train]
cache_seconds = time.perf_counter() - cache_started
assert len(train_cache) == len(train)

result['tokenization'] = {'kind': 'own byte-level BPE with Qwen-style pretokenization', 'loaded_from_experiment': '21 checkpoint75000', 'tokenizer_training_performed': False, 'target_model_vocabulary': TARGET_VOCAB,
    'actual_model_vocabulary': VOCAB, 'content_id_offset': TOKEN_OFFSET, 'special_ids': {'pad': PAD, 'bos': BOS, 'eos': EOS},
    'normalizer': None, 'add_prefix_space': False, 'use_regex': False, 'qwen_regex': QWEN_PATTERN, 'min_frequency': 2, 'tokenizer_training_seconds': tokenizer_training_seconds,
    'training_corpus': 'original19/21 fitting on all500000 allowed train questions and codes; loaded unchanged, not refit', 'training_pairs': TOKENIZER_TRAIN_SIZE, 'training_texts': 2 * TOKENIZER_TRAIN_SIZE,
    'training_corpus_sha256': train_hash, 'training_corpus_utf8_bytes': sum(len(r['question'].encode('utf-8')) + len(r['code'].encode('utf-8')) for r in train[:TOKENIZER_TRAIN_SIZE]), 'tokenizer_sha256': tokenizer_hash,
    'train_round_trip': audit_tokenization(train, train_cache), 'validation64_round_trip': audit_tokenization(validation),
    'extra192_round_trip': audit_tokenization(validation_extra),
    'new768_round_trip': audit_tokenization(validation_new), 'probe_round_trip_count': len(probes),
    'loading_and_audit_seconds': time.perf_counter() - tokenizer_started}
result['tokenization']['cache'] = {'pairs': len(train_cache), 'ids': sum(len(q) + len(c) for q, c in train_cache),
    'build_seconds_excluding_round_trip_audit': cache_seconds, 'dtype': 'Python integer IDs, CPU tensors assembled per step as16/B',
    'scope': 'cache setup and audits excluded from train timer; both counted in setup wall time'}
result['tokenization']['digit_probes'] = digit_probes
result['tokenization']['saved_bpe4k18_train_lengths'] = result['historical_bpe4k18']['tokenization']['train_round_trip']
result['phase'] = 'tokenizer_ready'
save_results()
print('TOKENIZATION', json.dumps(result['tokenization']))
for text in ['def solve():\n    return 1\n', '    ', 'return', '12345']:
    print('TOKENS', repr(text), [(i, repr(TOKEN_BYTES[i])) for i in encode(text)])

def source_batch(questions):
    return pad_sequence([torch.tensor([BOS] + encode(q) + [EOS], dtype=torch.long) for q in questions],
                        batch_first=True, padding_value=PAD).to(device)
def training_batch(examples):
    src = source_batch([r['question'] for r in examples])
    codes = [encode(r['code']) for r in examples]
    tgt = pad_sequence([torch.tensor([BOS] + c, dtype=torch.long) for c in codes],
                       batch_first=True, padding_value=PAD).to(device)
    labels = pad_sequence([torch.tensor(c + [EOS], dtype=torch.long) for c in codes],
                          batch_first=True, padding_value=PAD).to(device)
    return src, tgt, labels

def cached_training_batch(indices):
    src = pad_sequence([torch.tensor(train_cache[i][0], dtype=torch.long) for i in indices],
                       batch_first=True, padding_value=PAD).to(device)
    codes = [train_cache[i][1] for i in indices]
    tgt = pad_sequence([torch.tensor([BOS] + c, dtype=torch.long) for c in codes],
                       batch_first=True, padding_value=PAD).to(device)
    labels = pad_sequence([torch.tensor(c + [EOS], dtype=torch.long) for c in codes],
                          batch_first=True, padding_value=PAD).to(device)
    return src, tgt, labels

parity_rng = random.Random(SEED)
parity_indices = [parity_rng.sample(range(len(train)), BATCH_SIZE) for _ in range(16)]
parity_indices.append([0, len(train) - 1,
    max(range(len(train_cache)), key=lambda i: len(train_cache[i][0])),
    max(range(len(train_cache)), key=lambda i: len(train_cache[i][1]))])
for indices in parity_indices:
    online = training_batch([train[i] for i in indices])
    cached = cached_training_batch(indices)
    assert all(torch.equal(a, b) for a, b in zip(online, cached)), 'Cache changed inputs/labels'
    assert torch.equal(cached[1].ne(PAD), cached[2].ne(PAD))
result['tokenization']['online_cached_batch_parity_count'] = len(parity_indices)
result['setup_seconds_through_data_and_tokenizer'] = result['data']['download_and_selection_seconds'] + time.perf_counter() - tokenizer_started
save_results()
print('CACHE', json.dumps(result['tokenization']['cache']), 'PARITY', len(parity_indices), flush=True)
del online, cached


## 5. Прежняя модель и cosine на 75001…100000

Первое обновление использует 0.0001, последнее — 0.00001. Конструктор прежний; его веса будут заменены checkpoint. Счётчик LambdaLR остаётся глобальным.


In [ ]:
class SwiGLUInput(nn.Module):
    """Two input-dependent branches: SiLU(gate) * value, before FFN dropout."""
    def __init__(self, width, hidden):
        super().__init__()
        self.gate = nn.Linear(width, hidden)
        self.value = nn.Linear(width, hidden)
    def forward(self, x):
        return F.silu(self.gate(x)) * self.value(x)

def replace_ffn_with_swiglu(transformer):
    # Initialize from scratch; encoder depth changes the model RNG consumption.
    # Isolated CPU RNG initializes only the new FFNs; training RNG is restored on exit.
    with torch.random.fork_rng(devices=[]):
        torch.set_rng_state(torch.Generator().manual_seed(SEED).get_state())
        for layer in list(transformer.encoder.layers) + list(transformer.decoder.layers):
            layer.linear1 = SwiGLUInput(D_MODEL, SWIGLU_DIM)
            layer.linear2 = nn.Linear(SWIGLU_DIM, D_MODEL)
            layer.activation = nn.Identity()  # SiLU and multiplication already happen in linear1.
            if isinstance(layer, nn.TransformerEncoderLayer):
                # Fused encoder supports only ReLU/GELU and would bypass the gated module.
                layer.activation_relu_or_gelu = 0
            for module in [layer.linear1.gate, layer.linear1.value, layer.linear2]:
                nn.init.xavier_uniform_(module.weight)  # same matrix initializer as nn.Transformer

class SimpleTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.tokens = nn.Embedding(VOCAB, D_MODEL, padding_idx=PAD)
        self.positions = nn.Embedding(MAX_POSITIONS, D_MODEL)
        self.transformer = nn.Transformer(d_model=D_MODEL, nhead=4,
            num_encoder_layers=ENCODER_LAYERS, num_decoder_layers=DECODER_LAYERS,
            dim_feedforward=4 * D_MODEL, dropout=0.1, batch_first=True, norm_first=True)
        self.transformer.encoder.enable_nested_tensor = False
        self.transformer.encoder.use_nested_tensor = False
        self.head = nn.Linear(D_MODEL, VOCAB)
        replace_ffn_with_swiglu(self.transformer)
    def embed(self, ids):
        if ids.shape[1] > MAX_POSITIONS:
            raise ValueError('Sequence too long; no silent truncation')
        pos = torch.arange(ids.shape[1], device=ids.device)
        return self.tokens(ids) * math.sqrt(D_MODEL) + self.positions(pos)
    def decode_prefix(self, tgt, memory, source_padding):
        mask = torch.ones(tgt.shape[1], tgt.shape[1], device=tgt.device, dtype=torch.bool).triu(1)
        return self.transformer.decoder(self.embed(tgt), memory, tgt_mask=mask,
            tgt_key_padding_mask=tgt.eq(PAD), memory_key_padding_mask=source_padding)
    def forward(self, src, tgt):
        memory = self.transformer.encoder(self.embed(src), src_key_padding_mask=src.eq(PAD))
        return self.head(self.decode_prefix(tgt, memory, src.eq(PAD)))
    @torch.inference_mode()
    def generate(self, src, max_new_tokens=512):
        self.eval()
        memory = self.transformer.encoder(self.embed(src), src_key_padding_mask=src.eq(PAD))
        tgt = torch.full((len(src), 1), BOS, device=src.device, dtype=torch.long)
        done = torch.zeros(len(src), device=src.device, dtype=torch.bool)
        for _ in range(max_new_tokens):
            hidden = self.decode_prefix(tgt, memory, src.eq(PAD))
            logits = self.head(hidden[:, -1])
            logits[:, [PAD, BOS]] = -torch.inf
            token = logits.argmax(-1).masked_fill(done, PAD)
            done |= token.eq(EOS)
            tgt = torch.cat([tgt, token[:, None]], dim=1)
            if bool(done.all()):
                break
        return tgt[:, 1:]

model = SimpleTransformer().to(device)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
assert WARMUP_STEPS == 0 and INITIAL_STEP < TOTAL_STEPS
def learning_rate_for_update(update, start_step=INITIAL_STEP + 1, end_step=SCHEDULE_TOTAL_STEPS,
                             start_lr=CONTINUATION_START_LR, min_lr=CONTINUATION_MIN_LR):
    # First replayed update75001 uses1e-4; final100000 uses1e-5.
    # LambdaLR still counts GLOBAL completed updates; its counter is restored, not reset.
    assert end_step > start_step and 0 < min_lr <= start_lr
    progress = min(max((int(update) - start_step) / (end_step - start_step), 0.0), 1.0)
    return min_lr + 0.5 * (start_lr - min_lr) * (1 + math.cos(math.pi * progress))

def learning_rate_multiplier(completed_updates):
    return learning_rate_for_update(completed_updates + 1) / LEARNING_RATE

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=learning_rate_multiplier)
assert scheduler.last_epoch == 0
assert math.isclose(optimizer.param_groups[0]['lr'], CONTINUATION_START_LR)
TOKEN_BYTE_LENGTHS = torch.tensor([len(b) for b in TOKEN_BYTES], device=device)
config = {'seed': SEED, 'train_size': len(train), 'batch_size_train': BATCH_SIZE, 'total_steps': TOTAL_STEPS,
    'evaluation_steps': list(EVALUATION_STEPS), 'full_evaluation_steps': list(FULL_EVALUATION_STEPS), 'd_model': D_MODEL, 'encoder_layers': ENCODER_LAYERS,
    'decoder_layers': DECODER_LAYERS, 'heads': 4, 'ffn_dim': SWIGLU_DIM, 'relu_reference_ffn_dim': 4 * D_MODEL, 'dropout': 0.1,
    'activation': 'SwiGLU (SiLU gate * linear value)', 'ffn_bias': True, 'norm_first': True, 'lr': CONTINUATION_START_LR, 'optimizer': 'AdamW',
    'weight_decay': optimizer.param_groups[0]['weight_decay'], 'clip_norm': 1.0, 'scheduler': {'name': 'LambdaLR', 'schedule': 'cosine_on_replayed_tail',
        'total_steps': SCHEDULE_TOTAL_STEPS, 'warmup_steps': 0,
        'first_update': INITIAL_STEP + 1, 'last_update': TOTAL_STEPS,
        'additional_updates': TOTAL_STEPS - INITIAL_STEP, 'start_lr': CONTINUATION_START_LR,
        'eta_min': CONTINUATION_MIN_LR, 'optimizer_base_lr': LEARNING_RATE,
        'restarts': False, 'step_unit': 'optimizer update',
        'indexing': 'global completed-update counter restored from21; first75001=start; last100000=floor; clamped afterwards'},
    'dtype': 'float32', 'tokenizer': 'own Qwen-style byte-level BPE', 'vocab': VOCAB, 'content_id_offset': TOKEN_OFFSET,
    'special_ids': {'pad': PAD, 'bos': BOS, 'eos': EOS}, 'tokenizer_sha256': tokenizer_hash,
    'max_positions': MAX_POSITIONS, 'generation_cap': GENERATION_CAP,
    'batch_size_inference': INFERENCE_BATCH, 'inference': 'Pre-LN incremental KV cache'}
config['tokenizer_source_experiment'] = '19'
config['training_from_scratch'] = False
config['trained_from_scratch_chain'] = True
config['initial_step'] = INITIAL_STEP
config['additional_steps_planned'] = TOTAL_STEPS - INITIAL_STEP
config['ffn_initialization'] = 'All model weights and AdamW restored from21/75000; same4+4/SwiGLU; constructor initialization is overwritten before training'
config['scheduler']['lr_after_original_horizon'] = CONTINUATION_MIN_LR
result['config'] = config
result['parameters'] = sum(p.numel() for p in model.parameters())
assert result['parameters'] == 12086912
assert len(model.transformer.encoder.layers) == ENCODER_LAYERS == 4
assert len(model.transformer.decoder.layers) == DECODER_LAYERS == 4
result['phase'] = 'model_ready'
save_results()
print('PARAMETERS', result['parameters'])
assert tokenizer_hash != result['historical_bpe4k18']['config']['tokenizer_sha256']
config['parameter_change_vs18'] = result['parameters'] - result['historical_bpe4k18']['parameters']
config['parameter_change_vs19'] = result['parameters'] - result['historical_bpe8k19']['parameters']
config['parameter_change_vs20'] = result['parameters'] - result['historical_bpe8k20']['parameters']
assert config['parameter_change_vs20'] == 1576096
for key in ['seed', 'train_size', 'batch_size_train', 'total_steps', 'd_model', 'encoder_layers', 'decoder_layers',
            'heads', 'ffn_dim', 'relu_reference_ffn_dim', 'activation', 'ffn_bias', 'dropout', 'norm_first',
            'optimizer', 'weight_decay', 'clip_norm', 'dtype', 'vocab', 'tokenizer_sha256', 'max_positions',
            'generation_cap', 'batch_size_inference', 'special_ids', 'data_preparation']:
    if key in config:
        assert config[key] == result['historical_constant21']['config'][key], key
assert all(layer.norm_first for layer in list(model.transformer.encoder.layers) + list(model.transformer.decoder.layers))
assert all(layer.activation_relu_or_gelu == 0 for layer in model.transformer.encoder.layers)
assert model.transformer.encoder.norm is not None and model.transformer.decoder.norm is not None

config['experiment_factor'] = 'replay21/75000..100000 with cosineLR1e-4 to1e-5; same source weights/AdamW/RNG/batches/train500k/model/tokenizer/FP32/cache; existing constant1e-4 tail is the equal-budget control'
config['data_preparation'] = 'cached BPE IDs as16/B; FP32 weights/AdamW; no AMP'
config['training_coverage_tracking'] = 'per-row CPU byte flags and useful target token count, included in train timer'
seen_train_rows = bytearray(len(train))
unique_train_rows_seen = 0
useful_target_tokens_seen = 0
padded_target_slots_seen = 0
result['training_coverage'] = {'unique_train_rows_seen': 0, 'useful_target_tokens': 0, 'target_slots_with_padding': 0}
save_results()


In [ ]:
class CachedDecoder:
    @torch.inference_mode()
    def __init__(self, model, src, capacity):
        model.eval()
        if not 0 < capacity <= model.positions.num_embeddings:
            raise ValueError('Invalid cache capacity')
        self.model, self.capacity, self.position = model, capacity, 0
        self.batch_size = len(src)
        self.width = model.tokens.embedding_dim
        self.heads = model.transformer.decoder.layers[0].self_attn.num_heads
        self.head_dim = self.width // self.heads
        memory = model.transformer.encoder(model.embed(src), src_key_padding_mask=src.eq(PAD))
        # SDPA bool masks use True for allowed keys (opposite to MHA padding masks).
        self.source_allowed = src.ne(PAD)[:, None, None, :]
        self.valid = torch.zeros(len(src), capacity, dtype=torch.bool, device=src.device)
        self.position_vectors = model.positions(torch.arange(capacity, device=src.device))
        self.layers = []
        for layer in model.transformer.decoder.layers:
            if not layer.norm_first:
                raise ValueError('This experiment requires Pre-LN decoder layers')
            for attention in (layer.self_attn, layer.multihead_attn):
                if (not attention._qkv_same_embed_dim or attention.num_heads != self.heads
                        or attention.bias_k is not None or attention.bias_v is not None
                        or attention.add_zero_attn):
                    raise ValueError('Unsupported attention configuration')
            attention = layer.multihead_attn
            projected_kv = F.linear(memory, attention.in_proj_weight[self.width:],
                                    attention.in_proj_bias[self.width:])
            cross_k, cross_v = projected_kv.chunk(2, dim=-1)
            shape = (len(src), self.heads, capacity, self.head_dim)
            self.layers.append({
                'self_k': memory.new_empty(shape), 'self_v': memory.new_empty(shape),
                'cross_k': self.split_heads(cross_k), 'cross_v': self.split_heads(cross_v),
            })

    def split_heads(self, values):
        return values.reshape(self.batch_size, -1, self.heads, self.head_dim).transpose(1, 2)

    def attend(self, attention, q, k, v, allowed):
        # One query at the latest position; all supplied keys are from its past/current.
        # is_causal=True would use an upper-left 1 x T mask and incorrectly hide the past.
        output = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed,
                                                dropout_p=0.0, is_causal=False)
        output = output.transpose(1, 2).contiguous().reshape(self.batch_size, 1, self.width)
        return attention.out_proj(output)

    @torch.inference_mode()
    def step(self, token_ids):
        if self.position >= self.capacity:
            raise ValueError('Cache capacity exceeded')
        if token_ids.ndim != 1 or len(token_ids) != self.batch_size:
            raise ValueError('One token per batch row is required')
        t = self.position
        x = self.model.tokens(token_ids[:, None]) * math.sqrt(self.width)
        x = x + self.position_vectors[t][None, None, :]
        self.valid[:, t] = token_ids.ne(PAD)
        self_allowed = self.valid[:, None, None, :t + 1]
        for layer, cache in zip(self.model.transformer.decoder.layers, self.layers):
            attention = layer.self_attn
            q, k, v = F.linear(layer.norm1(x), attention.in_proj_weight, attention.in_proj_bias).chunk(3, dim=-1)
            cache['self_k'][:, :, t:t + 1] = self.split_heads(k)
            cache['self_v'][:, :, t:t + 1] = self.split_heads(v)
            self_output = self.attend(attention, self.split_heads(q),
                cache['self_k'][:, :, :t + 1], cache['self_v'][:, :, :t + 1], self_allowed)
            x = x + layer.dropout1(self_output)

            attention = layer.multihead_attn
            q = F.linear(layer.norm2(x), attention.in_proj_weight[:self.width], attention.in_proj_bias[:self.width])
            cross_output = self.attend(attention, self.split_heads(q),
                                      cache['cross_k'], cache['cross_v'], self.source_allowed)
            x = x + layer.dropout2(cross_output)
            feed_forward = layer.linear2(layer.dropout(layer.activation(layer.linear1(layer.norm3(x)))))
            x = x + layer.dropout3(feed_forward)

        if self.model.transformer.decoder.norm is not None:
            x = self.model.transformer.decoder.norm(x)
        self.position += 1
        return self.model.head(x[:, 0])

@torch.inference_mode()
def generate_cached(model, src, max_new_tokens=512, max_output_bytes=None):
    model.eval()
    cache = CachedDecoder(model, src, max_new_tokens)
    tgt = torch.full((len(src), 1), BOS, device=src.device, dtype=torch.long)
    done = torch.zeros(len(src), device=src.device, dtype=torch.bool)
    byte_stopped = done.clone()
    output_bytes = torch.zeros(len(src), device=src.device, dtype=torch.long)
    for _ in range(max_new_tokens):
        logits = cache.step(tgt[:, -1])
        logits[:, [PAD, BOS]] = -torch.inf
        token = logits.argmax(-1).masked_fill(done, PAD)
        if max_output_bytes is not None:
            overflow = (~done) & (token >= TOKEN_OFFSET) & (output_bytes + TOKEN_BYTE_LENGTHS[token] > max_output_bytes)
            # Stop BEFORE the whole overflowing BPE token; do not force a different token/EOS.
            byte_stopped |= overflow
            token = token.masked_fill(overflow, PAD)
            done |= overflow
        output_bytes += TOKEN_BYTE_LENGTHS[token]
        done |= token.eq(EOS)
        tgt = torch.cat([tgt, token[:, None]], dim=1)
        if bool(done.all()): break
    return tgt[:, 1:], byte_stopped

@torch.inference_mode()
def check_cached(rows, prefix_limit=64, greedy_cap=16):
    model.eval()
    src = source_batch([r['question'] for r in rows])
    tgt = pad_sequence([torch.tensor([BOS] + encode(r['code']) + [EOS]) for r in rows],
                       batch_first=True, padding_value=PAD).to(device)[:, :prefix_limit]
    memory = model.transformer.encoder(model.embed(src), src_key_padding_mask=src.eq(PAD))
    cache = CachedDecoder(model, src, tgt.shape[1])
    positions = {0, 1, 2, 7, 15, 31, tgt.shape[1] - 1}
    checks = []
    for t in range(tgt.shape[1]):
        actual = cache.step(tgt[:, t])
        if t in positions:
            expected = model.head(model.decode_prefix(tgt[:, :t+1], memory, src.eq(PAD))[:, -1])
            checks.append({'position': t, 'max_abs_error': (actual - expected).abs().max().item(),
                'all_close': bool(torch.isclose(actual, expected, atol=LOGIT_ATOL, rtol=LOGIT_RTOL).all())})
    ordinary = model.generate(src, max_new_tokens=greedy_cap)
    cached, _ = generate_cached(model, src, max_new_tokens=greedy_cap)
    return {'positions': checks, 'atol': LOGIT_ATOL, 'rtol': LOGIT_RTOL,
        'all_close': all(c['all_close'] for c in checks),
        'greedy_equal': ordinary.shape == cached.shape and bool(torch.equal(ordinary, cached)),
        'greedy_cap': greedy_cap}


## 6. Прежние диагностики и дополнительный ручной аудит

Reference888 остаётся прежним. Ручной74 оценивается исходным способом и с ограниченной поддержкой `+=/-=`. Сохранённые №21/100k — основной контроль.


In [ ]:
class Unsupported(Exception):
    pass

class Undefined(Exception):
    pass

def arithmetic(code):
    try:
        tree = ast.parse(code)
        if len(tree.body) != 1 or not isinstance(tree.body[0], ast.FunctionDef):
            raise Unsupported()
        f = tree.body[0]
        if f.name != 'solve' or f.decorator_list or f.args.args or f.args.posonlyargs or f.args.kwonlyargs or f.args.vararg or f.args.kwarg:
            raise Unsupported()
        if not all((isinstance(s, (ast.Assign, ast.Return)) for s in f.body)):
            raise Unsupported()
        if any((isinstance(n, (ast.Call, ast.Attribute, ast.Subscript, ast.NamedExpr)) for n in ast.walk(f))):
            raise Unsupported()
        env = {}
        visits = 0

        def number(v):
            if type(v) not in (int, float) or not math.isfinite(v) or abs(v) > 1000000000000000.0:
                raise Unsupported()
            return v

        def expr(n):
            nonlocal visits
            visits += 1
            if visits > 2000:
                raise Unsupported()
            if isinstance(n, ast.Constant):
                return number(n.value)
            if isinstance(n, ast.Name):
                if n.id not in env:
                    raise Undefined(n.id)
                return env[n.id]
            if isinstance(n, ast.UnaryOp):
                v = expr(n.operand)
                if isinstance(n.op, ast.USub):
                    return number(-v)
                if isinstance(n.op, ast.UAdd):
                    return v
            if isinstance(n, ast.BinOp):
                a, b = (expr(n.left), expr(n.right))
                if isinstance(n.op, ast.Add):
                    return number(a + b)
                if isinstance(n.op, ast.Sub):
                    return number(a - b)
                if isinstance(n.op, ast.Mult):
                    return number(a * b)
                if isinstance(n.op, ast.Div):
                    return number(a / b)
                if isinstance(n.op, ast.FloorDiv):
                    return number(a // b)
                if isinstance(n.op, ast.Mod):
                    return number(a % b)
                if isinstance(n.op, ast.Pow) and abs(b) <= 12:
                    return number(a ** b)
            raise Unsupported()
        for s in f.body:
            if isinstance(s, ast.Assign):
                if not all((isinstance(n, ast.Name) for n in s.targets)):
                    raise Unsupported()
                v = expr(s.value)
                for n in s.targets:
                    env[n.id] = v
            else:
                return {'status': 'numeric', 'value': expr(s.value)}
        return {'status': 'no_return'}
    except Undefined as e:
        return {'status': 'undefined_name', 'name': str(e)}
    except (SyntaxError, ValueError, RecursionError):
        return {'status': 'syntax_error'}
    except (ZeroDivisionError, OverflowError):
        return {'status': 'arithmetic_error'}
    except Unsupported:
        return {'status': 'unsupported'}

def arithmetic_augassign(code):
    if len(code) > 100000:
        return {'status': 'unsupported'}
    try:
        tree = ast.parse(code)
        if sum((1 for _ in ast.walk(tree))) > 3000:
            return {'status': 'unsupported'}
        for node in ast.walk(tree):
            if isinstance(node, ast.AugAssign) and (not isinstance(node.target, ast.Name)):
                return {'status': 'unsupported'}

        class Rewrite(ast.NodeTransformer):

            def visit_AugAssign(self, node):
                return ast.Assign(targets=[ast.Name(id=node.target.id, ctx=ast.Store())], value=ast.BinOp(left=ast.Name(id=node.target.id, ctx=ast.Load()), op=node.op, right=node.value))
        rewritten = ast.fix_missing_locations(Rewrite().visit(tree))
        return arithmetic(ast.unparse(rewritten))
    except (SyntaxError, ValueError, RecursionError):
        return {'status': 'syntax_error'}

def repetition_properties(code, ids):
    # Heuristics only: repeated token-tail and identical consecutive nonempty lines.
    ids = [i for i in ids if i >= TOKEN_OFFSET]
    tail = 0
    for period in range(1, min(16, len(ids) // 4) + 1):
        motif = ids[-period:]
        copies = 1
        while len(ids) >= (copies + 1) * period and ids[-(copies + 1) * period:-copies * period] == motif:
            copies += 1
        if copies >= 4: tail = max(tail, copies * period)
    longest, run, previous = 0, 0, None
    for line in (line.strip() for line in code.splitlines() if line.strip()):
        run = run + 1 if line == previous else 1
        previous, longest = line, max(longest, run)
    try:
        identifiers = [a.id for a in ast.walk(ast.parse(code)) if isinstance(a, ast.Name)]
        identifier_chars = max(map(len, identifiers), default=0)
    except (SyntaxError, ValueError, RecursionError): identifier_chars = None
    return {'repeated_tail_tokens': tail, 'repeated_tail_32_plus': tail >= 32,
        'max_consecutive_identical_lines': longest, 'consecutive_lines_3_plus': longest >= 3,
        'max_identifier_characters': identifier_chars}

def extra_properties(example):
    numeric = arithmetic(example['generated_code'])
    extra = {'numeric_ast': numeric, **repetition_properties(example['generated_code'], example['generated_token_ids'])}
    if 'reference_code' in example:
        reference = arithmetic(example['reference_code'])
        comparable = numeric['status'] == reference['status'] == 'numeric'
        extra.update(reference_numeric_ast=reference, numeric_ast_comparable=comparable,
            numeric_ast_match=comparable and math.isclose(numeric['value'], reference['value'], rel_tol=1e-7, abs_tol=1e-7))
    return extra

@torch.inference_mode()
def evaluate_loss(rows, source_questions=None):
    model.eval()
    if source_questions is not None: assert len(source_questions) == len(rows)
    loss_sum, label_count = 0.0, 0
    for offset in range(0, len(rows), 16):
        src, tgt, labels = training_batch(rows[offset:offset + 16])
        if source_questions is not None: src = source_batch(source_questions[offset:offset + 16])
        logits = model(src, tgt)
        loss_sum += F.cross_entropy(logits.reshape(-1, VOCAB), labels.reshape(-1),
            ignore_index=PAD, reduction='sum').item()
        label_count += labels.ne(PAD).sum().item()
    return loss_sum / label_count

@torch.inference_mode()
def check_question_conditioning(rows, correct_loss=None):
    assert len(rows) > 1
    questions = [row['question'] for row in rows]
    rotated = questions[1:] + questions[:1]
    correct = evaluate_loss(rows) if correct_loss is None else correct_loss
    shuffled = evaluate_loss(rows, source_questions=rotated)
    return {'examples': len(rows), 'correct_bpe_loss': correct, 'rotated_questions_bpe_loss': shuffled,
            'rotated_minus_correct': shuffled - correct,
            'scope': 'fixed rotation by1; only source questions change, gold codes/masks/weights unchanged; eval mode; sensitivity, not pass@1'}


def code_properties(code):
    def scoped_nodes(node):
        stack = [node]
        while stack:
            current = stack.pop()
            yield current
            if not isinstance(current, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.Lambda)):
                stack.extend(ast.iter_child_nodes(current))
    try:
        tree = ast.parse(code)
        functions = [n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == 'solve']
        has_return = any(isinstance(n, ast.Return) and n.value is not None
            for f in functions for stmt in f.body for n in scoped_nodes(stmt))
        return {'parses_as_python': True, 'defines_solve': bool(functions), 'solve_returns_value': has_return,
                'syntax_error_type': None, 'syntax_error_message': None, 'indentation_error': False}
    except (SyntaxError, ValueError, RecursionError) as error:
        return {'parses_as_python': False, 'defines_solve': False, 'solve_returns_value': False,
                'syntax_error_type': type(error).__name__, 'syntax_error_message': str(error),
                'indentation_error': isinstance(error, IndentationError)}
def inspect_ids(row, ids, byte_stopped=False):
    active = [i for i in ids if i != PAD]
    if EOS in active: active = active[:active.index(EOS) + 1]
    raw = decode_bytes(active)
    try:
        code = raw.decode('utf-8')
        valid_utf8 = True
    except UnicodeDecodeError:
        code, valid_utf8 = raw.decode('utf-8', errors='replace'), False
    eos = EOS in active
    example = {'question': row['question'], 'generated_code': code, 'generated_token_ids': active,
        'generated_tokens_including_eos': len(active), 'generated_utf8_bytes': len(raw), 'valid_utf8': valid_utf8,
        'eos_emitted': eos, 'reached_token_cap': len(active) == GENERATION_CAP,
        'token_cap_without_eos': len(active) == GENERATION_CAP and not eos,
        'byte_budget_stop': bool(byte_stopped),
        'stop_reason': 'eos' if eos else ('byte_budget' if byte_stopped else 'token_cap'),
        **code_properties(code)}
    if 'code' in row:
        example.update({'reference_code': row['code'], 'reference_utf8_bytes': len(row['code'].encode()),
            'reference_bpe_tokens_including_eos': len(encode(row['code'])) + 1,
            'exact_code_match': valid_utf8 and code == row['code'],
            'exact_tokens_with_eos': active == encode(row['code']) + [EOS]})
    return example

@torch.inference_mode()
def inspect_split(name, rows, max_output_bytes=None, with_loss=True):
    model.eval()
    fixed_loss = evaluate_loss(rows) if with_loss else None
    # Warm-up is outside generation timing and never changes weights.
    generate_cached(model, source_batch([rows[0]['question']]), max_new_tokens=8)
    sync()
    if device.type == 'cuda': torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    examples = []
    for offset in range(0, len(rows), INFERENCE_BATCH):
        batch = rows[offset:offset + INFERENCE_BATCH]
        tokens, stopped = generate_cached(model, source_batch([r['question'] for r in batch]),
            max_new_tokens=GENERATION_CAP, max_output_bytes=max_output_bytes)
        examples.extend(inspect_ids(row, ids, stop) for row, ids, stop
                        in zip(batch, tokens.cpu().tolist(), stopped.cpu().tolist()))
    sync()
    seconds = time.perf_counter() - started
    extra_started = time.perf_counter()
    for example in examples: example.update(extra_properties(example))
    codes = Counter(e['generated_code'] for e in examples)
    extra_seconds = time.perf_counter() - extra_started
    summary = {'split': name, 'examples': len(rows), 'bpe_loss': fixed_loss, 'seconds': seconds,
        'answers_per_second': len(rows) / seconds,
        'generated_tokens_including_eos': sum(e['generated_tokens_including_eos'] for e in examples),
        'generated_utf8_bytes': sum(e['generated_utf8_bytes'] for e in examples),
        'mean_tokens_including_eos': sum(e['generated_tokens_including_eos'] for e in examples) / len(rows),
        'mean_utf8_bytes': sum(e['generated_utf8_bytes'] for e in examples) / len(rows),
        'max_new_tokens': GENERATION_CAP, 'max_output_bytes': max_output_bytes,
        'batch_size_inference': INFERENCE_BATCH, 'peak_gpu_memory_mib':
            torch.cuda.max_memory_allocated() / 2**20 if device.type == 'cuda' else None,
        'timer_scope': 'same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics',
        'extra_diagnostics_seconds_excluded': extra_seconds,
        'unique_generated_code_count': len(codes),
        'most_common_generated_code_count': codes.most_common(1)[0][1],
        'most_common_generated_code_fraction': codes.most_common(1)[0][1] / len(examples),
        'numeric_ast_scope': 'restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1',
        'numeric_ast_statuses': dict(Counter(e['numeric_ast']['status'] for e in examples)),
        'numeric_ast_undefined_name_count': sum(e['numeric_ast']['status'] == 'undefined_name' for e in examples),
        'repeated_tail_32_plus_count': sum(e['repeated_tail_32_plus'] for e in examples),
        'consecutive_lines_3_plus_count': sum(e['consecutive_lines_3_plus'] for e in examples),
        'syntax_error_types': dict(Counter(e['syntax_error_type'] for e in examples if e['syntax_error_type']))}
    for metric in ['parses_as_python', 'defines_solve', 'solve_returns_value', 'indentation_error', 'valid_utf8',
                   'eos_emitted', 'reached_token_cap', 'token_cap_without_eos', 'byte_budget_stop']:
        summary[metric + '_count'] = sum(e[metric] for e in examples)
    if with_loss:
        summary['numeric_ast_comparable_count'] = sum(e['numeric_ast_comparable'] for e in examples)
        summary['numeric_ast_match_count'] = sum(e['numeric_ast_match'] for e in examples)
        summary['reference_numeric_ast_statuses'] = dict(Counter(e['reference_numeric_ast']['status'] for e in examples))
        for metric in ['exact_code_match', 'exact_tokens_with_eos']:
            summary[metric + '_count'] = sum(e[metric] for e in examples)
    if with_loss:
        target_count = sum(e['reference_bpe_tokens_including_eos'] for e in examples)
        target_bytes = sum(e['reference_utf8_bytes'] for e in examples)
        summary.update(reference_target_tokens_including_eos=target_count,
            reference_utf8_bytes=target_bytes, nll_sum_including_eos=fixed_loss * target_count,
            nll_per_utf8_byte=fixed_loss * target_count / target_bytes,
            likelihood_scope='teacher-forced canonical gold tokenization incl EOS; nats per original gold UTF8 byte; not functional accuracy')
    summary['bpe_tokens_per_second'] = summary['generated_tokens_including_eos'] / seconds
    summary['output_bytes_per_second'] = summary['generated_utf8_bytes'] / seconds
    print('DIAGNOSTICS', json.dumps(summary), flush=True)
    for e in examples[:3]: print('EXAMPLE', name, json.dumps(e, ensure_ascii=False), flush=True)
    return {'summary': summary, 'examples': examples}

def save_checkpoint(step):
    state = {'model': {k: v.detach().cpu() for k, v in model.state_dict().items()},
        'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
        'steps': step, 'config': config,
        'data_revision': revision, 'train_sha256': train_hash, 'validation_sha256': validation_hash,
        'tokenizer_json': TOKENIZER_FILE.read_text(encoding='utf-8'), 'tokenizer_sha256': tokenizer_hash,
        'python_random_state': random.getstate(), 'torch_rng_state': torch.get_rng_state(),
        'cuda_rng_state_all': torch.cuda.get_rng_state_all() if device.type == 'cuda' else [],
        'heldout1024_sha256': data_hash(heldout_all),
        'fresh_heldout_reserved_sha256': result['data']['fresh_heldout_reserved']['pairs_sha256'],
        'train_locations_sha256': result['data']['train_locations_sha256'],
        'training_coverage': {'seen_row_flags': torch.frombuffer(seen_train_rows, dtype=torch.uint8).clone(),
            'unique_train_rows_seen': unique_train_rows_seen, 'useful_target_tokens': useful_target_tokens_seen,
            'target_slots_with_padding': padded_target_slots_seen}}
    temporary = LATEST_CHECKPOINT.with_suffix('.pt.tmp')
    torch.save(state, temporary)
    temporary.replace(LATEST_CHECKPOINT)
    result['latest_checkpoint_step'] = step
    if step in FULL_EVALUATION_STEPS:
        name = f'step-{step:05d}.pt'
        target = RUN_DIR / name
        temporary = target.with_suffix('.pt.tmp')
        torch.save(state, temporary)
        temporary.replace(target)
        result['checkpoint_files'][str(step)] = name
    save_results()

def fixed_reference_numeric_audit(examples):
    # Fixed eligibility depends only on reference, never on generated-program success.
    selected = [e for e in examples if e['reference_numeric_ast']['status'] == 'numeric']
    statuses = Counter(e['numeric_ast']['status'] for e in selected)
    agreements = sum(e['numeric_ast_match'] for e in selected)
    pairs = [{'question': e['question'], 'reference_code': e['reference_code']} for e in examples]
    return {'heldout_pairs_total': len(examples), 'reference_numeric_pairs': len(selected),
        'reference_unsupported_pairs': len(examples) - len(selected),
        'questions_and_references_sha256': data_hash(pairs),
        'generated_statuses_on_fixed_reference_subset': dict(statuses),
        'numeric_reference_agreements': agreements,
        'wrong_numeric_count': statuses['numeric'] - agreements,
        'definite_failure_count': sum(n for k, n in statuses.items() if k not in ('numeric', 'unsupported')),
        'unresolved_generated_unsupported_count': statuses['unsupported'],
        'scope': 'same heldout pairs whose reference is supported numeric arithmetic; failures stay in denominator; unsupported unresolved; references may be wrong; not official pass@1'}

@contextlib.contextmanager
def preserve_evaluation_state():
    python_rng, cpu_rng = random.getstate(), torch.get_rng_state().clone()
    cuda_rng = torch.cuda.get_rng_state_all() if device.type == 'cuda' else []
    was_training = model.training
    try:
        yield
    finally:
        random.setstate(python_rng)
        torch.set_rng_state(cpu_rng)
        if device.type == 'cuda': torch.cuda.set_rng_state_all(cuda_rng)
        model.train(was_training)

def combine_heldout(groups):
    examples = [e for group in groups for e in group['examples']]
    counts = [sum(e['reference_bpe_tokens_including_eos'] for e in g['examples']) for g in groups]
    weighted_loss = sum(g['summary']['bpe_loss'] * n for g, n in zip(groups, counts)) / sum(counts)
    target_bytes = sum(e['reference_utf8_bytes'] for e in examples)
    nll_sum = sum(g['summary']['nll_sum_including_eos'] for g in groups)
    return {'examples': len(examples), 'bpe_loss': weighted_loss,
        'reference_target_tokens_including_eos': sum(counts),
        'reference_utf8_bytes': target_bytes, 'nll_sum_including_eos': nll_sum,
        'nll_per_utf8_byte': nll_sum / target_bytes,
        'numeric_audit': fixed_reference_numeric_audit(examples),
        'parses_as_python_count': sum(e['parses_as_python'] for e in examples),
        'eos_emitted_count': sum(e['eos_emitted'] for e in examples),
        'token_cap_without_eos_count': sum(e['token_cap_without_eos'] for e in examples),
        'undefined_name_count': sum(e['numeric_ast']['status'] == 'undefined_name' for e in examples),
        'generation_seconds_sum': sum(g['summary']['seconds'] for g in groups),
        'timing_scope': 'sum of separately timed heldout groups, not a new single inference pass; loss/extra arithmetic excluded'}

def manual_numeric_audit(examples, step, support_augassign=False):
    # Fixed labels from old heldout, evaluation only; unsupported remains unresolved.
    protocol = result['manual74_protocol']
    assert len(examples) == 1024 and len(protocol['labels']) == 74
    rows, correct_hashes = [], set()
    for label in protocol['labels']:
        example = examples[label['heldout1024_index']]
        question_hash = sha(example['question'].encode())
        assert question_hash == label['question_sha256'], 'Manual label/question identity mismatch'
        numeric = arithmetic_augassign(example['generated_code']) if support_augassign else example['numeric_ast']
        correct = numeric['status'] == 'numeric' and any(
            math.isclose(numeric['value'], expected, rel_tol=1e-7, abs_tol=1e-7)
            for expected in label['accepted_numeric_values'])
        score = 'correct' if correct else ('unknown' if numeric['status'] == 'unsupported' else 'wrong')
        if correct: correct_hashes.add(question_hash)
        rows.append({'heldout1024_index': label['heldout1024_index'], 'question': example['question'],
            'accepted_numeric_values': label['accepted_numeric_values'], 'reason': label['reason'],
            'numeric_ast': numeric, 'score': score})
    counts = Counter(row['score'] for row in rows)
    summary = {'scored': 74, 'correct': counts['correct'], 'wrong': counts['wrong'], 'unknown': counts['unknown'],
        'selected_questions': 100, 'excluded_questions': 26,
        'scope': protocol['provenance'], 'rubric': protocol['rubric']}
    comparisons = {}
    saved_protocol = result['manual74_augassign_protocol'] if support_augassign else protocol
    for experiment in ('21', '20', '19', '18'):
        key = f'{experiment}/{step}'
        previous = saved_protocol['saved_summaries'][key]
        previous_hashes = set(saved_protocol['saved_correct_question_hashes'][key])
        comparisons[f'saved{experiment}'] = {'saved_summary': previous,
            'retained_correct': len(previous_hashes & correct_hashes),
            'gained_correct': len(correct_hashes - previous_hashes),
            'lost_correct': len(previous_hashes - correct_hashes),
            'correct_difference': summary['correct'] - previous['correct']}
    summary['augassign_supported'] = support_augassign
    print('MANUAL74_AUGASSIGN' if support_augassign else 'MANUAL74_ORIGINAL', json.dumps({'step': step, 'summary': summary, 'paired_comparisons': comparisons}), flush=True)
    return {'summary': summary, 'paired_comparisons': comparisons, 'rows': rows}

@torch.inference_mode()
def evaluate_control(step):
    with preserve_evaluation_state():
        parity = check_cached(validation[:3])
        if not parity['all_close'] or not parity['greedy_equal']:
            raise RuntimeError('KV parity failed; download results instead of continuing')
        full = step in FULL_EVALUATION_STEPS
        control = {'cache_check': parity, 'full_heldout_evaluated': full}
        result['diagnostics'][str(step)] = control
        groups = [('train64', train_diagnostic), ('validation64', validation)]
        if full:
            groups += [('validation192_extra', validation_extra), ('validation768_new', validation_new)]
        for name, rows in groups:
            control[name] = inspect_split(name, rows)
            save_results()
        control['validation64_conditioning'] = check_question_conditioning(validation,
            correct_loss=control['validation64']['summary']['bpe_loss'])
        if full:
            original = [control['validation64'], control['validation192_extra']]
            control['heldout256'] = combine_heldout(original)
            control['heldout1024'] = combine_heldout(original + [control['validation768_new']])
            examples = [e for g in original + [control['validation768_new']] for e in g['examples']]
            new_matches = {sha(e['question'].encode()) for e in examples if e['numeric_ast_match']}
            control['manual74'] = manual_numeric_audit(examples, step)
            control['manual74_augassign'] = manual_numeric_audit(examples, step, support_augassign=True)
            # Saved summaries and agreement hashes are embedded; no old ZIP upload or rerun.
            for history_key, output_key, label, same_tokenizer in [
                ('historical_constant21', 'comparison_with_saved_constant21', 'saved_constant21', True),
                ('historical_bpe8k20', 'comparison_with_saved_bpe8k20', 'saved_bpe8k20', True),
                ('historical_bpe8k19', 'comparison_with_saved_bpe8k19', 'saved_bpe8k19', True),
                ('historical_bpe4k18', 'comparison_with_saved_bpe4k18', 'saved_bpe4k18', False),
            ]:
                prior = result[history_key]['controls'][str(step)]
                for name in ['heldout256', 'heldout1024']:
                    a, b = control[name]['numeric_audit'], prior[name]['numeric_audit']
                    assert a['questions_and_references_sha256'] == b['questions_and_references_sha256']
                    assert a['reference_numeric_pairs'] == b['reference_numeric_pairs']
                    assert control[name]['reference_utf8_bytes'] == prior[name]['reference_utf8_bytes']
                    if same_tokenizer:
                        assert control[name]['reference_target_tokens_including_eos'] == prior[name]['reference_target_tokens_including_eos']
                old_matches = set(prior['heldout1024']['agreement_question_hashes'])
                comparison = {
                    'provenance': ('Saved user T4 baseline ' + label + ' versus current22 cosine tail from21/75000. '
                        'Saved21/100000 is the primary equal-budget constantLR control;18/19/20 are secondary architecture/tokenizer controls. No old model is rerun.'),
                    'token_CE_comparable': same_tokenizer,
                    'heldout1024_nll_per_utf8_byte': {label: prior['heldout1024']['nll_per_utf8_byte'], 'current_cosine_tail22': control['heldout1024']['nll_per_utf8_byte']},
                    'reference_agreements_out_of_888': {label: prior['heldout1024']['numeric_audit']['numeric_reference_agreements'], 'current_cosine_tail22': control['heldout1024']['numeric_audit']['numeric_reference_agreements']},
                    'paired_reference_changes': {'retained': len(old_matches & new_matches), 'gained': len(new_matches - old_matches), 'lost': len(old_matches - new_matches)},
                    'eos_out_of_1024': {label: prior['heldout1024']['eos_emitted_count'], 'current_cosine_tail22': control['heldout1024']['eos_emitted_count']},
                    'undefined_names_out_of_1024': {label: prior['heldout1024']['undefined_name_count'], 'current_cosine_tail22': control['heldout1024']['undefined_name_count']},
                    'python_AST_out_of_1024': {label: prior['heldout1024']['parses_as_python_count'], 'current_cosine_tail22': control['heldout1024']['parses_as_python_count']},
                }
                if same_tokenizer:
                    comparison['heldout1024_token_CE'] = {label: prior['heldout1024']['bpe_loss'], 'current_cosine_tail22': control['heldout1024']['bpe_loss']}
                control[output_key] = comparison
                print('SAVED_CURRENT_COMPARISON', json.dumps({'step': step, 'baseline': label, **comparison}), flush=True)
        else:
            teacher_loss = evaluate_loss(heldout_all)
            tokens = sum(len(encode(row['code'])) + 1 for row in heldout_all)
            byte_count = sum(len(row['code'].encode('utf-8')) for row in heldout_all)
            control['heldout1024_teacher_forced'] = {'examples': len(heldout_all), 'bpe_loss': teacher_loss,
                'nll_per_utf8_byte': teacher_loss * tokens / byte_count, 'reference_target_tokens_including_eos': tokens,
                'scope': 'larger fixed teacher-forced loss only; full greedy/manual controls at75k and100k'}
            print('HELDOUT1024_LOSS', step, json.dumps(control['heldout1024_teacher_forced']), flush=True)
            print('EARLY_CONTROL', step, json.dumps(control['validation64']['summary']), flush=True)
        save_results()
    return control



## 7. Восстановить AdamW/RNG и проверить 75k

Строгая проверка конфигурации, весов, данных, tokenizer и покрытия. Затем KV-проверка и полный контроль восстановленных весов. Оценка сохраняет RNG и режим модели.


In [ ]:
def restore_source_checkpoint_state(state):
    global seen_train_rows, unique_train_rows_seen, useful_target_tokens_seen, padded_target_slots_seen
    assert not optimizer.state and result['actual_steps'] == 0, 'Checkpoint уже восстановлен; используй новую сессию'
    assert state['steps'] == INITIAL_STEP == 75000
    assert state['config'] == result['historical_constant21']['config']
    allowed_changes = {'evaluation_steps', 'full_evaluation_steps', 'lr', 'scheduler', 'training_from_scratch',
                       'ffn_initialization', 'experiment_factor'}
    for key, value in state['config'].items():
        if key not in allowed_changes:
            assert config[key] == value, ('Config changed', key)
    assert state['data_revision'] == revision and state['train_sha256'] == train_hash
    assert state['validation_sha256'] == validation_hash and state['heldout1024_sha256'] == data_hash(heldout_all)
    assert state['train_locations_sha256'] == result['data']['train_locations_sha256']
    assert state['fresh_heldout_reserved_sha256'] == result['data']['fresh_heldout_reserved']['pairs_sha256']
    assert state['tokenizer_sha256'] == tokenizer_hash and state['tokenizer_json'].encode('utf-8') == TOKENIZER_FILE.read_bytes()
    assert all(torch.isfinite(v).all() and v.dtype == torch.float32 for v in state['model'].values())
    assert sum(v.numel() for v in state['model'].values()) == 12086912
    opt_steps = {int(s['step']) for s in state['optimizer']['state'].values()}
    assert opt_steps == {INITIAL_STEP}
    assert all(torch.isfinite(v).all() for s in state['optimizer']['state'].values() for v in s.values() if torch.is_tensor(v))
    assert state['scheduler']['last_epoch'] == INITIAL_STEP and state['scheduler']['_step_count'] == INITIAL_STEP + 1
    assert state['scheduler']['base_lrs'] == [LEARNING_RATE] and state['scheduler']['_last_lr'] == [CONTINUATION_START_LR]
    assert all(g['lr'] == CONTINUATION_START_LR for g in state['optimizer']['param_groups'])
    coverage = state['training_coverage']
    flags = coverage['seen_row_flags']
    assert flags.dtype == torch.uint8 and flags.shape == (len(train),)
    assert bool(((flags == 0) | (flags == 1)).all())
    assert int(flags.sum()) == coverage['unique_train_rows_seen']
    assert all(coverage[k] == v for k, v in EXPECTED_SOURCE_COVERAGE.items())
    assert state['torch_rng_state'].dtype == torch.uint8
    if device.type == 'cuda':
        assert len(state['cuda_rng_state_all']) == torch.cuda.device_count() == 1
    model.load_state_dict(state['model'], strict=True)
    optimizer.load_state_dict(state['optimizer'])
    # LambdaLR.load_state_dict restores counters/base_lrs; the new trusted cosine
    # function remains attached. Its index is NOT restarted at zero.
    scheduler.load_state_dict(state['scheduler'])
    assert scheduler.last_epoch == INITIAL_STEP
    assert all(math.isclose(g['lr'], learning_rate_for_update(INITIAL_STEP + 1), rel_tol=1e-12)
               for g in optimizer.param_groups)
    seen_train_rows = bytearray(flags.numpy().tobytes())
    unique_train_rows_seen = coverage['unique_train_rows_seen']
    useful_target_tokens_seen = coverage['useful_target_tokens']
    padded_target_slots_seen = coverage['target_slots_with_padding']
    random.setstate(state['python_random_state'])
    torch.set_rng_state(state['torch_rng_state'])
    if device.type == 'cuda':
        torch.cuda.set_rng_state_all(state['cuda_rng_state_all'])
    result['actual_steps'] = INITIAL_STEP
    result['additional_steps'] = 0
    result['training_coverage'] = {'unique_train_rows_seen': unique_train_rows_seen,
        'fraction_train_rows_seen': unique_train_rows_seen / len(train), 'pair_presentations': INITIAL_STEP * BATCH_SIZE,
        'useful_target_tokens': useful_target_tokens_seen, 'target_slots_with_padding': padded_target_slots_seen,
        'target_padding_fraction': 1 - useful_target_tokens_seen / padded_target_slots_seen}
    result['source_checkpoint'] = {**result['source_artifact'], 'optimizer_steps': sorted(opt_steps),
        'lr_next': optimizer.param_groups[0]['lr'], 'scheduler_global_counter': scheduler.last_epoch,
        'restored': ['model', 'AdamW', 'LambdaLR counter/base_lrs', 'Python RNG', 'Torch CPU RNG', 'per-row coverage']
            + (['Torch CUDA RNG'] if device.type == 'cuda' else []),
        'intervention': 'only future LR function changes; source AdamW moments and counters retained'}
    result['phase'] = 'checkpoint_restored'
    save_checkpoint(INITIAL_STEP)
    print('RESTORED', json.dumps(result['source_checkpoint']), flush=True)

restore_source_checkpoint_state(source_checkpoint_state)
del source_checkpoint_state
with preserve_evaluation_state():
    result['initial_cache_check'] = check_cached(validation[:3])
    assert result['initial_cache_check']['all_close'] and result['initial_cache_check']['greedy_equal']
control = evaluate_control(INITIAL_STEP)
saved = result['historical_constant21']['controls'][str(INITIAL_STEP)]
examples = [e for name in ('validation64', 'validation192_extra', 'validation768_new') for e in control[name]['examples']]
loss_delta = control['heldout1024']['bpe_loss'] - saved['heldout1024']['bpe_loss']
result['baseline_reproduction'] = {'loss_delta_vs_saved21': loss_delta,
    'loss_close_atol1e-4': abs(loss_delta) <= 1e-4,
    'all1024_generated_token_ids_equal_saved21': data_hash([e['generated_token_ids'] for e in examples]) == saved['heldout1024_generated_token_ids_sha256'],
    'reference_agreements': {'saved21': saved['heldout1024']['numeric_audit']['numeric_reference_agreements'],
        'restored22': control['heldout1024']['numeric_audit']['numeric_reference_agreements']},
    'manual74_original': control['manual74']['summary'], 'manual74_augassign': control['manual74_augassign']['summary'],
    'scope': 'new inference on exact source weights; saved21 comparison, floating point/runtime can affect close greedy choices'}
assert result['baseline_reproduction']['loss_close_atol1e-4'], 'Loss восстановленных весов не совпал: скачай ZIP для проверки'
result['baseline_ready'] = True
result['phase'] = 'baseline_ready'
save_checkpoint(INITIAL_STEP)
print('BASELINE_READY', json.dumps(result['baseline_reproduction']), flush=True)


## 8. Повторить 75001…100000 с меньшим LR

Latest каждые 1000 шагов; полные checkpoints75k/100k. Скорость и таймер относятся к добавочным 25k, покрытие — ко всей цепочке.


In [ ]:
assert result.get('baseline_ready') and result['actual_steps'] == INITIAL_STEP, 'Сначала восстанови75k; train нельзя запускать повторно'
assert not result.get('continuation_started'), 'Продолжение уже запускалось; сохрани ZIP'
assert optimizer.state and scheduler.last_epoch == INITIAL_STEP
assert math.isclose(optimizer.param_groups[0]['lr'], CONTINUATION_START_LR)
assert result['initial_cache_check']['all_close'] and result['initial_cache_check']['greedy_equal']
result['continuation_started'] = True
assert next(model.parameters()).dtype == torch.float32
result['phase'] = 'training'
result['training_timer_scope'] = 'cached-ID batch construction, transfers, forward/backward, clipping, AdamW, scheduler, coverage counters, block-end CUDA sync; cache setup excluded'
result['training_log'].append({'step': INITIAL_STEP, 'additional_step': 0, 'train_batch_loss': None,
    'validation_loss_16': evaluate_loss(validation[:16]),
    'lr_next': optimizer.param_groups[0]['lr'], 'training_seconds': 0.0})
save_results()
run_started = time.perf_counter()
training_seconds = 0.0
completed_steps = INITIAL_STEP
source_useful_tokens = useful_target_tokens_seen
source_padded_slots = padded_target_slots_seen
sync()
block_started, block_active = time.perf_counter(), True
try:
    for step in range(INITIAL_STEP + 1, TOTAL_STEPS + 1):
        model.train()
        batch_indices = random.sample(range(len(train)), BATCH_SIZE)
        src, tgt, labels = cached_training_batch(batch_indices)
        logits = model(src, tgt)
        loss = criterion(logits.reshape(-1, VOCAB), labels.reshape(-1))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        gradient_norm_before_clip = nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        lr_used = optimizer.param_groups[0]['lr']
        optimizer.step()
        scheduler.step()
        completed_steps = step
        for index in batch_indices:
            if not seen_train_rows[index]:
                seen_train_rows[index] = 1
                unique_train_rows_seen += 1
        useful_target_tokens_seen += sum(len(train_cache[i][1]) + 1 for i in batch_indices)
        padded_target_slots_seen += labels.numel()
        if step == INITIAL_STEP + 1 or step % LOG_EVERY == 0 or step in EVALUATION_STEPS or step == TOTAL_STEPS:
            sync()
            training_seconds += time.perf_counter() - block_started
            block_active = False
            result['actual_steps'] = completed_steps
            result['training_seconds_excluding_evaluation_logging_and_checkpoints'] = training_seconds
            train_loss, val_loss = loss.item(), evaluate_loss(validation[:16])
            if not math.isfinite(train_loss) or not math.isfinite(val_loss):
                raise FloatingPointError('Non-finite loss; last finite checkpoint is kept')
            result['training_coverage'] = {'unique_train_rows_seen': unique_train_rows_seen,
                'fraction_train_rows_seen': unique_train_rows_seen / len(train),
                'pair_presentations': step * BATCH_SIZE, 'useful_target_tokens': useful_target_tokens_seen,
                'target_slots_with_padding': padded_target_slots_seen,
                'target_padding_fraction': 1 - useful_target_tokens_seen / padded_target_slots_seen}
            entry = {'step': step, 'additional_step': step - INITIAL_STEP, 'train_batch_loss': train_loss, 'validation_loss_16': val_loss,
                'gradient_norm_before_clip': float(gradient_norm_before_clip),
                'lr_used': lr_used, 'lr_next': optimizer.param_groups[0]['lr'],
                'training_seconds': training_seconds, 'updates_per_second': (step - INITIAL_STEP) / training_seconds,
                'mean_pair_presentations': step * BATCH_SIZE / len(train),
                'unique_train_rows_seen': unique_train_rows_seen, 'fraction_train_rows_seen': unique_train_rows_seen / len(train),
                'useful_target_tokens_per_second': (useful_target_tokens_seen - source_useful_tokens) / training_seconds,
                'target_padding_fraction': 1 - useful_target_tokens_seen / padded_target_slots_seen,
                'additional_target_padding_fraction': 1 - (useful_target_tokens_seen - source_useful_tokens) / (padded_target_slots_seen - source_padded_slots)}
            result['training_log'].append(entry)
            print('LOSS', json.dumps(entry), flush=True)
            save_results()
            if step in EVALUATION_STEPS:
                evaluate_control(step)
            if step % CHECKPOINT_EVERY == 0 or step in EVALUATION_STEPS or step == TOTAL_STEPS:
                save_checkpoint(completed_steps)
                print('CHECKPOINT', completed_steps, flush=True)
            sync()
            if step < TOTAL_STEPS:
                block_started, block_active = time.perf_counter(), True
    result['run_finished'] = True
    result['phase'] = 'training_finished'
except BaseException as error:
    result['phase'] = 'training_stopped'
    result['training_error'] = {'type': type(error).__name__, 'message': str(error)}
    raise
finally:
    sync()
    if block_active: training_seconds += time.perf_counter() - block_started
    result['actual_steps'] = completed_steps
    result['additional_steps'] = completed_steps - INITIAL_STEP
    result['total_chain_training_seconds'] = result['historical_constant21']['training_seconds_to75000'] + training_seconds
    result['training_seconds_excluding_evaluation_logging_and_checkpoints'] = training_seconds
    result['training_coverage'] = {'unique_train_rows_seen': unique_train_rows_seen,
        'fraction_train_rows_seen': unique_train_rows_seen / len(train), 'pair_presentations': completed_steps * BATCH_SIZE,
        'useful_target_tokens': useful_target_tokens_seen, 'target_slots_with_padding': padded_target_slots_seen,
        'target_padding_fraction': 1 - useful_target_tokens_seen / padded_target_slots_seen if padded_target_slots_seen else None}
    result['active_wall_seconds_including_diagnostics_and_checkpoints'] = time.perf_counter() - run_started
    if result.get('training_error', {}).get('type') != 'FloatingPointError':
        save_checkpoint(completed_steps)
    save_results()


## 9. Итоговое сравнение с постоянным LR

Loss, byte NLL, reference888, оба ручных счёта и парные приобретения/потери на одинаковых 100k.


In [ ]:
assert result['run_finished'] and result['actual_steps'] == TOTAL_STEPS
result['phase'] = 'final_validation'
result['final_validation'] = {'validation64_byte512': inspect_split('validation64_byte512', validation,
    max_output_bytes=BYTE_BUDGET)}
control = result['diagnostics'][str(TOTAL_STEPS)]
result['fixed_reference_numeric_audit'] = control['heldout256']['numeric_audit']
result['expanded_heldout_numeric_audit'] = control['heldout1024']['numeric_audit']
result['final_heldout_comparison_with_saved_bpe4k18'] = control['comparison_with_saved_bpe4k18']
result['final_heldout_comparison_with_saved_bpe8k19'] = control['comparison_with_saved_bpe8k19']
result['final_heldout_comparison_with_saved_bpe8k20'] = control['comparison_with_saved_bpe8k20']
result['final_manual74'] = control['manual74']
result['final_manual74_augassign'] = control['manual74_augassign']
result['final_heldout_comparison_with_saved_constant21'] = control['comparison_with_saved_constant21']
result['primary_equal_budget_comparison'] = {'same_final_updates': TOTAL_STEPS, 'same_source_checkpoint': INITIAL_STEP,
    'source_checkpoint_sha256': EXPECTED_SOURCE_CHECKPOINT_SHA256,
    'new_schedule': config['scheduler'], 'control_tail_lr': CONTINUATION_START_LR,
    'loss_and_reference': control['comparison_with_saved_constant21'],
    'manual74_original': control['manual74']['paired_comparisons']['saved21'],
    'manual74_augassign': control['manual74_augassign']['paired_comparisons']['saved21'],
    'training_seconds_new_tail': result['training_seconds_excluding_evaluation_logging_and_checkpoints'],
    'training_seconds_saved_tail': result['historical_constant21']['training_seconds_to100000'] - result['historical_constant21']['training_seconds_to75000'],
    'scope': 'existing constant tail versus cosine replay; same initial weights/AdamW/RNG/batches and final100k; separate sessions, not a paired speed benchmark'}
print('PRIMARY_LR_COMPARISON', json.dumps(result['primary_equal_budget_comparison']), flush=True)
result['phase'] = 'validation_finished'
save_results()


## 10. Полный test1919

Прежние greedy/KV/batch32/cap512 и порядок. Таймер включает encoding и генерацию; загрузка/setup/loss вне него. Граница официального таймера пока неизвестна.


In [ ]:
assert result['run_finished'] and result['actual_steps'] == TOTAL_STEPS
result['phase'] = 'test_data'
save_results()
test_file = cache / 'test.parquet'
if not test_file.exists(): urlretrieve(f'{base_url}/data/test/test.parquet', test_file)
assert sha(test_file.read_bytes()) == '8bde59c8d8bb7edc4e8a503be22fc132c4a47773025d333432e78c5489e6522a'
test_questions = pq.read_table(test_file, columns=['question'])['question'].to_pylist()
assert len(test_questions) == 1919
assert all(len(encode(q)) + 2 <= MAX_POSITIONS for q in test_questions)
test_rows = [{'question': q} for q in test_questions]
result['test_data'] = {'rows': len(test_rows), 'questions_sha256': data_hash(test_rows),
    'file_sha256': sha(test_file.read_bytes()), 'row_order': 'unchanged parquet order', 'dataset_revision': revision}
result['phase'] = 'test_generation'
save_results()
test_output = inspect_split('test1919', test_rows, with_loss=False)
TEST_GENERATIONS = RUN_DIR / 'test-generations.json'
write_json(TEST_GENERATIONS, test_output['examples'])
result['test'] = {'summary': test_output['summary'], 'examples_file': TEST_GENERATIONS.name}
result['test']['summary']['within_180_seconds'] = test_output['summary']['seconds'] <= 180
result['test']['comparison_with_saved_times'] = {
    'saved_constant21_seconds': result['historical_constant21']['test']['summary']['seconds'],
    'saved_bpe8k20_seconds': result['historical_bpe8k20']['test']['summary']['seconds'],
    'saved_bpe8k19_seconds': result['historical_bpe8k19']['test']['summary']['seconds'],
    'saved_bpe4k18_seconds': result['historical_bpe4k18']['test']['summary']['seconds'],
    'current_cosine_tail22_seconds': test_output['summary']['seconds'],
    'scope': 'same timer region; separate T4 sessions, not paired GPU benchmark',
}
result['test_evaluated'] = True
result['phase'] = 'complete'
save_results()
print('TEST_SUMMARY', json.dumps(result['test']['summary']), flush=True)


## 11. Скачать ZIP и notebook с outputs

Ячейка только упаковывает и работает после прерывания. Вернуть ZIP `hw02-encoder4-decoder4-qwen-bpe8k-swiglu-500k-cosine-tail-75000-to-100000-results.zip` и notebook с outputs: File → Download → Download.ipynb.


In [ ]:
# This cell never trains, generates or restarts anything.
assert RESULTS_FILE.exists(), 'Сначала должны появиться результаты запуска'
result['archive_prepared'] = True
save_results()
README_FILE = RUN_DIR / 'README.txt'
README_FILE.write_text(
    'Experiment22: exact checkpoint21/75000 replayed to100000 with cosineLR1e-4 to1e-5.\n'
    'Same4+4/width256/QwenBPE8192/Pre-LN/SwiGLU/train500k/batch8/FP32/cache; AdamW moments and RNG restored.\n'
    'No optimizer reset or new warmup. Scheduler GLOBAL completed-update counter preserved.\n'
    'Saved21/100000 constantLR1e-4 is the primary equal-budget control; older18/19/20 are secondary.\n'
    'Original reference888 and manual74 scores retained; supplementary manual74 also supports Name AugAssign.\n'
    'Manual labels are evaluation only, never training/tokenizer fitting. Fresh1024 remains unscored.\n'
    'Training seconds/updates per second describe only added25k; total_chain adds source21 prefix75k timer.\n'
    'Latest and step checkpoints include model/AdamW/scheduler/data/tokenizer/RNG/coverage.\n'
    'Return ZIP and notebook with outputs. Packaging also works after interruption and does not restart training.\n',
    encoding='utf-8')
files_to_pack = sorted(p for p in RUN_DIR.iterdir() if p.is_file()
                       and p.name != 'manifest.json' and not p.name.endswith('.tmp'))
manifest = {'experiment': result['experiment'], 'actual_steps': result['actual_steps'], 'phase': result['phase'],
    'files': [{'name': p.name, 'bytes': p.stat().st_size, 'sha256': sha(p.read_bytes())} for p in files_to_pack]}
write_json(RUN_DIR / 'manifest.json', manifest)
with zipfile.ZipFile(ARCHIVE_FILE, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for p in files_to_pack + [RUN_DIR / 'manifest.json']: archive.write(p, arcname=p.name)
print('ARCHIVE', str(ARCHIVE_FILE), ARCHIVE_FILE.stat().st_size, 'bytes', '| steps:', result['actual_steps'])
from google.colab import files
files.download(str(ARCHIVE_FILE))
